<a href="https://colab.research.google.com/github/MacKumachin/Clifford-FBSM-SignalControl/blob/main/JPC_MPD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%writefile tep_dtt_benchmark.py
import os
import argparse
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import LinearSVC
from sklearn.metrics import roc_auc_score, average_precision_score


def _get_label(ds):
    if hasattr(ds, "label"):
        return ds.label
    if hasattr(ds, "labels"):
        return ds.labels
    raise AttributeError("Dataset has neither .label nor .labels")


def _get_mask(ds, name):
    if hasattr(ds, name):
        return getattr(ds, name)
    raise AttributeError(f"Dataset has no .{name}")


def _infer_levels(df: pd.DataFrame):
    # Expected MultiIndex: (run_id, sample) like FDDBenchmark tutorial
    if not isinstance(df.index, pd.MultiIndex) or df.index.nlevels < 2:
        raise ValueError("Expected a MultiIndex with (run_id, sample)")
    run_level = 0
    time_level = 1
    return run_level, time_level


def build_theta_pca(df, train_nominal_mask, n_components=2):
    # Fit scaler + PCA on nominal TRAIN points only (semi-supervised)
    scaler = StandardScaler()
    X_nom = df.loc[train_nominal_mask].values
    X_nom_s = scaler.fit_transform(X_nom)

    pca = PCA(n_components=n_components, random_state=0)
    pca.fit(X_nom_s)

    X_all_s = scaler.transform(df.values)
    theta = pca.transform(X_all_s)
    theta_df = pd.DataFrame(theta, index=df.index, columns=[f"theta{i+1}" for i in range(n_components)])
    return scaler, pca, theta_df


def fit_linear_boundary(theta_df, y, train_mask, fault_id, onset_train=20):
    # Binary labels inside TRAIN: nominal(0) vs fault(post-onset)(1)
    run_level, time_level = _infer_levels(theta_df)
    t = theta_df.index.get_level_values(time_level).astype(int)

    in_train = train_mask.astype(bool)
    is_fault = (y == fault_id)
    is_nominal = (y == 0)

    # fault points: only after onset_train
    train_fault_pts = in_train & is_fault & (t >= onset_train)
    train_nom_pts = in_train & (is_nominal | (is_fault & (t < onset_train)))

    theta_tr = pd.concat([theta_df.loc[train_nom_pts], theta_df.loc[train_fault_pts]], axis=0)
    y_tr = np.r_[np.zeros(train_nom_pts.sum(), dtype=int), np.ones(train_fault_pts.sum(), dtype=int)]

    if theta_tr.shape[0] < 100:
        raise ValueError("Too few training points. Try a different fault_id or dataset=rieth_tep.")

    clf = LinearSVC(C=1.0, class_weight="balanced", random_state=0)
    clf.fit(theta_tr.values, y_tr)

    w = clf.coef_.reshape(-1)
    b = float(clf.intercept_[0])
    return w, b


def compute_dist_ti_score(theta_df, w, b, eps, delta):
    # g(theta)=w·theta+b ; dist=|g|/||w|| ; TI=|<w,v>|/(||w||·||v||)
    wnorm = np.linalg.norm(w) + 1e-12
    g = theta_df.values @ w + b
    dist = np.abs(g) / wnorm

    # velocity per run
    run_level, time_level = _infer_levels(theta_df)
    v = theta_df.groupby(level=run_level).diff().fillna(0.0).values
    vnorm = np.linalg.norm(v, axis=1) + 1e-12
    ti = np.abs(v @ w) / (wnorm * vnorm)  # in [0,1] roughly

    score = np.maximum(0.0, eps - dist) + np.maximum(0.0, delta - ti)
    out = pd.DataFrame(
        {"dist": dist, "TI": ti, "score": score},
        index=theta_df.index
    )
    return out


def pick_threshold_from_nominal(features_df, y, test_mask, fault_id, onset_test=160, fpr=0.05):
    run_level, time_level = _infer_levels(features_df)
    t = features_df.index.get_level_values(time_level).astype(int)

    in_test = test_mask.astype(bool)
    is_fault = (y == fault_id)
    is_nominal = (y == 0)

    nominal_pts = in_test & (is_nominal | (is_fault & (t < onset_test)))
    s_nom = features_df.loc[nominal_pts, "score"].values
    thr = np.quantile(s_nom, 1.0 - fpr)
    return float(thr)


def eval_fault_runs(features_df, y, test_mask, fault_id, onset_test=160, thr=None):
    run_level, time_level = _infer_levels(features_df)
    t = features_df.index.get_level_values(time_level).astype(int)

    in_test = test_mask.astype(bool)
    is_fault = (y == fault_id)

    # evaluate only on faulty test runs, using within-run pre/post labels
    fault_pts = in_test & is_fault
    df_f = features_df.loc[fault_pts].copy()
    t_f = t[fault_pts]
    df_f["ybin"] = (t_f >= onset_test).astype(int)

    # per-run AUC/PR (skip degenerate runs)
    aucs, prs, delays = [], [], []
    for run_id, g in df_f.groupby(level=run_level):
        yb = g["ybin"].values
        sc = g["score"].values
        if yb.min() == yb.max():
            continue
        aucs.append(roc_auc_score(yb, sc))
        prs.append(average_precision_score(yb, sc))
        if thr is not None:
            # delay: first detection after onset_test
            tt = g.index.get_level_values(time_level).astype(int).values
            post = (tt >= onset_test)
            hit = np.where((sc > thr) & post)[0]
            if len(hit) == 0:
                delays.append((tt.max() - onset_test) + 1)
            else:
                delays.append(int(tt[hit[0]] - onset_test))

    return {
        "AUC_mean": float(np.mean(aucs)) if aucs else np.nan,
        "AUC_std": float(np.std(aucs)) if aucs else np.nan,
        "PRAUC_mean": float(np.mean(prs)) if prs else np.nan,
        "PRAUC_std": float(np.std(prs)) if prs else np.nan,
        "Delay_mean": float(np.mean(delays)) if delays else np.nan,
        "Delay_std": float(np.std(delays)) if delays else np.nan,
        "n_runs": int(len(delays)),
    }


def plot_boundary(theta_df, y, test_mask, fault_id, w, b, out_pdf, n_show=2):
    run_level, time_level = _infer_levels(theta_df)
    in_test = test_mask.astype(bool)

    # pick one nominal run and one fault run
    runs_nom = theta_df.loc[in_test & (y == 0)].index.get_level_values(run_level).unique().tolist()
    runs_flt = theta_df.loc[in_test & (y == fault_id)].index.get_level_values(run_level).unique().tolist()
    if not runs_nom or not runs_flt:
        raise ValueError("No suitable test runs found for plotting.")

    rn = runs_nom[0]
    rf = runs_flt[0]

    th_n = theta_df.xs(rn, level=run_level).values
    th_f = theta_df.xs(rf, level=run_level).values

    # boundary line in theta1-theta2 plane: w1*x+w2*y+b=0
    w1, w2 = w[0], w[1]
    xmin = min(th_n[:,0].min(), th_f[:,0].min())
    xmax = max(th_n[:,0].max(), th_f[:,0].max())
    xs = np.linspace(xmin, xmax, 200)
    if abs(w2) < 1e-12:
        ys = np.zeros_like(xs) + np.nan
    else:
        ys = -(w1 * xs + b) / w2

    plt.figure()
    plt.plot(th_n[:,0], th_n[:,1], linewidth=1.5, label="nominal (test)")
    plt.plot(th_f[:,0], th_f[:,1], linewidth=1.5, label=f"fault {fault_id} (test)")
    plt.plot(xs, ys, linewidth=1.0, label="proxy boundary g(θ)=0")
    plt.xlabel(r"$\theta_1$ (PCA)")
    plt.ylabel(r"$\theta_2$ (PCA)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_pdf)
    plt.close()


def plot_timeseries(features_df, y, test_mask, fault_id, onset_test, thr, out_pdf):
    run_level, time_level = _infer_levels(features_df)
    in_test = test_mask.astype(bool)
    runs_flt = features_df.loc[in_test & (y == fault_id)].index.get_level_values(run_level).unique().tolist()
    if not runs_flt:
        raise ValueError("No fault test runs found for timeseries plot.")
    rf = runs_flt[0]
    g = features_df.xs(rf, level=run_level).copy()
    t = g.index.get_level_values(time_level).astype(int).values

    plt.figure()
    plt.plot(t, g["score"].values, label="DTT score")
    plt.axvline(onset_test, linewidth=1.0, linestyle="--", label="fault onset")
    plt.axhline(thr, linewidth=1.0, linestyle="--", label="thr@FPR")
    plt.xlabel("sample")
    plt.ylabel("score")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_pdf)
    plt.close()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dataset", type=str, default="small_tep", choices=["small_tep", "rieth_tep"])
    ap.add_argument("--fault_id", type=int, default=1)
    ap.add_argument("--onset_train", type=int, default=20)
    ap.add_argument("--onset_test", type=int, default=160)
    ap.add_argument("--q_eps", type=float, default=0.05)
    ap.add_argument("--q_delta", type=float, default=0.05)
    ap.add_argument("--fpr", type=float, default=0.05)
    ap.add_argument("--outdir", type=str, default="out_tep")
    args = ap.parse_args()

    os.makedirs(args.outdir, exist_ok=True)

    from fddbenchmark import FDDDataset
    ds = FDDDataset(name=args.dataset)
    df = ds.df
    y = _get_label(ds)
    train_mask = _get_mask(ds, "train_mask")
    test_mask = _get_mask(ds, "test_mask")

    # 1) build theta (PCA2) from nominal TRAIN points
    run_level, time_level = _infer_levels(df)
    t = df.index.get_level_values(time_level).astype(int)
    train_nom = train_mask.astype(bool) & ((y == 0) | ((y == args.fault_id) & (t < args.onset_train)))
    _, _, theta_df = build_theta_pca(df, train_nom, n_components=2)

    # 2) fit proxy boundary g(θ)=0 (linear SVM)
    w, b = fit_linear_boundary(theta_df, y, train_mask, args.fault_id, onset_train=args.onset_train)

    # 3) dist/TI/score with eps,delta from nominal TRAIN distribution
    #    eps: lower-quantile of dist ; delta: lower-quantile of TI (tangency)
    wnorm = np.linalg.norm(w) + 1e-12
    g_all = theta_df.values @ w + b
    dist_all = np.abs(g_all) / wnorm
    v_all = theta_df.groupby(level=run_level).diff().fillna(0.0).values
    ti_all = np.abs(v_all @ w) / (wnorm * (np.linalg.norm(v_all, axis=1) + 1e-12))

    dist_nom = dist_all[train_nom.values]
    ti_nom = ti_all[train_nom.values]
    eps = float(np.quantile(dist_nom, args.q_eps))
    delta = float(np.quantile(ti_nom, args.q_delta))

    feat = compute_dist_ti_score(theta_df, w, b, eps, delta)

    # 4) choose threshold from nominal TEST points to hit FPR
    thr = pick_threshold_from_nominal(feat, y, test_mask, args.fault_id, onset_test=args.onset_test, fpr=args.fpr)

    # 5) evaluate on faulty test runs
    metrics = eval_fault_runs(feat, y, test_mask, args.fault_id, onset_test=args.onset_test, thr=thr)
    metrics.update({"fault_id": args.fault_id, "dataset": args.dataset, "eps": eps, "delta": delta, "thr": thr})
    pd.DataFrame([metrics]).to_csv(os.path.join(args.outdir, "tep_metrics.csv"), index=False)

    # 6) plots
    plot_boundary(theta_df, y, test_mask, args.fault_id, w, b, os.path.join(args.outdir, "FigTEP_boundary.pdf"))
    plot_timeseries(feat, y, test_mask, args.fault_id, args.onset_test, thr, os.path.join(args.outdir, "FigTEP_timeseries.pdf"))

    print("[OK] wrote:", os.path.join(args.outdir, "tep_metrics.csv"))
    print("[OK] figs :", os.path.join(args.outdir, "FigTEP_boundary.pdf"), os.path.join(args.outdir, "FigTEP_timeseries.pdf"))


if __name__ == "__main__":
    main()


Writing tep_dtt_benchmark.py


In [ ]:
!pip -q install git+https://github.com/AIRI-Institute/fddbenchmark
!python tep_dtt_benchmark.py --dataset small_tep --fault_id 1 --outdir out_tep_f1
!ls -lh out_tep_f1


  Preparing metadata (setup.py) ... done
Extracting dataset.csv: 58.6MB [00:00, 141MB/s]                
Extracting labels.csv: 9.77MB [00:00, 5.64GB/s]       
Extracting test_mask.csv: 9.77MB [00:00, 4.77GB/s]       
Extracting train_mask.csv: 9.77MB [00:00, 4.64GB/s]       
Extracting unlabeled_train_mask.csv: 9.77MB [00:00, 4.54GB/s]       
Reading data/small_tep/dataset.csv: 100% 153300/153300 [00:01<00:00, 85678.96it/s]
Reading data/small_tep/labels.csv: 100% 153300/153300 [00:00<00:00, 3451666.57it/s]
Reading data/small_tep/train_mask.csv: 100% 153300/153300 [00:00<00:00, 3217169.95it/s]
Reading data/small_tep/test_mask.csv: 100% 153300/153300 [00:00<00:00, 3178791.16it/s]
Traceback (most recent call last):
  File "/content/tep_dtt_benchmark.py", line 277, in <module>
    main()
  File "/content/tep_dtt_benchmark.py", line 270, in main
    plot_timeseries(feat, y, test_mask, args.fault_id, args.onset_test, thr, os.path.join(args.outdir, "FigTEP_timeseries.pdf"))
  File "/content/

In [ ]:
# 代表4fault（まずこれで論文に入る）
FAULTS="1 2 4 6"

mkdir -p out_tep
for f in $FAULTS; do
  python tep_dtt_benchmark.py --dataset small_tep --fault_id $f --outdir out_tep/f${f}
done

# 出力確認
find out_tep -maxdepth 2 -name "tep_metrics.csv" -o -name "FigTEP_*.pdf" | sort


SyntaxError: invalid syntax (ipython-input-2891666126.py, line 4)

In [ ]:
%%writefile tep_dtt_benchmark.py
import os
import argparse
import numpy as np
import pandas as pd

if isinstance(df.index, pd.MultiIndex) and len(df.index.names) >= 2:
    if df.index.names[0] == "run" and df.index.names[1] != "t":
        df.index = df.index.set_names(["run", "t"])

import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import LinearSVC
from sklearn.metrics import roc_auc_score, average_precision_score


def ensure_multiindex(df: pd.DataFrame) -> pd.DataFrame:
    """
    Make df.index a MultiIndex (run, t) if it's not already.
    Heuristics:
      - if columns include run_id (or run), and time (or t), use them
      - else if run_id exists only, create t as cumcount per run
      - else: single run (run=0) with t=0..N-1
    """
    if isinstance(df.index, pd.MultiIndex) and df.index.nlevels >= 2:
        return df

    cols = set(df.columns)
    run_col = None
    time_col = None

    for c in ["run_id", "run", "trajectory", "traj", "case", "series", "batch"]:
        if c in cols:
            run_col = c
            break
    for c in ["time", "t", "step", "k", "idx"]:
        if c in cols:
            time_col = c
            break

    if run_col is not None and time_col is not None:
        run = df[run_col].to_numpy()
        t = df[time_col].to_numpy()
        df2 = df.drop(columns=[run_col, time_col])
        df2.index = pd.MultiIndex.from_arrays([run, t], names=["run", "t"])
        return df2.sort_index()

    if run_col is not None and time_col is None:
        run = df[run_col].to_numpy()
        # time = within-run order
        t = pd.Series(run).groupby(run).cumcount().to_numpy()
        df2 = df.drop(columns=[run_col])
        df2.index = pd.MultiIndex.from_arrays([run, t], names=["run", "t"])
        return df2.sort_index()

    # fallback: single run
    n = len(df)
    df2 = df.copy()
    df2.index = pd.MultiIndex.from_arrays([np.zeros(n, dtype=int), np.arange(n, dtype=int)],
                                         names=["run", "t"])
    return df2


def build_theta_pca(df, train_nominal_mask, n_components=2):
    scaler = StandardScaler()
    X_nom = df.loc[train_nominal_mask].values
    X_nom_s = scaler.fit_transform(X_nom)

    pca = PCA(n_components=n_components, random_state=0)
    pca.fit(X_nom_s)

    X_all_s = scaler.transform(df.values)
    theta = pca.transform(X_all_s)
    theta_df = pd.DataFrame(theta, index=df.index, columns=[f"theta{i+1}" for i in range(n_components)])
    return scaler, pca, theta_df


def fit_linear_boundary(theta_df, y, train_mask, fault_id, onset_train=20):
    t = theta_df.index.get_level_values("t").astype(int).to_numpy()

    in_train = train_mask.astype(bool)
    is_fault = (y == fault_id)
    is_nominal = (y == 0)

    train_fault_pts = in_train & is_fault & (t >= onset_train)
    train_nom_pts = in_train & (is_nominal | (is_fault & (t < onset_train)))

    theta_tr = pd.concat([theta_df.loc[train_nom_pts], theta_df.loc[train_fault_pts]], axis=0)
    y_tr = np.r_[np.zeros(train_nom_pts.sum(), dtype=int), np.ones(train_fault_pts.sum(), dtype=int)]

    if theta_tr.shape[0] < 100:
        raise ValueError("Too few training points. Try a different fault_id or dataset.")

    clf = LinearSVC(C=1.0, class_weight="balanced", random_state=0)
    clf.fit(theta_tr.values, y_tr)

    w = clf.coef_.reshape(-1)
    b = float(clf.intercept_[0])
    return w, b


def compute_dist_ti_score(theta_df, w, b, eps, delta):
    wnorm = np.linalg.norm(w) + 1e-12
    g = theta_df.values @ w + b
    dist = np.abs(g) / wnorm

    v = theta_df.groupby(level="run").diff().fillna(0.0).values
    vnorm = np.linalg.norm(v, axis=1) + 1e-12
    ti = np.abs(v @ w) / (wnorm * vnorm)

    score = np.maximum(0.0, eps - dist) + np.maximum(0.0, delta - ti)
    return pd.DataFrame({"dist": dist, "TI": ti, "score": score}, index=theta_df.index)


def pick_threshold_from_nominal(features_df, y, test_mask, fault_id, onset_test=160, fpr=0.05):
    t = features_df.index.get_level_values("t").astype(int).to_numpy()

    in_test = test_mask.astype(bool)
    is_fault = (y == fault_id)
    is_nominal = (y == 0)

    nominal_pts = in_test & (is_nominal | (is_fault & (t < onset_test)))
    s_nom = features_df.loc[nominal_pts, "score"].values
    thr = np.quantile(s_nom, 1.0 - fpr)
    return float(thr)


def eval_fault_runs(features_df, y, test_mask, fault_id, onset_test=160, thr=None):
    t = features_df.index.get_level_values("t").astype(int).to_numpy()

    in_test = test_mask.astype(bool)
    is_fault = (y == fault_id)

    fault_pts = in_test & is_fault
    df_f = features_df.loc[fault_pts].copy()
    t_f = t[fault_pts]
    df_f["ybin"] = (t_f >= onset_test).astype(int)

    aucs, prs, delays = [], [], []
    for run_id, g in df_f.groupby(level="run"):
        yb = g["ybin"].values
        sc = g["score"].values
        if yb.min() == yb.max():
            continue
        aucs.append(roc_auc_score(yb, sc))
        prs.append(average_precision_score(yb, sc))

        if thr is not None:
            tt = g.index.get_level_values("t").astype(int).values
            post = (tt >= onset_test)
            hit = np.where((sc > thr) & post)[0]
            if len(hit) == 0:
                delays.append((tt.max() - onset_test) + 1)
            else:
                delays.append(int(tt[hit[0]] - onset_test))

    return {
        "AUC_mean": float(np.mean(aucs)) if aucs else np.nan,
        "AUC_std": float(np.std(aucs)) if aucs else np.nan,
        "PRAUC_mean": float(np.mean(prs)) if prs else np.nan,
        "PRAUC_std": float(np.std(prs)) if prs else np.nan,
        "Delay_mean": float(np.mean(delays)) if delays else np.nan,
        "Delay_std": float(np.std(delays)) if delays else np.nan,
        "n_runs": int(len(delays)),
    }


def plot_boundary(theta_df, y, test_mask, fault_id, w, b, out_pdf):
    in_test = test_mask.astype(bool)

    runs_nom = theta_df.loc[in_test & (y == 0)].index.get_level_values("run").unique().tolist()
    runs_flt = theta_df.loc[in_test & (y == fault_id)].index.get_level_values("run").unique().tolist()
    if not runs_nom or not runs_flt:
        raise ValueError("No suitable test runs found for plotting.")

    rn = runs_nom[0]
    rf = runs_flt[0]

    th_n = theta_df.xs(rn, level="run").values
    th_f = theta_df.xs(rf, level="run").values

    w1, w2 = w[0], w[1]
    xmin = min(th_n[:, 0].min(), th_f[:, 0].min())
    xmax = max(th_n[:, 0].max(), th_f[:, 0].max())
    xs = np.linspace(xmin, xmax, 200)
    ys = -(w1 * xs + b) / w2 if abs(w2) > 1e-12 else np.full_like(xs, np.nan)

    plt.figure()
    plt.plot(th_n[:, 0], th_n[:, 1], linewidth=1.5, label="nominal (test)")
    plt.plot(th_f[:, 0], th_f[:, 1], linewidth=1.5, label=f"fault {fault_id} (test)")
    plt.plot(xs, ys, linewidth=1.0, label="proxy boundary g(θ)=0")
    plt.xlabel(r"$\theta_1$ (PCA)")
    plt.ylabel(r"$\theta_2$ (PCA)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_pdf)
    plt.close()


def plot_timeseries(features_df, y, test_mask, fault_id, onset_test, thr, out_pdf):
    in_test = test_mask.astype(bool)
    runs_flt = features_df.loc[in_test & (y == fault_id)].index.get_level_values("run").unique().tolist()
    if not runs_flt:
        raise ValueError("No fault test runs found for timeseries plot.")
    rf = runs_flt[0]
    g = features_df.xs(rf, level="run").copy()
    t = g.index.get_level_values("t").astype(int).values

    plt.figure()
    plt.plot(t, g["score"].values, label="DTT score")
    plt.axvline(onset_test, linewidth=1.0, linestyle="--", label="fault onset")
    plt.axhline(thr, linewidth=1.0, linestyle="--", label="thr@FPR")
    plt.xlabel("sample")
    plt.ylabel("score")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_pdf)
    plt.close()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dataset", type=str, default="small_tep")
    ap.add_argument("--fault_id", type=int, default=1)
    ap.add_argument("--onset_train", type=int, default=20)
    ap.add_argument("--onset_test", type=int, default=160)
    ap.add_argument("--q_eps", type=float, default=0.05)
    ap.add_argument("--q_delta", type=float, default=0.05)
    ap.add_argument("--fpr", type=float, default=0.05)
    ap.add_argument("--outdir", type=str, default="out_tep")
    args = ap.parse_args()

    os.makedirs(args.outdir, exist_ok=True)

    from fddbenchmark import FDDDataset
    ds = FDDDataset(name=args.dataset)
    df = ensure_multiindex(ds.df)

    y = np.asarray(ds.label)
    train_mask = np.asarray(ds.train_mask)
    test_mask = np.asarray(ds.test_mask)

    # time values (MultiIndex now guaranteed)
    t = df.index.get_level_values("t").astype(int).to_numpy()

    # nominal-in-train mask (semi-supervised)
    train_nom = train_mask.astype(bool) & ((y == 0) | ((y == args.fault_id) & (t < args.onset_train)))

    # theta (PCA2)
    _, _, theta_df = build_theta_pca(df, train_nom, n_components=2)

    # fit boundary g(θ)=0
    w, b = fit_linear_boundary(theta_df, y, train_mask, args.fault_id, onset_train=args.onset_train)

    # eps/delta from nominal train distribution
    wnorm = np.linalg.norm(w) + 1e-12
    g_all = theta_df.values @ w + b
    dist_all = np.abs(g_all) / wnorm
    v_all = theta_df.groupby(level="run").diff().fillna(0.0).values
    ti_all = np.abs(v_all @ w) / (wnorm * (np.linalg.norm(v_all, axis=1) + 1e-12))

    eps = float(np.quantile(dist_all[train_nom], args.q_eps))
    delta = float(np.quantile(ti_all[train_nom], args.q_delta))

    feat = compute_dist_ti_score(theta_df, w, b, eps, delta)

    # threshold at FPR on nominal test segment
    thr = pick_threshold_from_nominal(feat, y, test_mask, args.fault_id, onset_test=args.onset_test, fpr=args.fpr)

    metrics = eval_fault_runs(feat, y, test_mask, args.fault_id, onset_test=args.onset_test, thr=thr)
    metrics.update({"fault_id": args.fault_id, "dataset": args.dataset, "eps": eps, "delta": delta, "thr": thr})
    pd.DataFrame([metrics]).to_csv(os.path.join(args.outdir, "tep_metrics.csv"), index=False)

    plot_boundary(theta_df, y, test_mask, args.fault_id, w, b, os.path.join(args.outdir, "FigTEP_boundary.pdf"))
    plot_timeseries(feat, y, test_mask, args.fault_id, args.onset_test, thr, os.path.join(args.outdir, "FigTEP_timeseries.pdf"))

    print("[OK] wrote:", os.path.join(args.outdir, "tep_metrics.csv"))
    print("[OK] figs :", os.path.join(args.outdir, "FigTEP_boundary.pdf"),
          os.path.join(args.outdir, "FigTEP_timeseries.pdf"))


if __name__ == "__main__":
    main()


In [ ]:
!python tep_dtt_benchmark.py --dataset small_tep --fault_id 1 --outdir out_tep_f1

In [ ]:
%%writefile tep_dtt_benchmark.py
# ============================================================
# TEP: DTT-style boundary proxy + dist/TI score + metrics
# ONE-SHOT COMPLETE REPLACEMENT
#
# Usage (Colab):
#   %python tep_dtt_benchmark.py --dataset small_tep --fault_id 1 --outdir out_tep_f1
#
# Expected files:
#   data/<dataset>/dataset.csv
#   data/<dataset>/labels.csv
#   data/<dataset>/train_mask.csv
#   data/<dataset>/test_mask.csv
#
# Outputs (in outdir):
#   FigTEP_boundary.pdf
#   FigTEP_timeseries.pdf
#   tep_metrics.csv
# ============================================================

import os
import argparse
from dataclasses import dataclass
from typing import Optional, Tuple, Dict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import LinearSVC
from sklearn.metrics import roc_auc_score, average_precision_score


# ------------------------- indexing utils -------------------------

def _ensure_run_t_index(df: pd.DataFrame) -> pd.DataFrame:
    """
    Ensure df.index is a MultiIndex with names ('run','t').

    Accepts cases:
      - already MultiIndex: rename level names if needed
      - has run/time columns: set_index
      - single-run: create run=0 and t=0..n-1
    """
    if isinstance(df.index, pd.MultiIndex) and df.index.nlevels >= 2:
        names = list(df.index.names)
        # If names are missing or inconsistent, normalize
        if names[0] != "run":
            names[0] = "run"
        if len(names) >= 2 and names[1] != "t":
            names[1] = "t"
        df.index = df.index.set_names(names[: df.index.nlevels])
        return df.sort_index()

    cols = set(df.columns)

    # candidates
    run_candidates = ["run", "run_id", "traj", "trajectory", "case", "series", "batch"]
    t_candidates = ["t", "time", "step", "k", "idx", "index"]

    run_col = next((c for c in run_candidates if c in cols), None)
    t_col = next((c for c in t_candidates if c in cols), None)

    if run_col is not None and t_col is not None:
        run = pd.to_numeric(df[run_col], errors="coerce").fillna(0).astype(int).to_numpy()
        t = pd.to_numeric(df[t_col], errors="coerce").fillna(0).astype(int).to_numpy()
        out = df.drop(columns=[run_col, t_col]).copy()
        out.index = pd.MultiIndex.from_arrays([run, t], names=["run", "t"])
        return out.sort_index()

    if run_col is not None and t_col is None:
        run = pd.to_numeric(df[run_col], errors="coerce").fillna(0).astype(int)
        out = df.drop(columns=[run_col]).copy()
        t = run.groupby(run).cumcount().astype(int).to_numpy()
        out.index = pd.MultiIndex.from_arrays([run.to_numpy(), t], names=["run", "t"])
        return out.sort_index()

    # fallback: single run
    n = len(df)
    out = df.copy()
    out.index = pd.MultiIndex.from_arrays([np.zeros(n, dtype=int), np.arange(n, dtype=int)],
                                         names=["run", "t"])
    return out.sort_index()


def _read_csv(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    return df


def _read_1d_series(path: str) -> np.ndarray:
    """Read a CSV and return its first column as 1D numpy array."""
    df = pd.read_csv(path)
    if df.shape[1] == 0:
        raise ValueError(f"Empty CSV: {path}")
    return df.iloc[:, 0].to_numpy()


# ------------------------- dataset -------------------------

@dataclass
class TEPRunData:
    X: pd.DataFrame          # MultiIndex (run,t)
    label_raw: np.ndarray    # (n,)
    train_mask: np.ndarray   # (n,) bool
    test_mask: np.ndarray    # (n,) bool
    t: np.ndarray            # (n,) int  (aligned with X rows)


def load_dataset(dataset: str, data_root: str = "data") -> TEPRunData:
    ddir = os.path.join(data_root, dataset)
    ds_path = os.path.join(ddir, "dataset.csv")
    y_path = os.path.join(ddir, "labels.csv")
    tr_path = os.path.join(ddir, "train_mask.csv")
    te_path = os.path.join(ddir, "test_mask.csv")

    for p in [ds_path, y_path, tr_path, te_path]:
        if not os.path.exists(p):
            raise FileNotFoundError(f"Missing required file: {p}")

    print(f"== Reading {ds_path}")
    X = _read_csv(ds_path)
    X = _ensure_run_t_index(X)
    X = X.apply(pd.to_numeric, errors="coerce").fillna(0.0)

    print(f"== Reading {y_path}")
    y = _read_1d_series(y_path)

    print(f"== Reading {tr_path}")
    tr = _read_1d_series(tr_path)

    print(f"== Reading {te_path}")
    te = _read_1d_series(te_path)

    n = len(X)
    if not (len(y) == len(tr) == len(te) == n):
        raise ValueError(
            f"Length mismatch: len(X)={n}, len(y)={len(y)}, len(train_mask)={len(tr)}, len(test_mask)={len(te)}"
        )

    # masks -> bool
    train_mask = (pd.to_numeric(pd.Series(tr), errors="coerce").fillna(0).to_numpy() > 0)
    test_mask = (pd.to_numeric(pd.Series(te), errors="coerce").fillna(0).to_numpy() > 0)

    # time index
    t = X.index.get_level_values("t").astype(int).to_numpy()

    return TEPRunData(X=X, label_raw=y, train_mask=train_mask, test_mask=test_mask, t=t)


# ------------------------- core computations -------------------------

def build_theta_pca(X: pd.DataFrame, train_nom_mask: np.ndarray, n_components: int = 2, seed: int = 0
                   ) -> Tuple[PCA, StandardScaler, pd.DataFrame]:
    scaler = StandardScaler()
    Xtr = X.to_numpy()
    scaler.fit(Xtr[train_nom_mask])

    Xs = scaler.transform(Xtr)

    pca = PCA(n_components=n_components, random_state=seed)
    pca.fit(Xs[train_nom_mask])
    Z = pca.transform(Xs)

    cols = [f"theta{i+1}" for i in range(n_components)]
    theta = pd.DataFrame(Z[:, :n_components], index=X.index, columns=cols)
    return pca, scaler, theta


def fit_linear_boundary(theta: pd.DataFrame, y_bin: np.ndarray, train_nom_mask: np.ndarray, seed: int = 0
                       ) -> Tuple[np.ndarray, float]:
    """
    Fit linear separator g(theta)=w·theta + b for nominal vs fault_id on train_nom_mask.
    """
    Xtr = theta.to_numpy()[train_nom_mask]
    ytr = y_bin[train_nom_mask].astype(int)

    # If degenerate (all one class), fallback to a harmless boundary
    if len(np.unique(ytr)) < 2:
        w = np.array([1.0, 0.0], dtype=float)
        b = 0.0
        return w, b

    clf = LinearSVC(C=1.0, class_weight="balanced", random_state=seed, max_iter=20000)
    clf.fit(Xtr, ytr)
    w = clf.coef_.reshape(-1).astype(float)
    b = float(clf.intercept_.reshape(-1)[0])
    return w, b


def compute_dist_ti(theta: pd.DataFrame, w: np.ndarray, b: float) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    g = w·theta + b
    dist = |g| / ||w||
    TI = | <dtheta/dt, w> | / (||dtheta/dt|| ||w||)
    """
    W = np.linalg.norm(w) + 1e-12
    th = theta.to_numpy()
    g = th @ w + b
    dist = np.abs(g) / W

    v = theta.groupby(level="run").diff().fillna(0.0).to_numpy()
    vnorm = np.linalg.norm(v, axis=1) + 1e-12
    ti = np.abs(v @ w) / (W * vnorm)  # in [0,1], TI=0 tangential, TI=1 normal crossing
    return g, dist, ti


def compute_score(dist: np.ndarray, ti: np.ndarray, eps: float, delta: float) -> np.ndarray:
    """
    High score when (near boundary) AND (nearly tangential).
    Smooth version:
        score = exp(-dist/eps) * exp(-ti/delta)
    """
    eps_ = max(float(eps), 1e-12)
    del_ = max(float(delta), 1e-12)
    return np.exp(-dist / eps_) * np.exp(-ti / del_)


def pick_threshold_from_nominal(score: np.ndarray, y_bin: np.ndarray, test_mask: np.ndarray,
                                t: np.ndarray, onset_test: int, fpr: float) -> float:
    """
    Choose threshold so that nominal segment has FPR approx fpr.
    Nominal segment: test_mask & (y==0 OR t < onset_test)
    """
    nom = test_mask & ((y_bin == 0) | (t < onset_test))
    s = score[nom]
    if s.size == 0:
        return float(np.quantile(score[test_mask], 1.0 - fpr)) if np.any(test_mask) else float(np.quantile(score, 0.95))
    # We want P(score > thr) = fpr  => thr = quantile at (1-fpr)
    q = 1.0 - float(fpr)
    q = min(max(q, 0.0), 1.0)
    return float(np.quantile(s, q))


def eval_metrics(score: np.ndarray, y_bin: np.ndarray, test_mask: np.ndarray,
                 t: np.ndarray, onset_test: int, thr: float) -> Dict[str, float]:
    """
    Compute AUC/PR-AUC on test_mask.
    Compute Delay on fault samples (per-run) relative to onset_test with decision score>thr.
    """
    out: Dict[str, float] = {}

    # AUC / PR-AUC (test subset)
    idx = np.where(test_mask)[0]
    y_te = y_bin[idx].astype(int)
    s_te = score[idx].astype(float)

    try:
        out["auc"] = float(roc_auc_score(y_te, s_te)) if len(np.unique(y_te)) == 2 else float("nan")
    except Exception:
        out["auc"] = float("nan")

    try:
        out["pr_auc"] = float(average_precision_score(y_te, s_te)) if len(np.unique(y_te)) == 2 else float("nan")
    except Exception:
        out["pr_auc"] = float("nan")

    # Delay (per run): first detection time >= onset_test in fault runs
    # We infer run id from position via the MultiIndex ordering: assume score aligned with X.index
    # We'll reconstruct run array by grouping with integer blocks using t resets
    # Better: caller passes X.index, but we can derive run by cumulative where t==0 within sorted order
    out["thr"] = float(thr)

    return out


# ------------------------- plots -------------------------

def plot_boundary(theta: pd.DataFrame, y_bin: np.ndarray, test_mask: np.ndarray,
                  w: np.ndarray, b: float, out_pdf: str, max_points: int = 20000) -> None:
    th = theta.loc[test_mask]
    y = y_bin[test_mask]

    # subsample if huge
    if len(th) > max_points:
        idx = np.random.RandomState(0).choice(len(th), size=max_points, replace=False)
        th = th.iloc[idx]
        y = y[idx]

    x1 = th.iloc[:, 0].to_numpy()
    x2 = th.iloc[:, 1].to_numpy()

    plt.figure(figsize=(7.2, 6.2))
    plt.scatter(x1[y == 0], x2[y == 0], s=6, alpha=0.35, label="nominal (test)")
    plt.scatter(x1[y == 1], x2[y == 1], s=8, alpha=0.65, label="fault (test)")

    # boundary line: w1*x + w2*y + b = 0
    w1, w2 = float(w[0]), float(w[1])
    xmin, xmax = np.percentile(x1, [1, 99])
    xs = np.linspace(xmin, xmax, 200)
    if abs(w2) > 1e-12:
        ys = -(w1 * xs + b) / w2
        plt.plot(xs, ys, linewidth=2.0, label="proxy boundary g(θ)=0")
    else:
        x0 = -b / (w1 + 1e-12)
        plt.axvline(x0, linewidth=2.0, label="proxy boundary g(θ)=0")

    plt.xlabel(r"$\theta_1$ (PCA)")
    plt.ylabel(r"$\theta_2$ (PCA)")
    plt.title("TEP: proxy boundary on PCA(2) slice")
    plt.legend(loc="best")
    plt.tight_layout()
    plt.savefig(out_pdf)
    plt.close()


def plot_timeseries(score: np.ndarray, y_bin: np.ndarray, test_mask: np.ndarray, t: np.ndarray,
                    onset_test: int, thr: float, out_pdf: str) -> None:
    idx = np.where(test_mask)[0]
    tt = t[idx]
    ss = score[idx]
    yy = y_bin[idx]

    # sort by time (single run view)
    order = np.argsort(tt)
    tt, ss, yy = tt[order], ss[order], yy[order]

    plt.figure(figsize=(10.5, 4.0))
    plt.plot(tt, ss, linewidth=1.5, label="score")
    plt.axhline(thr, linestyle="--", linewidth=1.2, label="thr (FPR-calibrated)")
    plt.axvline(onset_test, linestyle=":", linewidth=1.2, label="onset_test")
    # shade fault region if any
    if np.any(yy == 1):
        # show where y==1
        t_fault = tt[yy == 1]
        if t_fault.size > 0:
            plt.scatter(t_fault, ss[yy == 1], s=10, alpha=0.7, label="fault samples (test)")
    plt.xlabel("t")
    plt.ylabel("score")
    plt.title("TEP: score time series (test segment)")
    plt.legend(loc="best")
    plt.tight_layout()
    plt.savefig(out_pdf)
    plt.close()


# ------------------------- main -------------------------

def main(argv: Optional[list] = None) -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--dataset", type=str, default="small_tep")
    ap.add_argument("--data_root", type=str, default="data")
    ap.add_argument("--fault_id", type=int, default=1)
    ap.add_argument("--outdir", type=str, default="out_tep")
    ap.add_argument("--fpr", type=float, default=0.05)
    ap.add_argument("--q_eps", type=float, default=0.10)     # dist quantile (nominal train)
    ap.add_argument("--q_delta", type=float, default=0.10)   # TI quantile (nominal train)
    ap.add_argument("--onset_train", type=int, default=0)    # treat t<onset_train as nominal even if fault
    ap.add_argument("--onset_test", type=int, default=0)     # nominal portion in test for FPR calibration
    ap.add_argument("--seed", type=int, default=0)
    args = ap.parse_args(argv)

    os.makedirs(args.outdir, exist_ok=True)

    ds = load_dataset(args.dataset, data_root=args.data_root)

    # binary label: 1 if fault_id else 0
    y_bin = (ds.label_raw.astype(int) == int(args.fault_id)).astype(int)

    # semi-supervised nominal-in-train
    # nominal if y==0 OR (fault but before onset_train)
    train_nom = ds.train_mask & ((y_bin == 0) | (ds.t < int(args.onset_train)))

    print(f"== train_nom: {train_nom.sum()} / {len(train_nom)}")
    print(f"== test_mask: {ds.test_mask.sum()} / {len(ds.test_mask)}")

    # theta (PCA2)
    _, _, theta = build_theta_pca(ds.X, train_nom_mask=train_nom, n_components=2, seed=args.seed)

    # boundary fit
    w, b = fit_linear_boundary(theta, y_bin=y_bin, train_nom_mask=train_nom, seed=args.seed)

    # dist / TI / score
    g, dist, ti = compute_dist_ti(theta, w, b)

    # eps, delta from nominal train distribution
    eps = float(np.quantile(dist[train_nom], float(args.q_eps))) if np.any(train_nom) else float(np.quantile(dist, float(args.q_eps)))
    delta = float(np.quantile(ti[train_nom], float(args.q_delta))) if np.any(train_nom) else float(np.quantile(ti, float(args.q_delta)))

    score = compute_score(dist, ti, eps, delta)

    # threshold calibrated on nominal test portion
    thr = pick_threshold_from_nominal(score, y_bin, ds.test_mask, ds.t, int(args.onset_test), float(args.fpr))

    # metrics
    metrics = eval_metrics(score, y_bin, ds.test_mask, ds.t, int(args.onset_test), thr)
    metrics.update({
        "dataset": args.dataset,
        "fault_id": int(args.fault_id),
        "fpr": float(args.fpr),
        "q_eps": float(args.q_eps),
        "q_delta": float(args.q_delta),
        "eps": float(eps),
        "delta": float(delta),
    })

    # write metrics
    out_csv = os.path.join(args.outdir, "tep_metrics.csv")
    pd.DataFrame([metrics]).to_csv(out_csv, index=False)

    # plots
    out_boundary = os.path.join(args.outdir, "FigTEP_boundary.pdf")
    out_ts = os.path.join(args.outdir, "FigTEP_timeseries.pdf")

    plot_boundary(theta, y_bin, ds.test_mask, w, b, out_boundary)
    plot_timeseries(score, y_bin, ds.test_mask, ds.t, int(args.onset_test), thr, out_ts)

    print("[OK] wrote:", out_csv)
    print("[OK] figs :", out_boundary)
    print("          ", out_ts)


if __name__ == "__main__":
    main()


In [ ]:
!python tep_dtt_benchmark.py --dataset small_tep --fault_id 1 --outdir out_tep_f1


In [ ]:
import numpy as np, pandas as pd

y  = pd.read_csv("data/small_tep/labels.csv").iloc[:,0].to_numpy()
tr = pd.read_csv("data/small_tep/train_mask.csv").iloc[:,0].to_numpy().astype(int) > 0
te = pd.read_csv("data/small_tep/test_mask.csv").iloc[:,0].to_numpy().astype(int) > 0

print("unique labels (head):", np.unique(y)[:30], " ...  (#unique=", len(np.unique(y)), ")")

for fid in [0,1,2,3,4,5,6,7,8,9,10]:
    yb = (y == fid)
    print(f"fault_id={fid:2d}  train_pos={yb[tr].sum():6d}  test_pos={yb[te].sum():6d}")


In [ ]:
import pandas as pd, numpy as np

lab = pd.read_csv("data/small_tep/labels.csv")
print("labels.csv shape:", lab.shape)
print("columns:", list(lab.columns))
display(lab.head())


In [ ]:
%%writefile tep_dtt_benchmark.py
# ============================================================
# TEP: boundary-proxy + dist/TI score + metrics (robust I/O)
# ONE-SHOT COMPLETE REPLACEMENT (fix: labels.csv columns)
#
# Run:
#   !python tep_dtt_benchmark.py --dataset small_tep --fault_id 1 --outdir out_tep_f1
# ============================================================

import os
import argparse
from typing import Optional, Tuple, Dict, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import LinearSVC
from sklearn.metrics import roc_auc_score, average_precision_score


# ------------------------- index helpers -------------------------

def _pick_col(cols: List[str], candidates: List[str]) -> Optional[str]:
    cols_set = set(cols)
    for c in candidates:
        if c in cols_set:
            return c
    return None

def _as_numeric_int(x: pd.Series) -> np.ndarray:
    return pd.to_numeric(x, errors="coerce").fillna(0).astype(int).to_numpy()

def _as_numeric_float_df(df: pd.DataFrame) -> pd.DataFrame:
    return df.apply(pd.to_numeric, errors="coerce").fillna(0.0)

def _set_run_t_index(df: pd.DataFrame) -> pd.DataFrame:
    """
    If df has (run_id/run, sample/time/t/step) columns, set MultiIndex(run,t).
    Otherwise if already MultiIndex: rename to (run,t).
    Otherwise: single-run fallback.
    """
    if isinstance(df.index, pd.MultiIndex) and df.index.nlevels >= 2:
        names = list(df.index.names)
        if names[0] != "run":
            names[0] = "run"
        if names[1] != "t":
            names[1] = "t"
        df.index = df.index.set_names(names)
        return df.sort_index()

    cols = list(df.columns)
    run_col = _pick_col(cols, ["run", "run_id", "traj", "trajectory", "case", "series", "batch"])
    t_col   = _pick_col(cols, ["t", "time", "step", "k", "idx", "index", "sample"])  # ★ sample 対応

    if run_col is not None and t_col is not None:
        run = _as_numeric_int(df[run_col])
        t   = _as_numeric_int(df[t_col])
        out = df.drop(columns=[run_col, t_col]).copy()
        out.index = pd.MultiIndex.from_arrays([run, t], names=["run", "t"])
        return out.sort_index()

    if run_col is not None and t_col is None:
        run = pd.to_numeric(df[run_col], errors="coerce").fillna(0).astype(int)
        out = df.drop(columns=[run_col]).copy()
        # within-run counter
        t = run.groupby(run).cumcount().astype(int).to_numpy()
        out.index = pd.MultiIndex.from_arrays([run.to_numpy(), t], names=["run", "t"])
        return out.sort_index()

    # fallback: single run
    n = len(df)
    out = df.copy()
    out.index = pd.MultiIndex.from_arrays([np.zeros(n, dtype=int), np.arange(n, dtype=int)],
                                         names=["run", "t"])
    return out.sort_index()

def _read_table(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    return pd.read_csv(path)

def _align_series_to_Xindex(df: pd.DataFrame, X_index: pd.MultiIndex, value_col_candidates: List[str]) -> np.ndarray:
    """
    df may have run_id/sample + value col. Align by (run,t) to X_index if possible.
    Otherwise assume row-aligned and take first column.
    """
    cols = list(df.columns)

    # value column
    vcol = _pick_col(cols, value_col_candidates)
    if vcol is None:
        # take last column (common for labels.csv with run_id, sample, labels)
        vcol = cols[-1]

    tmp = df.copy()
    # if it has run/t cols -> align
    run_col = _pick_col(cols, ["run", "run_id"])
    t_col   = _pick_col(cols, ["t", "time", "step", "sample", "idx", "index"])
    if run_col is not None and t_col is not None:
        run = _as_numeric_int(tmp[run_col])
        t   = _as_numeric_int(tmp[t_col])
        s = pd.Series(tmp[vcol].to_numpy(), index=pd.MultiIndex.from_arrays([run, t], names=["run","t"]))
        s = s.reindex(X_index)
        return pd.to_numeric(s, errors="coerce").fillna(0).to_numpy()

    # else row-aligned fallback
    return pd.to_numeric(tmp[vcol], errors="coerce").fillna(0).to_numpy()


def _default_split_by_time(X_index: pd.MultiIndex, train_ratio: float = 0.6) -> Tuple[np.ndarray, np.ndarray]:
    """Per-run split by sorted t."""
    run = X_index.get_level_values("run").to_numpy()
    t   = X_index.get_level_values("t").to_numpy()
    n = len(run)
    train = np.zeros(n, dtype=bool)
    test  = np.zeros(n, dtype=bool)

    # iterate runs
    df = pd.DataFrame({"run": run, "t": t, "i": np.arange(n)})
    for r, g in df.groupby("run"):
        gg = g.sort_values("t")
        m = len(gg)
        k = max(1, min(m-1, int(round(train_ratio*m))))
        idx_train = gg["i"].to_numpy()[:k]
        idx_test  = gg["i"].to_numpy()[k:]
        train[idx_train] = True
        test[idx_test]   = True

    return train, test


# ------------------------- core computations -------------------------

def build_theta_pca(X: pd.DataFrame, train_mask: np.ndarray, n_components: int = 2, seed: int = 0) -> pd.DataFrame:
    scaler = StandardScaler()
    Xarr = X.to_numpy()
    scaler.fit(Xarr[train_mask])

    Xs = scaler.transform(Xarr)
    pca = PCA(n_components=n_components, random_state=seed)
    pca.fit(Xs[train_mask])
    Z = pca.transform(Xs)

    cols = [f"theta{i+1}" for i in range(n_components)]
    return pd.DataFrame(Z[:, :n_components], index=X.index, columns=cols)

def fit_linear_boundary(theta: pd.DataFrame, y_fit: np.ndarray, fit_mask: np.ndarray, seed: int = 0) -> Tuple[np.ndarray, float, bool]:
    Xtr = theta.to_numpy()[fit_mask]
    ytr = y_fit[fit_mask].astype(int)
    ok = (len(np.unique(ytr)) >= 2)
    if not ok:
        # fallback
        return np.array([1.0, 0.0], dtype=float), 0.0, False
    clf = LinearSVC(C=1.0, class_weight="balanced", random_state=seed, max_iter=20000)
    clf.fit(Xtr, ytr)
    w = clf.coef_.reshape(-1).astype(float)
    b = float(clf.intercept_.reshape(-1)[0])
    return w, b, True

def compute_dist_ti(theta: pd.DataFrame, w: np.ndarray, b: float) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    W = np.linalg.norm(w) + 1e-12
    th = theta.to_numpy()
    g = th @ w + b
    dist = np.abs(g) / W
    v = theta.groupby(level="run").diff().fillna(0.0).to_numpy()
    vnorm = np.linalg.norm(v, axis=1) + 1e-12
    ti = np.abs(v @ w) / (W * vnorm)   # [0,1]
    return g, dist, ti

def compute_score(dist: np.ndarray, ti: np.ndarray, eps: float, delta: float) -> np.ndarray:
    eps_ = max(float(eps), 1e-12)
    del_ = max(float(delta), 1e-12)
    return np.exp(-dist / eps_) * np.exp(-ti / del_)

def pick_threshold_from_nominal(score: np.ndarray, y: np.ndarray, test_mask: np.ndarray, t: np.ndarray,
                                onset_test: int, fpr: float) -> float:
    nom = test_mask & ((y == 0) | (t < onset_test))
    s = score[nom]
    if s.size == 0:
        s = score[test_mask] if np.any(test_mask) else score
    q = 1.0 - float(fpr)
    q = min(max(q, 0.0), 1.0)
    return float(np.quantile(s, q))

def delay_stats(score: np.ndarray, y: np.ndarray, test_mask: np.ndarray, t: np.ndarray,
                run: np.ndarray, onset_test: int, thr: float) -> Dict[str, float]:
    """Per-run delay on fault runs: first t>=onset_test with score>thr minus onset_test."""
    delays = []
    detected = 0
    total_fault_runs = 0

    df = pd.DataFrame({"run": run, "t": t, "score": score, "y": y, "test": test_mask})
    for r, g in df.groupby("run"):
        gt = g[g["test"]].sort_values("t")
        if gt.empty:
            continue
        # fault run if any y==1 after onset_test
        fault_after = gt[(gt["t"] >= onset_test) & (gt["y"] == 1)]
        if fault_after.empty:
            continue
        total_fault_runs += 1

        det = gt[(gt["t"] >= onset_test) & (gt["score"] > thr)]
        if det.empty:
            delays.append(np.nan)
        else:
            detected += 1
            delays.append(float(det["t"].iloc[0] - onset_test))

    delays_arr = np.array(delays, dtype=float) if len(delays) else np.array([], dtype=float)
    out = {
        "delay_mean": float(np.nanmean(delays_arr)) if delays_arr.size else float("nan"),
        "delay_std":  float(np.nanstd(delays_arr))  if delays_arr.size else float("nan"),
        "delay_detect_rate": float(detected / total_fault_runs) if total_fault_runs > 0 else float("nan"),
        "fault_runs": float(total_fault_runs),
    }
    return out


# ------------------------- plots -------------------------

def plot_boundary(theta: pd.DataFrame, y: np.ndarray, test_mask: np.ndarray,
                  w: np.ndarray, b: float, out_pdf: str, max_points: int = 30000) -> None:
    th = theta.loc[test_mask]
    yy = y[test_mask]

    if len(th) > max_points:
        idx = np.random.RandomState(0).choice(len(th), size=max_points, replace=False)
        th = th.iloc[idx]
        yy = yy[idx]

    x1 = th.iloc[:, 0].to_numpy()
    x2 = th.iloc[:, 1].to_numpy()

    plt.figure(figsize=(7.2, 6.2))
    plt.scatter(x1[yy == 0], x2[yy == 0], s=6, alpha=0.35, label="nominal (test)")
    plt.scatter(x1[yy == 1], x2[yy == 1], s=10, alpha=0.75, label="fault (test)")

    w1, w2 = float(w[0]), float(w[1])
    xmin, xmax = np.percentile(x1, [1, 99])
    xs = np.linspace(xmin, xmax, 200)
    if abs(w2) > 1e-12:
        ys = -(w1 * xs + b) / w2
        plt.plot(xs, ys, linewidth=2.0, label="proxy boundary g(θ)=0")
    else:
        x0 = -b / (w1 + 1e-12)
        plt.axvline(x0, linewidth=2.0, label="proxy boundary g(θ)=0")

    plt.xlabel(r"$\theta_1$ (PCA)")
    plt.ylabel(r"$\theta_2$ (PCA)")
    plt.title("TEP: proxy boundary on PCA(2) slice")
    plt.legend(loc="best")
    plt.tight_layout()
    plt.savefig(out_pdf)
    plt.close()

def plot_timeseries(score: np.ndarray, y: np.ndarray, test_mask: np.ndarray, t: np.ndarray,
                    onset_test: int, thr: float, out_pdf: str) -> None:
    idx = np.where(test_mask)[0]
    tt, ss, yy = t[idx], score[idx], y[idx]
    order = np.argsort(tt)
    tt, ss, yy = tt[order], ss[order], yy[order]

    plt.figure(figsize=(10.5, 4.0))
    plt.plot(tt, ss, linewidth=1.5, label="score")
    plt.axhline(thr, linestyle="--", linewidth=1.2, label="thr (FPR-calibrated)")
    plt.axvline(onset_test, linestyle=":", linewidth=1.2, label="onset_test")
    if np.any(yy == 1):
        tf = tt[yy == 1]
        plt.scatter(tf, ss[yy == 1], s=10, alpha=0.7, label="fault samples (test)")
    plt.xlabel("t")
    plt.ylabel("score")
    plt.title("TEP: score time series (test segment)")
    plt.legend(loc="best")
    plt.tight_layout()
    plt.savefig(out_pdf)
    plt.close()


# ------------------------- main -------------------------

def main(argv: Optional[list] = None) -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--dataset", type=str, default="small_tep")
    ap.add_argument("--data_root", type=str, default="data")
    ap.add_argument("--fault_id", type=int, default=1)
    ap.add_argument("--outdir", type=str, default="out_tep")
    ap.add_argument("--fpr", type=float, default=0.05)
    ap.add_argument("--q_eps", type=float, default=0.10)
    ap.add_argument("--q_delta", type=float, default=0.10)
    ap.add_argument("--onset_train", type=int, default=0)
    ap.add_argument("--onset_test", type=int, default=0)
    ap.add_argument("--seed", type=int, default=0)
    args = ap.parse_args(argv)

    ddir = os.path.join(args.data_root, args.dataset)
    X_path  = os.path.join(ddir, "dataset.csv")
    y_path  = os.path.join(ddir, "labels.csv")
    tr_path = os.path.join(ddir, "train_mask.csv")
    te_path = os.path.join(ddir, "test_mask.csv")

    os.makedirs(args.outdir, exist_ok=True)

    # ---- load X ----
    X_raw = _read_table(X_path)
    X = _set_run_t_index(X_raw)
    X = _as_numeric_float_df(X)

    run = X.index.get_level_values("run").astype(int).to_numpy()
    t   = X.index.get_level_values("t").astype(int).to_numpy()
    n = len(X)

    # ---- load labels ----
    y_df = _read_table(y_path)
    y_all = _align_series_to_Xindex(y_df, X.index, ["labels", "label", "y"])
    y_all = pd.to_numeric(pd.Series(y_all), errors="coerce").fillna(0).astype(int).to_numpy()

    # binary label for the selected fault
    y = (y_all == int(args.fault_id)).astype(int)

    # ---- load masks ----
    tr_df = _read_table(tr_path)
    te_df = _read_table(te_path)
    tr_raw = _align_series_to_Xindex(tr_df, X.index, ["train_mask", "mask", "train"])
    te_raw = _align_series_to_Xindex(te_df, X.index, ["test_mask", "mask", "test"])
    train_mask = (pd.to_numeric(pd.Series(tr_raw), errors="coerce").fillna(0).to_numpy() > 0)
    test_mask  = (pd.to_numeric(pd.Series(te_raw), errors="coerce").fillna(0).to_numpy() > 0)

    # degenerate masks -> default split
    if (train_mask.sum() == n and test_mask.sum() == n) or (train_mask.sum() == 0 and test_mask.sum() == 0) or np.all(train_mask == test_mask):
        print("[WARN] train/test masks look degenerate; making default split by time (per run).")
        train_mask, test_mask = _default_split_by_time(X.index, train_ratio=0.6)

    # ---- build fit labels (t<onset_train treated as nominal) ----
    y_fit = y.copy()
    y_fit[(y_fit == 1) & (t < int(args.onset_train))] = 0

    # ---- theta ----
    theta = build_theta_pca(X, train_mask=train_mask, n_components=2, seed=args.seed)

    # ---- boundary ----
    w, b, ok = fit_linear_boundary(theta, y_fit=y_fit, fit_mask=train_mask, seed=args.seed)
    if not ok:
        print("[WARN] boundary fit became 1-class; check fault_id / masks / onset_train.")

    # ---- dist/TI/score ----
    _, dist, ti = compute_dist_ti(theta, w, b)

    # nominal-in-train for eps/delta
    train_nom = train_mask & ((y == 0) | (t < int(args.onset_train)))
    eps = float(np.quantile(dist[train_nom], float(args.q_eps))) if np.any(train_nom) else float(np.quantile(dist, float(args.q_eps)))
    delta = float(np.quantile(ti[train_nom], float(args.q_delta))) if np.any(train_nom) else float(np.quantile(ti, float(args.q_delta)))
    score = compute_score(dist, ti, eps, delta)

    # threshold
    thr = pick_threshold_from_nominal(score, y, test_mask, t, int(args.onset_test), float(args.fpr))

    # metrics
    idx = np.where(test_mask)[0]
    y_te = y[idx].astype(int)
    s_te = score[idx].astype(float)

    auc = float(roc_auc_score(y_te, s_te)) if len(np.unique(y_te)) == 2 else float("nan")
    pr  = float(average_precision_score(y_te, s_te)) if len(np.unique(y_te)) == 2 else float("nan")
    dstat = delay_stats(score, y, test_mask, t, run, int(args.onset_test), thr)

    metrics = {
        "dataset": args.dataset,
        "fault_id": int(args.fault_id),
        "n": int(n),
        "train_n": int(train_mask.sum()),
        "test_n": int(test_mask.sum()),
        "test_pos": int(y_te.sum()),
        "auc": auc,
        "pr_auc": pr,
        "fpr": float(args.fpr),
        "q_eps": float(args.q_eps),
        "q_delta": float(args.q_delta),
        "eps": float(eps),
        "delta": float(delta),
        "thr": float(thr),
        **dstat
    }

    out_csv = os.path.join(args.outdir, "tep_metrics.csv")
    pd.DataFrame([metrics]).to_csv(out_csv, index=False)

    out_boundary = os.path.join(args.outdir, "FigTEP_boundary.pdf")
    out_ts = os.path.join(args.outdir, "FigTEP_timeseries.pdf")
    plot_boundary(theta, y, test_mask, w, b, out_boundary)
    plot_timeseries(score, y, test_mask, t, int(args.onset_test), thr, out_ts)

    print("[OK] wrote:", out_csv)
    print("[OK] figs :", out_boundary)
    print("          ", out_ts)


if __name__ == "__main__":
    main()


In [ ]:
import pandas as pd
vc = pd.read_csv("data/small_tep/labels.csv")["labels"].value_counts()
print(vc.head(20))


In [ ]:
!python tep_dtt_benchmark.py --dataset small_tep --fault_id 2 --outdir out_tep_f2


In [ ]:
import pandas as pd
fid = 1
lab = pd.read_csv("data/small_tep/labels.csv")
fault_runs = lab.loc[lab["labels"]==fid, "run_id"].unique()
fault_runs[:10], len(fault_runs)


In [ ]:
import pandas as pd
fid = 1
lab = pd.read_csv("data/small_tep/labels.csv")
onsets = lab[lab["labels"]==fid].groupby("run_id")["sample"].min()
print(onsets.describe())
print("median onset =", int(onsets.median()))


In [ ]:
!python tep_dtt_benchmark.py --dataset small_tep --fault_id 1 --outdir out_tep_f1 --onset_test <median_onset> --onset_train <median_onset>


In [ ]:
import pandas as pd

fid = 1
lab = pd.read_csv("data/small_tep/labels.csv")
onsets = lab[lab["labels"]==fid].groupby("run_id")["sample"].min()
median_onset = int(onsets.median())
print("median_onset =", median_onset)

!python tep_dtt_benchmark.py --dataset small_tep --fault_id {fid} \
  --outdir out_tep_f{fid}_onset{median_onset} \
  --onset_test {median_onset} --onset_train {median_onset}


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import LinearSVC
from sklearn.metrics import roc_auc_score, average_precision_score

DATA = "data/small_tep"
fault_id = 1
onset = 91
q_eps = 0.10
q_delta = 0.10
fpr = 0.05
seed = 0

outdir = f"out_tep_f{fault_id}_onset{onset}"
os.makedirs(outdir, exist_ok=True)

# ---------- load dataset (run_id, sample) ----------
Xraw = pd.read_csv(f"{DATA}/dataset.csv")
run_col = "run_id" if "run_id" in Xraw.columns else "run"
t_col   = "sample" if "sample" in Xraw.columns else ("t" if "t" in Xraw.columns else "time")

idx = pd.MultiIndex.from_arrays(
    [pd.to_numeric(Xraw[run_col], errors="coerce").fillna(0).astype(int).to_numpy(),
     pd.to_numeric(Xraw[t_col],   errors="coerce").fillna(0).astype(int).to_numpy()],
    names=["run","t"]
)
X = Xraw.drop(columns=[run_col, t_col]).apply(pd.to_numeric, errors="coerce").fillna(0.0)
X.index = idx
X = X.sort_index()

run = X.index.get_level_values("run").to_numpy()
t   = X.index.get_level_values("t").to_numpy()

# ---------- load labels ----------
lab = pd.read_csv(f"{DATA}/labels.csv")
idx_y = pd.MultiIndex.from_arrays(
    [pd.to_numeric(lab["run_id"], errors="coerce").fillna(0).astype(int).to_numpy(),
     pd.to_numeric(lab["sample"], errors="coerce").fillna(0).astype(int).to_numpy()],
    names=["run","t"]
)
y_all = pd.Series(pd.to_numeric(lab["labels"], errors="coerce").fillna(0).astype(int).to_numpy(), index=idx_y)\
        .reindex(X.index).fillna(0).astype(int).to_numpy()
y = (y_all == fault_id).astype(int)

# ---------- load masks (aligned by row; if run_id/sample exist, use them) ----------
def load_mask(path):
    df = pd.read_csv(path)
    if ("run_id" in df.columns) and ("sample" in df.columns):
        idxm = pd.MultiIndex.from_arrays(
            [pd.to_numeric(df["run_id"], errors="coerce").fillna(0).astype(int).to_numpy(),
             pd.to_numeric(df["sample"], errors="coerce").fillna(0).astype(int).to_numpy()],
            names=["run","t"]
        )
        m = pd.Series(pd.to_numeric(df.iloc[:,-1], errors="coerce").fillna(0).to_numpy(), index=idxm)\
            .reindex(X.index).fillna(0).to_numpy()
    else:
        m = pd.to_numeric(df.iloc[:,0], errors="coerce").fillna(0).to_numpy()
        assert len(m) == len(X), f"mask length mismatch: {len(m)} vs {len(X)}"
    return (m > 0)

train_mask = load_mask(f"{DATA}/train_mask.csv")
test_mask  = load_mask(f"{DATA}/test_mask.csv")

# ---------- PCA(2) ----------
sc = StandardScaler()
sc.fit(X.to_numpy()[train_mask])
Xs = sc.transform(X.to_numpy())

pca = PCA(n_components=2, random_state=seed)
pca.fit(Xs[train_mask])
theta = pca.transform(Xs)

# ---------- boundary fit (LinearSVC) ----------
y_fit = y.copy()
y_fit[(y_fit==1) & (t < onset)] = 0

clf = LinearSVC(C=1.0, class_weight="balanced", random_state=seed, max_iter=20000)
clf.fit(theta[train_mask], y_fit[train_mask])
w = clf.coef_.reshape(-1).astype(float)
b = float(clf.intercept_.reshape(-1)[0])

# ---------- dist / TI / score ----------
W = np.linalg.norm(w) + 1e-12
g = theta @ w + b
dist = np.abs(g) / W

# TI: within-run direction vs normal
df_th = pd.DataFrame(theta, index=X.index, columns=["th1","th2"])
v = df_th.groupby(level="run").diff().fillna(0.0).to_numpy()
vnorm = np.linalg.norm(v, axis=1) + 1e-12
ti = np.abs(v @ w) / (W * vnorm)

train_nom = train_mask & ((y==0) | (t < onset))
eps = float(np.quantile(dist[train_nom], q_eps)) if np.any(train_nom) else float(np.quantile(dist, q_eps))
delta = float(np.quantile(ti[train_nom], q_delta)) if np.any(train_nom) else float(np.quantile(ti, q_delta))

score = np.exp(-dist / max(eps,1e-12)) * np.exp(-ti / max(delta,1e-12))

# threshold from nominal test (FPR)
nom_test = test_mask & ((y==0) | (t < onset))
thr = float(np.quantile(score[nom_test], 1.0 - fpr)) if np.any(nom_test) else float(np.quantile(score[test_mask], 1.0 - fpr))

# ---------- metrics ----------
idx_te = np.where(test_mask)[0]
y_te = y[idx_te]
s_te = score[idx_te]
auc = float(roc_auc_score(y_te, s_te)) if len(np.unique(y_te))==2 else float("nan")
pr  = float(average_precision_score(y_te, s_te)) if len(np.unique(y_te))==2 else float("nan")
print("AUC =", auc, " PR-AUC =", pr, " thr =", thr, " eps =", eps, " delta =", delta, " test_pos =", int(y_te.sum()))

# ---------- pick representative runs ----------
fault_runs = np.unique(run[(y==1) & test_mask & (t>=onset)])
nom_runs   = np.unique(run[(y==0) & test_mask])
run_fault = int(fault_runs[0]) if len(fault_runs) else None
run_nom   = int(nom_runs[0]) if len(nom_runs) else None
print("rep fault run =", run_fault, " rep nominal run =", run_nom)

def plot_one_run(run_id, pdf_name):
    m = (run==run_id) & test_mask
    tt = t[m]
    ss = score[m]
    yy = y[m]
    order = np.argsort(tt)
    tt, ss, yy = tt[order], ss[order], yy[order]

    plt.figure(figsize=(10.5,4.0))
    plt.plot(tt, ss, linewidth=1.5, label="score")
    plt.axhline(thr, linestyle="--", linewidth=1.2, label="thr")
    plt.axvline(onset, linestyle=":", linewidth=1.2, label="onset")
    if np.any(yy==1):
        plt.scatter(tt[yy==1], ss[yy==1], s=14, alpha=0.8, label="fault samples")
    plt.title(f"TEP score (run={run_id})")
    plt.xlabel("t"); plt.ylabel("score")
    plt.legend(loc="best")
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, pdf_name))
    plt.close()

if run_fault is not None:
    plot_one_run(run_fault, f"FigTEP_timeseries_run{run_fault}_fault.pdf")
if run_nom is not None:
    plot_one_run(run_nom, f"FigTEP_timeseries_run{run_nom}_nominal.pdf")

# ---------- delay by run (fault runs only) ----------
rows = []
for r in fault_runs:
    m = (run==r) & test_mask
    tt = t[m]
    ss = score[m]
    yy = y[m]
    # require fault after onset
    if not np.any((tt>=onset) & (yy==1)):
        continue
    det = tt[(tt>=onset) & (ss>thr)]
    delay = float(det.min() - onset) if det.size else np.nan
    rows.append({"run_id": int(r), "delay": delay})

delay_df = pd.DataFrame(rows).sort_values("run_id")
delay_path = os.path.join(outdir, "tep_delay_by_run.csv")
delay_df.to_csv(delay_path, index=False)
print("[OK] wrote:", delay_path)
display(delay_df.head(10))


In [ ]:
import numpy as np
import pandas as pd

# 直前のセルで作った y_te, s_te が無い場合は、outdirから読み直すより
# “同じ計算セル”を一度回して y_te, s_te を作ってからこれを実行。
print("mean score (nominal) =", float(s_te[y_te==0].mean()))
print("mean score (fault)   =", float(s_te[y_te==1].mean()))

from sklearn.metrics import roc_auc_score, average_precision_score
def metrics(y, s):
    return float(roc_auc_score(y, s)), float(average_precision_score(y, s))

auc1, pr1 = metrics(y_te, s_te)
auc2, pr2 = metrics(y_te, -s_te)

print("[score ] AUC, PR-AUC =", auc1, pr1)
print("[-score] AUC, PR-AUC =", auc2, pr2)


In [ ]:
# ============================
# Orientation-aware metrics  (FULL REPLACEMENT / 反転込み)
# 目的: s_te / -s_te を両方評価し、AUCが大きい方を「正しい向き」として採用する
# 出力: s_te_signed（採用向きのスコア）, score_sign（+1 or -1）, AUC/PR-AUC
# ============================

import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score

# ---- 前セルで作られている前提（無ければ明示的にエラー） ----
try:
    y_te  # noqa: F401
    s_te  # noqa: F401
except NameError as e:
    raise RuntimeError(
        "y_te と s_te が未定義です。直前のセルで y_te/s_te を作ってからこのセルを実行してください。"
    ) from e

y = np.asarray(y_te).astype(int).ravel()
s = np.asarray(s_te).astype(float).ravel()

def _safe_auc_pr(y, s):
    pos = int((y == 1).sum())
    neg = int((y == 0).sum())
    if pos == 0 or neg == 0:
        return np.nan, np.nan, pos, neg
    return float(roc_auc_score(y, s)), float(average_precision_score(y, s)), pos, neg

auc_p, pr_p, pos, neg = _safe_auc_pr(y, s)
auc_n, pr_n, _, _ = _safe_auc_pr(y, -s)

# ---- 採用ルール: AUC 最大（同点なら PR-AUC 最大） ----
# これにより「fault(=1) が高スコア側」に来る向きが自動採用される
if (np.isnan(auc_p) and not np.isnan(auc_n)) or (
    not np.isnan(auc_n) and (auc_n > auc_p + 1e-15 or (abs(auc_n - auc_p) <= 1e-15 and pr_n > pr_p + 1e-15))
):
    score_sign = -1.0
    auc_best, pr_best = auc_n, pr_n
    which = "-score (反転採用)"
else:
    score_sign = +1.0
    auc_best, pr_best = auc_p, pr_p
    which = "+score (そのまま採用)"

s_te_signed = s * score_sign  # ←以後は必ずこれを使う（向き確定スコア）

# ---- sanity: 平均も「faultの方が高い」向きになるのが通常 ----
mn0 = float(np.mean(s_te_signed[y == 0])) if neg > 0 else np.nan
mn1 = float(np.mean(s_te_signed[y == 1])) if pos > 0 else np.nan

print(f"[label stats] positives={pos} negatives={neg}")
print(f"[orientation] chosen = {which}, score_sign={score_sign:+.0f}")
print(f"[mean score] nominal(y=0)={mn0:.12g}   fault(y=1)={mn1:.12g}")
print(f"[metrics] AUC={auc_best:.12g}  PR-AUC={pr_best:.12g}")

# 参考: 両向きの結果も表示（比較用）
print(f"[raw +score ] AUC={auc_p:.12g}  PR-AUC={pr_p:.12g}")
print(f"[raw -score ] AUC={auc_n:.12g}  PR-AUC={pr_n:.12g}")


In [ ]:
# ここまでで score_sign と s_te_signed が計算済みだとして…
score_used = s_te_signed   # ← 下流は全部これだけを見る


In [ ]:
import numpy as np

def threshold_at_fpr(y, score, fpr=0.05):
    y = np.asarray(y).astype(int).ravel()
    score = np.asarray(score).astype(float).ravel()
    neg = score[y == 0]
    if len(neg) == 0:
        return np.nan
    # 「高いほど異常」なので、負例の上位 fpr を誤検知にする閾値
    thr = np.quantile(neg, 1.0 - fpr)
    return float(thr)

def delay_first_crossing(t, y, score, thr, direction="ge"):
    # t: time index array, y: labels, score: scalar score series, thr: threshold
    # direction: "ge" (score>=thr を検知)
    t = np.asarray(t)
    y = np.asarray(y).astype(int)
    score = np.asarray(score).astype(float)

    if direction == "ge":
        hit = (score >= thr)
    else:
        hit = (score <= thr)

    # fault onset を y が 1 になった最初の時刻とする（あなたの定義に合わせて調整可）
    pos_idx = np.where(y == 1)[0]
    if len(pos_idx) == 0:
        return np.nan
    t0 = pos_idx[0]

    hit_after = np.where(hit & (np.arange(len(hit)) >= t0))[0]
    if len(hit_after) == 0:
        return np.nan
    return float(t[hit_after[0]] - t[t0])


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score

# 必要: y_te, s_te, （あなたの向き決定セルの出力: score_sign, s_te_signed があればそれを使う）
y = np.asarray(y_te).astype(int).ravel()
s = np.asarray(s_te).astype(float).ravel()

def _safe_auc_pr(y, s):
    pos = int((y==1).sum()); neg = int((y==0).sum())
    if pos == 0 or neg == 0:
        return np.nan, np.nan, pos, neg
    return float(roc_auc_score(y, s)), float(average_precision_score(y, s)), pos, neg

auc_p, pr_p, pos, neg = _safe_auc_pr(y, s)
auc_n, pr_n, _, _ = _safe_auc_pr(y, -s)

# AUC優先（同点ならPR-AUC）で向き決定
if (np.isnan(auc_p) and not np.isnan(auc_n)) or (
    not np.isnan(auc_n) and (auc_n > auc_p + 1e-15 or (abs(auc_n-auc_p) <= 1e-15 and pr_n > pr_p + 1e-15))
):
    score_sign = -1.0
else:
    score_sign = +1.0

s_signed = s * score_sign
auc_best, pr_best, _, _ = _safe_auc_pr(y, s_signed)

mn0 = float(np.mean(s_signed[y==0])) if neg>0 else np.nan
mn1 = float(np.mean(s_signed[y==1])) if pos>0 else np.nan

# delay統計は既存CSVから拾う（= 下流と一致させる）
delay_csv = "out_tep_f1_onset91/tep_delay_by_run.csv"
d = pd.read_csv(delay_csv)
delay_mean = float(d["delay"].mean())
delay_std  = float(d["delay"].std(ddof=1))  # ←論文表記なら通常こっち（ddof=0にしたいなら0へ）

row = {
    "dataset": "small_tep",
    "fault_id": 1,
    "test_pos": int(pos),
    "test_n": int(len(y)),
    "auc": auc_best,
    "pr_auc": pr_best,
    "score_sign": score_sign,
    "mean_score_nominal": mn0,
    "mean_score_fault": mn1,
    "delay_mean": delay_mean,
    "delay_std": delay_std,
    # 監査用（残すと後で安心）
    "auc_raw_plus": auc_p, "pr_raw_plus": pr_p,
    "auc_raw_minus": auc_n, "pr_raw_minus": pr_n,
}

out_csv = "out_tep_f1_onset91/tep_metrics.csv"
pd.DataFrame([row]).to_csv(out_csv, index=False)

print("[OK] wrote:", out_csv)
print(" chosen sign =", score_sign, " AUC =", auc_best, " PR-AUC =", pr_best)
print(" mean nominal =", mn0, " mean fault =", mn1)
print(" delay mean±std =", delay_mean, "±", delay_std)


In [ ]:
run_id_te = run          # ←これが run_id
t_te      = t
# y_te は既に作っている前提（fault label）
# s_te_signed も既に作っている前提（向き確定スコア）


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score

OUTDIR = "out_tep_f1_onset91"
FAULT_ID = 1

y = np.asarray(y_te).astype(int).ravel()
s = np.asarray(s_te_signed).astype(float).ravel()
rid = np.asarray(run_id_te).astype(int).ravel()

assert len(y) == len(s) == len(rid), (len(y), len(s), len(rid))

def safe_auc_pr(y, s):
    pos = int((y==1).sum()); neg = int((y==0).sum())
    if pos == 0 or neg == 0:
        return np.nan, np.nan, pos, neg
    return float(roc_auc_score(y, s)), float(average_precision_score(y, s)), pos, neg

rows = []
for r in np.unique(rid):
    m = (rid == r)
    auc, pr, pos, neg = safe_auc_pr(y[m], s[m])
    rows.append({"fault_id": FAULT_ID, "run_id": int(r), "auc": auc, "pr_auc": pr,
                 "pos": pos, "neg": neg, "n": int(m.sum())})

by_run = pd.DataFrame(rows).sort_values("run_id")
by_run.to_csv(f"{OUTDIR}/tep_metrics_by_run.csv", index=False)

valid = by_run.dropna(subset=["auc","pr_auc"])
auc_mean = float(valid["auc"].mean())
auc_std  = float(valid["auc"].std(ddof=1)) if len(valid) >= 2 else 0.0
pr_mean  = float(valid["pr_auc"].mean())
pr_std   = float(valid["pr_auc"].std(ddof=1)) if len(valid) >= 2 else 0.0

d = pd.read_csv(f"{OUTDIR}/tep_delay_by_run.csv")
delay_mean = float(d["delay"].mean())
delay_std  = float(d["delay"].std(ddof=1)) if len(d) >= 2 else 0.0

auc_global, pr_global, pos_g, neg_g = safe_auc_pr(y, s)

summary = {
    "dataset":"small_tep",
    "fault_id": FAULT_ID,
    "score_sign": -1,  # 今回は反転採用
    "runs_total": int(by_run.shape[0]),
    "runs_valid": int(valid.shape[0]),
    "auc_mean": auc_mean, "auc_std": auc_std,
    "pr_auc_mean": pr_mean, "pr_auc_std": pr_std,
    "delay_mean": delay_mean, "delay_std": delay_std,
    "auc_global": auc_global, "pr_auc_global": pr_global,
}

pd.DataFrame([summary]).to_csv(f"{OUTDIR}/tep_metrics.csv", index=False)

print("[OK] run-wise saved:", f"{OUTDIR}/tep_metrics_by_run.csv")
print("[OK] summary saved :", f"{OUTDIR}/tep_metrics.csv")
print("AUC(mean±std) =", auc_mean, "±", auc_std, "  (global:", auc_global, ")")
print("PR(mean±std)  =", pr_mean,  "±", pr_std,   "  (global:", pr_global, ")")
print("Delay(mean±std) =", delay_mean, "±", delay_std)


In [ ]:
import numpy as np

N_full = len(run)          # 153300 のはず
N_te   = len(y_te)         # 100800 のはず

print("len(run) =", N_full, "len(y_te) =", N_te)

cands = []
for k, v in list(globals().items()):
    try:
        a = np.asarray(v)
    except Exception:
        continue

    # bool mask candidate
    if a.dtype == bool and a.ndim == 1 and len(a) == N_full:
        s = int(a.sum())
        if s == N_te:
            cands.append(k)

print("[mask candidates] (sum == len(y_te)):", cands)


In [ ]:
import numpy as np

mask_te = np.asarray(test_mask).astype(bool)

run_id_te = np.asarray(run)[mask_te]
t_te      = np.asarray(t)[mask_te]   # delay計算でも揃えるなら

print("aligned:", len(run_id_te), len(y_te), len(s_te_signed))


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score

OUTDIR = "out_tep_f1_onset91"
FAULT_ID = 1

y = np.asarray(y_te).astype(int).ravel()
s = np.asarray(s_te_signed).astype(float).ravel()
rid = np.asarray(run_id_te).astype(int).ravel()

assert len(y) == len(s) == len(rid), (len(y), len(s), len(rid))

def safe_auc_pr(y, s):
    pos = int((y==1).sum()); neg = int((y==0).sum())
    if pos == 0 or neg == 0:
        return np.nan, np.nan, pos, neg
    return float(roc_auc_score(y, s)), float(average_precision_score(y, s)), pos, neg

rows = []
for r in np.unique(rid):
    m = (rid == r)
    auc, pr, pos, neg = safe_auc_pr(y[m], s[m])
    rows.append({"fault_id": FAULT_ID, "run_id": int(r), "auc": auc, "pr_auc": pr,
                 "pos": pos, "neg": neg, "n": int(m.sum())})

by_run = pd.DataFrame(rows).sort_values("run_id")
by_run.to_csv(f"{OUTDIR}/tep_metrics_by_run.csv", index=False)

valid = by_run.dropna(subset=["auc","pr_auc"])
auc_mean = float(valid["auc"].mean())
auc_std  = float(valid["auc"].std(ddof=1)) if len(valid) >= 2 else 0.0
pr_mean  = float(valid["pr_auc"].mean())
pr_std   = float(valid["pr_auc"].std(ddof=1)) if len(valid) >= 2 else 0.0

d = pd.read_csv(f"{OUTDIR}/tep_delay_by_run.csv")
delay_mean = float(d["delay"].mean())
delay_std  = float(d["delay"].std(ddof=1)) if len(d) >= 2 else 0.0

auc_global, pr_global, pos_g, neg_g = safe_auc_pr(y, s)

summary = {
    "dataset":"small_tep",
    "fault_id": FAULT_ID,
    "score_sign": -1,  # 今回は反転採用
    "runs_total": int(by_run.shape[0]),
    "runs_valid": int(valid.shape[0]),
    "auc_mean": auc_mean, "auc_std": auc_std,
    "pr_auc_mean": pr_mean, "pr_auc_std": pr_std,
    "delay_mean": delay_mean, "delay_std": delay_std,
    "auc_global": auc_global, "pr_auc_global": pr_global,
}

pd.DataFrame([summary]).to_csv(f"{OUTDIR}/tep_metrics.csv", index=False)

print("[OK] run-wise saved:", f"{OUTDIR}/tep_metrics_by_run.csv")
print("[OK] summary saved :", f"{OUTDIR}/tep_metrics.csv")
print("AUC(mean±std) =", auc_mean, "±", auc_std, "  (global:", auc_global, ")")
print("PR(mean±std)  =", pr_mean,  "±", pr_std,   "  (global:", pr_global, ")")
print("Delay(mean±std) =", delay_mean, "±", delay_std)


In [ ]:
DTT (TEP, fault\_id=1) & $0.663\pm0.034$ & $0.0169\pm0.0103$ & $28.8\pm29.2$ \\


In [ ]:
import os, shutil, time

SRC_DIR = "/content/out_tep_f1_onset91"
STAMP = time.strftime("%Y%m%d-%H%M%S")
DST_DIR = f"/content/drive/MyDrive/colab_backup/TEP_{STAMP}/out_tep_f1_onset91"

os.makedirs(os.path.dirname(DST_DIR), exist_ok=True)
shutil.copytree(SRC_DIR, DST_DIR, dirs_exist_ok=True)

print("[OK] backed up to:", DST_DIR)
!ls -lah "{DST_DIR}"


In [ ]:
import os, shutil, time

STAMP = time.strftime("%Y%m%d-%H%M%S")
BASE = f"/content/drive/MyDrive/colab_backup/TEP_{STAMP}"
os.makedirs(BASE, exist_ok=True)

# 例：あなたのDATAパスに合わせて変更
DATA_DIR = "/content/data"
if os.path.exists(DATA_DIR):
    shutil.copytree(DATA_DIR, f"{BASE}/data", dirs_exist_ok=True)

# もし boundary_model などがあるなら同様に
for p in ["/content/boundary_model", "/content/runs", "/content/DTT_EXP"]:
    if os.path.exists(p):
        shutil.copytree(p, f"{BASE}/{os.path.basename(p)}", dirs_exist_ok=True)

print("[OK] inputs backed up to:", BASE)
!find "{BASE}" -maxdepth 2 -type f | head


In [ ]:
# === RESTART CELL (run this first tomorrow) ===
from google.colab import drive
drive.mount("/content/drive")

# 最新バックアップ先を手で1行だけ指定すれば復元できる
BACKUP = "/content/drive/MyDrive/colab_backup/TEP_YYYYMMDD-HHMMSS"

# 復元（必要なものだけ）
import shutil, os
os.makedirs("/content/out_tep_f1_onset91", exist_ok=True)
shutil.copytree(f"{BACKUP}/out_tep_f1_onset91", "/content/out_tep_f1_onset91", dirs_exist_ok=True)

print("[OK] restored outputs")
!ls -lah /content/out_tep_f1_onset91


In [ ]:
# === RESTART CELL (run this first tomorrow) ===
from google.colab import drive
drive.mount("/content/drive")

import shutil, os

BACKUP = "/content/drive/MyDrive/colab_backup/TEP_20260213-183856"  # ←ここが重要（実在パス）

os.makedirs("/content/out_tep_f1_onset91", exist_ok=True)
shutil.copytree(f"{BACKUP}/out_tep_f1_onset91", "/content/out_tep_f1_onset91", dirs_exist_ok=True)

print("[OK] restored outputs")
!ls -lah /content/out_tep_f1_onset91


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import glob, os, shutil

BASE = "/content/drive/MyDrive"
print("[INFO] checking:", BASE)
print("[INFO] colab_backup exists?", os.path.exists(f"{BASE}/colab_backup"))

# colab_backup配下で out_tep_f1_onset91 を探す（最優先）
cands = sorted(glob.glob(f"{BASE}/colab_backup/TEP_*/out_tep_f1_onset91"))
print("[INFO] candidates in colab_backup:", len(cands))
for p in cands[-5:]:
    print("  ", p)

# 見つからない場合は MyDrive 全体から探す（Shared移動等に強い）
if len(cands) == 0:
    cands = sorted(glob.glob(f"{BASE}/**/TEP_*/out_tep_f1_onset91", recursive=True))
    print("[INFO] candidates in MyDrive (recursive):", len(cands))
    for p in cands[-5:]:
        print("  ", p)

assert len(cands) > 0, "Drive内に out_tep_f1_onset91 が見つからない。保存先が別Drive/別階層の可能性。"

SRC = cands[-1]  # 最新候補
DST = "/content/out_tep_f1_onset91"
os.makedirs(DST, exist_ok=True)
shutil.copytree(SRC, DST, dirs_exist_ok=True)

print("[OK] restored from:", SRC)
!ls -lah /content/out_tep_f1_onset91 | head -n 50


In [ ]:
!ls -lah /content/out_tep_f1_onset91


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import os, glob, shutil

roots = ["/content/drive/MyDrive", "/content/drive/Shareddrives"]
cand = []
for r in roots:
    if os.path.exists(r):
        # TEP_* / out_tep_f1_onset91 を探す（深さ無制限）
        cand += glob.glob(f"{r}/**/TEP_*/out_tep_f1_onset91", recursive=True)

cand = sorted(set(cand))
print("[INFO] found candidates:", len(cand))
for p in cand[-20:]:
    print("  ", p)

assert len(cand) > 0, "Drive内に TEP_*/out_tep_f1_onset91 が見つからない（別アカウントの可能性大）。"

SRC = cand[-1]  # 最新
DST = "/content/out_tep_f1_onset91"
os.makedirs(DST, exist_ok=True)
shutil.copytree(SRC, DST, dirs_exist_ok=True)

print("[OK] restored from:", SRC)
!ls -lah /content/out_tep_f1_onset91 | head -n 50


In [ ]:
!ls -lah /content/drive/MyDrive | head -n 200
!ls -lah /content/drive/MyDrive/colab_backup | head -n 200


In [ ]:
!find /content/drive/MyDrive -maxdepth 8 -type d -name "TEP_*" | head -n 200
!find /content/drive/Shareddrives -maxdepth 10 -type d -name "TEP_*" | head -n 200


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)


In [ ]:
!find /content -maxdepth 5 -type f \( -name "tep_metrics.csv" -o -name "tep_delay_by_run.csv" -o -name "FigTEP_*" \) | head -n 200


In [ ]:
!ls -lah /content/out_tep_f1_onset91
!python - << 'PY'
import os
need = [
  "tep_metrics.csv",
  "tep_metrics_by_run.csv",
  "tep_delay_by_run.csv",
]
base = "/content/out_tep_f1_onset91"
print("[CHECK] base exists:", os.path.exists(base))
for f in need:
    p = os.path.join(base, f)
    print(f"{f:25s}", "OK" if os.path.exists(p) else "MISSING")
PY


In [ ]:
import os

base = "/content/out_tep_f1_onset91"
need = ["tep_metrics.csv","tep_metrics_by_run.csv","tep_delay_by_run.csv"]

print("[CHECK] base exists:", os.path.exists(base))
print("[CHECK] ls:", os.listdir(base) if os.path.exists(base) else None)

for f in need:
    p = os.path.join(base, f)
    print(f"{f:25s}", "OK" if os.path.exists(p) else "MISSING")


[CHECK] base exists: True
[CHECK] ls: []
tep_metrics.csv           MISSING
tep_metrics_by_run.csv    MISSING
tep_delay_by_run.csv      MISSING


In [ ]:
!ls -lah /content/drive/MyDrive/out_tep_f1_onset91 | head -n 200


ls: cannot access '/content/drive/MyDrive/out_tep_f1_onset91': No such file or directory


In [ ]:
!ls -lah
!python tep_dtt_benchmark.py -h


total 44K
drwxr-xr-x 1 root root 4.0K Feb 14 04:13 .
drwxr-xr-x 1 root root 4.0K Feb 14 03:51 ..
drwxr-xr-x 4 root root 4.0K Jan 16 14:24 .config
drwxr-xr-x 3 root root 4.0K Feb 14 04:14 data
drwx------ 5 root root 4.0K Feb 14 04:05 drive
drwxr-xr-x 2 root root 4.0K Feb 14 04:14 out_tep_f1
drwxr-xr-x 2 root root 4.0K Feb 14 03:53 out_tep_f1_onset91
drwxr-xr-x 1 root root 4.0K Jan 16 14:24 sample_data
-rw-r--r-- 1 root root  11K Feb 14 04:13 tep_dtt_benchmark.py
usage: tep_dtt_benchmark.py [-h] [--dataset {small_tep,rieth_tep}]
                            [--fault_id FAULT_ID] [--onset_train ONSET_TRAIN]
                            [--onset_test ONSET_TEST] [--q_eps Q_EPS]
                            [--q_delta Q_DELTA] [--fpr FPR] [--outdir OUTDIR]

options:
  -h, --help            show this help message and exit
  --dataset {small_tep,rieth_tep}
  --fault_id FAULT_ID
  --onset_train ONSET_TRAIN
  --onset_test ONSET_TEST
  --q_eps Q_EPS
  --q_delta Q_DELTA
  --fpr FPR
  --outdir OUTDIR

In [ ]:
# out_tep_f1 にCSVが残ってないか確認
!ls -lah /content/out_tep_f1 | head -n 200

# 残ってたら onset91 にコピー（必要ファイルだけ）
!mkdir -p /content/out_tep_f1_onset91
!cp -av /content/out_tep_f1/tep_*.csv /content/out_tep_f1_onset91/ 2>/dev/null || true
!cp -av /content/out_tep_f1/FigTEP_*.pdf /content/out_tep_f1_onset91/ 2>/dev/null || true

# 確認
!ls -lah /content/out_tep_f1_onset91 | head -n 200


total 48K
drwxr-xr-x 2 root root 4.0K Feb 14 04:14 .
drwxr-xr-x 1 root root 4.0K Feb 14 04:13 ..
-rw-r--r-- 1 root root  33K Feb 14 04:14 FigTEP_boundary.pdf
-rw-r--r-- 1 root root  177 Feb 14 04:14 tep_metrics.csv
'/content/out_tep_f1/tep_metrics.csv' -> '/content/out_tep_f1_onset91/tep_metrics.csv'
'/content/out_tep_f1/FigTEP_boundary.pdf' -> '/content/out_tep_f1_onset91/FigTEP_boundary.pdf'
total 48K
drwxr-xr-x 2 root root 4.0K Feb 14 04:22 .
drwxr-xr-x 1 root root 4.0K Feb 14 04:13 ..
-rw-r--r-- 1 root root  33K Feb 14 04:14 FigTEP_boundary.pdf
-rw-r--r-- 1 root root  177 Feb 14 04:14 tep_metrics.csv


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import os, shutil, time

OUTDIR = "/content/out_tep_f1_onset91"   # ←いま存在してるやつ
assert os.path.isdir(OUTDIR), f"OUTDIR not found: {OUTDIR}"

STAMP = time.strftime("%Y%m%d-%H%M%S")
BASE  = "/content/drive/MyDrive/colab_backup"
BKP   = f"{BASE}/TEP_{STAMP}"
os.makedirs(BKP, exist_ok=True)

def copy_any(src):
    if not os.path.exists(src):
        print("[SKIP]", src)
        return
    dst = f"{BKP}/{os.path.basename(src)}"
    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        shutil.copy2(src, dst)
    print("[OK]", src, "->", dst)

# 1) 出力（最重要）
copy_any(OUTDIR)

# 2) スクリプト（再現性のため）
copy_any("/content/tep_dtt_benchmark.py")

# 3) 入力ZIP（小さいなら一緒に）
copy_any("/content/data/small_tep.zip")

# 4) ついでに outdir を zip 化（保険）
zip_path = f"/content/tep_snapshot_{STAMP}.zip"
shutil.make_archive(zip_path.replace(".zip",""), "zip", OUTDIR)
copy_any(zip_path)

print("\n[SAVED] backup folder =", BKP)


Mounted at /content/drive
[OK] /content/out_tep_f1_onset91 -> /content/drive/MyDrive/colab_backup/TEP_20260214-044513/out_tep_f1_onset91
[OK] /content/tep_dtt_benchmark.py -> /content/drive/MyDrive/colab_backup/TEP_20260214-044513/tep_dtt_benchmark.py
[OK] /content/data/small_tep.zip -> /content/drive/MyDrive/colab_backup/TEP_20260214-044513/small_tep.zip
[OK] /content/tep_snapshot_20260214-044513.zip -> /content/drive/MyDrive/colab_backup/TEP_20260214-044513/tep_snapshot_20260214-044513.zip

[SAVED] backup folder = /content/drive/MyDrive/colab_backup/TEP_20260214-044513


In [ ]:
# ===== ONE-SHOT REGEN CELL: per-run metrics + delay CSV + summary CSV =====
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os, glob, subprocess, shutil, zipfile, textwrap, pandas as pd

# ---------- paths ----------
DATA_ZIP = "/content/data/small_tep.zip"
DATA_DIR = "/content/data"                 # 展開先
SCRIPT   = "/content/tep_dtt_benchmark.py" # あなたのベンチスクリプト
OUTDIR   = "/content/out_tep_f1_onset91"   # 出力先（onset91）

# ---------- sanity ----------
assert os.path.exists(SCRIPT), f"missing: {SCRIPT}"
os.makedirs(OUTDIR, exist_ok=True)

# ---------- ensure dataset exists (unzip if needed) ----------
if os.path.exists(DATA_ZIP):
    # small_tep.zip を /content/data に展開（既にあっても上書きしない）
    with zipfile.ZipFile(DATA_ZIP, "r") as z:
        # zip 内トップ階層を推定
        names = z.namelist()
        top = names[0].split("/")[0] if names else ""
        # 既に展開されてそうならスキップ
        marker = os.path.join(DATA_DIR, top) if top else None
        if marker and os.path.exists(marker):
            print("[OK] dataset already unpacked:", marker)
        else:
            z.extractall(DATA_DIR)
            print("[OK] unpacked dataset to:", DATA_DIR)
else:
    print("[WARN] small_tep.zip not found; assuming dataset already available under /content/data")

# ---------- locate dataset root (must contain dataset.csv) ----------
cand = []
for p in [DATA_DIR, "/content"]:
    cand += glob.glob(os.path.join(p, "**", "dataset.csv"), recursive=True)
assert len(cand) > 0, "dataset.csv not found under /content/data or /content"
dataset_csv = sorted(cand)[0]
dataset_root = os.path.dirname(dataset_csv)
print("[OK] dataset root:", dataset_root)

# ---------- run benchmark script to regenerate outdir csvs ----------
# NOTE: あなたの tep_dtt_benchmark.py の CLI に合わせて呼ぶ。
#       ここでは一般的な形（--dataset, --fault_id, --onset_test, --outdir, --fpr）で投げ、
#       もし引数名が違って落ちたら、ヘルプを表示して止める。

FAULT_ID   = 1
ONSET_TEST = 91
FPR        = 0.05

cmd = [
    "python", SCRIPT,
    "--dataset", "small_tep",
    "--fault_id", str(FAULT_ID),
    "--onset_test", str(ONSET_TEST),
    "--fpr", str(FPR),
    "--outdir", OUTDIR,
]

print("[RUN]", " ".join(cmd))
proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr)
    # ヘルプを出して終了（引数名の不一致をここで検出できる）
    print("\n[HINT] tep_dtt_benchmark.py の引数が違う可能性。下の help を見て、cmd を書き換えて再実行。")
    subprocess.run(["python", SCRIPT, "-h"])
    raise SystemExit("benchmark script failed")

# ---------- verify outputs (regen targets) ----------
need = ["tep_metrics_by_run.csv", "tep_delay_by_run.csv"]
missing = [f for f in need if not os.path.exists(os.path.join(OUTDIR, f))]
if missing:
    print("[WARN] missing expected outputs:", missing)
    print("[INFO] OUTDIR contains:")
    for x in sorted(os.listdir(OUTDIR))[:200]:
        print("  -", x)
    raise SystemExit("Expected CSVs not generated (script version mismatch).")

print("[OK] found:", [os.path.join(OUTDIR,f) for f in need])

# ---------- (optional) rebuild/overwrite summary tep_metrics.csv from by_run + delay_by_run ----------
by_run = pd.read_csv(os.path.join(OUTDIR, "tep_metrics_by_run.csv"))
delay  = pd.read_csv(os.path.join(OUTDIR, "tep_delay_by_run.csv"))

# by_run の列名揺れを吸収
# 想定: run_id / auc / pr_auc がある（あなたの現状コードは run_id, auc, pr_auc）
col_run = "run_id" if "run_id" in by_run.columns else ("run" if "run" in by_run.columns else None)
assert col_run is not None, f"cannot find run id column in by_run: {by_run.columns.tolist()}"

def pick(col_candidates):
    for c in col_candidates:
        if c in by_run.columns: return c
    return None

col_auc = pick(["auc","roc_auc","roc-auc","ROC-AUC"])
col_pr  = pick(["pr_auc","prauc","pr-auc","PR-AUC"])
assert col_auc and col_pr, f"cannot find auc/pr columns in by_run: {by_run.columns.tolist()}"

valid = by_run.dropna(subset=[col_auc, col_pr]).copy()
auc_mean = float(valid[col_auc].mean()) if len(valid) else float("nan")
auc_std  = float(valid[col_auc].std(ddof=1)) if len(valid) >= 2 else 0.0
pr_mean  = float(valid[col_pr].mean()) if len(valid) else float("nan")
pr_std   = float(valid[col_pr].std(ddof=1)) if len(valid) >= 2 else 0.0

delay_mean = float(delay["delay"].mean()) if "delay" in delay.columns and len(delay) else float("nan")
delay_std  = float(delay["delay"].std(ddof=1)) if "delay" in delay.columns and len(delay) >= 2 else 0.0

summary = pd.DataFrame([{
    "dataset": "small_tep",
    "fault_id": int(FAULT_ID),
    "onset_test": int(ONSET_TEST),
    "fpr": float(FPR),
    "runs_total": int(by_run.shape[0]),
    "runs_valid": int(valid.shape[0]),
    "auc_mean": auc_mean, "auc_std": auc_std,
    "pr_auc_mean": pr_mean, "pr_auc_std": pr_std,
    "delay_mean": delay_mean, "delay_std": delay_std,
}])

summary_path = os.path.join(OUTDIR, "tep_metrics.csv")
summary.to_csv(summary_path, index=False)
print("[OK] wrote summary:", summary_path)
print(summary)

print("\n[NEXT] Table 転記用:")
print(f"  AUC   = {auc_mean:.6g} ± {auc_std:.6g}")
print(f"  PR    = {pr_mean:.6g} ± {pr_std:.6g}")
print(f"  Delay = {delay_mean:.6g} ± {delay_std:.6g}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[OK] unpacked dataset to: /content/data
[OK] dataset root: /content/data
[RUN] python /content/tep_dtt_benchmark.py --dataset small_tep --fault_id 1 --onset_test 91 --fpr 0.05 --outdir /content/out_tep_f1_onset91


Reading data/small_tep/dataset.csv: 100%|██████████| 153300/153300 [00:01<00:00, 113038.00it/s]

Reading data/small_tep/labels.csv: 100%|██████████| 153300/153300 [00:00<00:00, 3046132.58it/s]

Reading data/small_tep/train_mask.csv: 100%|██████████| 153300/153300 [00:00<00:00, 2987176.73it/s]

Reading data/small_tep/test_mask.csv: 100%|██████████| 153300/153300 [00:00<00:00, 3215046.54it/s]
Traceback (most recent call last):
  File "/content/tep_dtt_benchmark.py", line 277, in <module>
    main()
  File "/content/tep_dtt_benchmark.py", line 270, in main
    plot_timeseries(feat, y, test_mask, args.fault_id, args.onset_test, thr, os.path.join(args.o

SystemExit: benchmark script failed

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
# ===== ONE-SHOT: patch to skip plots + rerun benchmark =====
import os, re, subprocess

SCRIPT = "/content/tep_dtt_benchmark.py"
OUTDIR = "/content/out_tep_f1_onset91"
FAULT_ID = 1
ONSET_TEST = 91
FPR = 0.05

assert os.path.exists(SCRIPT), f"missing: {SCRIPT}"
os.makedirs(OUTDIR, exist_ok=True)

# --- patch: wrap plot_timeseries / boundary plotting calls (idempotent) ---
txt = open(SCRIPT, "r", encoding="utf-8").read().splitlines()
if any("DTT_SKIP_PLOTS" in line for line in txt):
    print("[OK] patch already applied (DTT_SKIP_PLOTS found).")
else:
    out = []
    patched_any = False
    for line in txt:
        # 代表的に FigTEP_timeseries を作る plot_timeseries 呼び出しをガード
        if ("plot_timeseries(" in line) and ("FigTEP_timeseries" in line) and ("DTT_SKIP_PLOTS" not in line):
            indent = re.match(r"^(\s*)", line).group(1)
            out.append(indent + 'if os.environ.get("DTT_SKIP_PLOTS","0") != "1":')
            out.append(indent + "    " + line.strip())
            patched_any = True
            continue

        # ついでに boundary 図も同様に（該当があれば）
        if ("FigTEP_boundary" in line) and ("plot_" in line) and ("DTT_SKIP_PLOTS" not in line) and ("(" in line):
            indent = re.match(r"^(\s*)", line).group(1)
            out.append(indent + 'if os.environ.get("DTT_SKIP_PLOTS","0") != "1":')
            out.append(indent + "    " + line.strip())
            patched_any = True
            continue

        out.append(line)

    # import os が無いと困るので保険で先頭に入れる（既にあればそのまま）
    if not any(re.match(r"^\s*import\s+os\b", l) or re.match(r"^\s*from\s+os\b", l) for l in out):
        # 先頭の shebang/encoding/comment 群のあとに差し込む
        k = 0
        while k < len(out) and (out[k].startswith("#") or out[k].strip()=="" or "coding" in out[k]):
            k += 1
        out = out[:k] + ["import os  # (auto-added for DTT_SKIP_PLOTS)"] + out[k:]

    open(SCRIPT, "w", encoding="utf-8").write("\n".join(out) + "\n")
    print("[OK] patched script (skip-plots guard inserted).")

# --- rerun with plots skipped ---
env = dict(os.environ, DTT_SKIP_PLOTS="1")
cmd = [
    "python", SCRIPT,
    "--dataset", "small_tep",
    "--fault_id", str(FAULT_ID),
    "--onset_test", str(ONSET_TEST),
    "--fpr", str(FPR),
    "--outdir", OUTDIR,
]
print("[RUN]", " ".join(cmd))
p = subprocess.run(cmd, text=True, capture_output=True, env=env)
print(p.stdout)
if p.returncode != 0:
    print(p.stderr)
    raise SystemExit("benchmark script failed even with plots skipped")

# --- check outputs ---
import pandas as pd
need = ["tep_metrics_by_run.csv", "tep_delay_by_run.csv", "tep_metrics.csv"]
for f in need:
    path = os.path.join(OUTDIR, f)
    print("[CHECK]", f, "=>", "OK" if os.path.exists(path) else "MISSING", path)

print("\n[HEAD] tep_metrics.csv")
display(pd.read_csv(os.path.join(OUTDIR,"tep_metrics.csv")).head(3))

print("\n[HEAD] tep_delay_by_run.csv")
display(pd.read_csv(os.path.join(OUTDIR,"tep_delay_by_run.csv")).head(10))


[OK] patched script (skip-plots guard inserted).
[RUN] python /content/tep_dtt_benchmark.py --dataset small_tep --fault_id 1 --onset_test 91 --fpr 0.05 --outdir /content/out_tep_f1_onset91
[OK] wrote: /content/out_tep_f1_onset91/tep_metrics.csv
[OK] figs : /content/out_tep_f1_onset91/FigTEP_boundary.pdf /content/out_tep_f1_onset91/FigTEP_timeseries.pdf

[CHECK] tep_metrics_by_run.csv => MISSING /content/out_tep_f1_onset91/tep_metrics_by_run.csv
[CHECK] tep_delay_by_run.csv => MISSING /content/out_tep_f1_onset91/tep_delay_by_run.csv
[CHECK] tep_metrics.csv => OK /content/out_tep_f1_onset91/tep_metrics.csv

[HEAD] tep_metrics.csv


,AUC_mean,AUC_std,PRAUC_mean,PRAUC_std,Delay_mean,Delay_std,n_runs,fault_id,dataset,eps,delta,thr
0,NaN,NaN,NaN,NaN,NaN,NaN,0,1,small_tep,0.503966,0.029125,0.273455



[HEAD] tep_delay_by_run.csv


FileNotFoundError: [Errno 2] No such file or directory: '/content/out_tep_f1_onset91/tep_delay_by_run.csv'

In [ ]:
# ===== ONE-SHOT: patch tep_dtt_benchmark.py to export by-run + delay CSVs, then run =====
import os, re, textwrap, subprocess
import pandas as pd

SCRIPT = "/content/tep_dtt_benchmark.py"

# ---- run config (必要ならここだけ変える) ----
DATASET    = "small_tep"
FAULT_ID   = 1
ONSET_TEST = 91
FPR        = 0.05
OUTDIR     = "/content/out_tep_f1_onset91"
# ----------------------------------------

assert os.path.exists(SCRIPT), f"missing: {SCRIPT}"
os.makedirs(OUTDIR, exist_ok=True)

src = open(SCRIPT, "r", encoding="utf-8").read()
marker = "AUTO_RUNWISE_EXPORT_v2026_02_14"

# (A) まず「plot を殺す」ガードを main(args parse直後)に注入
if "TEP_SKIP_PLOTS" not in src:
    # import os が無いと困るので一応先頭付近へ
    if "import os" in src:
        src = src.replace("import os", "import os\nSKIP_PLOTS = os.environ.get('TEP_SKIP_PLOTS','0')=='1'\n", 1)
    else:
        src = "import os\nSKIP_PLOTS = os.environ.get('TEP_SKIP_PLOTS','0')=='1'\n" + src

# args = parser.parse_args() の直後に、plot関数をnoop化
# （plot_timeseries が落ちる系をまとめて潰す）
pat_args = re.compile(r"(?m)^(?P<indent>\s*)args\s*=\s*parser\.parse_args\(\)\s*$")
m_args = pat_args.search(src)
if m_args and "AUTO_SKIP_PLOTS_NOOP" not in src:
    indent = m_args.group("indent")
    noop_block = textwrap.dedent(f"""
    # === AUTO_SKIP_PLOTS_NOOP ===
    if SKIP_PLOTS:
        def _noop(*a, **k):
            return None
        for _fn in ["plot_timeseries", "plot_boundary", "plot_phase_boundary", "plot_tepslice"]:
            if _fn in globals():
                globals()[_fn] = _noop
    # === END AUTO_SKIP_PLOTS_NOOP ===
    """)
    noop_block = "\n".join(indent + line if line.strip() else line for line in noop_block.splitlines()) + "\n"
    src = src[:m_args.end()] + "\n" + noop_block + src[m_args.end():]

# (B) tep_metrics.csv を書く直前に「run別CSV生成」ブロックを注入
if marker not in src:
    pat_metrics = re.compile(r'(?m)^(?P<indent>\s*).*(to_csv)\(.*tep_metrics\.csv.*\)\s*$')
    m_met = pat_metrics.search(src)
    assert m_met, "could not locate the line that writes tep_metrics.csv in tep_dtt_benchmark.py"

    indent = m_met.group("indent")

    runwise_block = textwrap.dedent(f"""
    # === {marker} ===
    # Export run-wise AUC/PR-AUC and detection delay into:
    #   - tep_metrics_by_run.csv
    #   - tep_delay_by_run.csv
    try:
        import numpy as _np
        import pandas as _pd
        from sklearn.metrics import roc_auc_score as _roc_auc
        from sklearn.metrics import average_precision_score as _ap

        _loc = locals()

        # outdir を推定（OUTDIR/outdir/args.outdir どれでもOK）
        _outdir = _loc.get("OUTDIR", _loc.get("outdir", None))
        if _outdir is None and "args" in _loc:
            _outdir = getattr(args, "outdir", None)
        if _outdir is None:
            _outdir = "{OUTDIR}"

        def _as1d(v):
            a = _np.asarray(v)
            return a.ravel() if a.ndim != 1 else a

        # y(0/1) を自動検出
        def _pick_y():
            cand=[]
            for k,v in _loc.items():
                try: a=_as1d(v)
                except Exception: continue
                if a.size < 50: continue
                if not (_np.issubdtype(a.dtype,_np.integer) or _np.issubdtype(a.dtype,_np.bool_) or _np.issubdtype(a.dtype,_np.floating)):
                    continue
                aa=_np.asarray(a, dtype=float)
                if _np.isnan(aa).all(): continue
                u=set(_np.unique(_np.nan_to_num(aa, nan=-1)).tolist())
                if u.issubset({0.0,1.0}) or u.issubset({0.0,1.0,-1.0}):
                    cand.append((0 if ("y" in k or "label" in k) else 1, k, aa.astype(int)))
            if not cand: return None, None
            cand.sort(key=lambda x:(x[0], x[1]))
            return cand[0][1], cand[0][2]

        y_name, y = _pick_y()
        if y is None:
            raise RuntimeError("cannot find label vector y in locals()")

        N = int(len(y))

        # run_id / t / score を同じ長さNから自動検出
        def _pick_vec(kind):
            cand=[]
            for k,v in _loc.items():
                try: a=_as1d(v)
                except Exception: continue
                if a.size != N: continue

                if kind == "rid":
                    try: aa=_np.nan_to_num(_np.asarray(a, dtype=float), nan=-1).astype(int)
                    except Exception: continue
                    uniq=_np.unique(aa)
                    if uniq.size>=2 and uniq.size <= max(2, N//20):
                        cand.append((0 if "run" in k else 1, k, aa))

                if kind == "t":
                    try: aa=_np.nan_to_num(_np.asarray(a, dtype=float), nan=-1).astype(int)
                    except Exception: continue
                    if (aa.max()-aa.min()) >= 10:
                        cand.append((0 if (k=="t" or "time" in k or "sample" in k) else 1, k, aa))

                if kind == "s":
                    if not (_np.issubdtype(a.dtype,_np.floating) or _np.issubdtype(a.dtype,_np.integer)):
                        continue
                    aa=_np.asarray(a, dtype=float)
                    if _np.nanstd(aa) < 1e-12: continue
                    u=set(_np.unique(_np.round(aa[~_np.isnan(aa)][:2000],6)).tolist())
                    if u.issubset({0.0,1.0}):  # binaryはスコアとして除外
                        continue
                    cand.append((0 if ("score" in k or k in ("s","score","s_signed","score_signed") or "risk" in k) else 1, k, aa))

            if not cand: return None, None
            cand.sort(key=lambda x:(x[0], x[1]))
            return cand[0][1], cand[0][2]

        rid_name, rid = _pick_vec("rid")
        if rid is None:
            raise RuntimeError("cannot find run_id vector in locals() (length mismatch?)")

        t_name, tt = _pick_vec("t")
        if tt is None:
            tt = _np.arange(N, dtype=int)

        s_name, s = _pick_vec("s")
        if s is None:
            raise RuntimeError("cannot find score vector in locals()")

        _fpr   = float(getattr(args, "fpr", {FPR})) if "args" in _loc else float({FPR})
        _onset = int(getattr(args, "onset_test", {ONSET_TEST})) if "args" in _loc else int({ONSET_TEST})
        _fault = int(getattr(args, "fault_id", {FAULT_ID})) if "args" in _loc else int({FAULT_ID})

        neg_scores = s[y==0]
        thr_run = float(_np.quantile(neg_scores, 1.0-_fpr)) if neg_scores.size>0 else float("nan")

        rows=[]
        drows=[]
        for r in _np.unique(rid):
            m = (rid==r)
            yy=y[m]; ss=s[m]; tt_r=tt[m]
            pos=int((yy==1).sum()); neg=int((yy==0).sum()); nn=int(m.sum())

            auc=_np.nan; pr=_np.nan
            if pos>0 and neg>0:
                auc=float(_roc_auc(yy, ss))
                pr=float(_ap(yy, ss))

            det=False; tdet=_np.nan; delay=_np.nan
            if nn>0 and _np.isfinite(thr_run):
                after = (tt_r >= _onset)
                if after.any():
                    cand = tt_r[after][ss[after] >= thr_run]
                    if cand.size>0:
                        tdet=int(cand.min()); det=True; delay=float(tdet - _onset)

            rows.append(dict(fault_id=_fault, run_id=int(r), auc=auc, pr_auc=pr, pos=pos, neg=neg, n=nn))
            drows.append(dict(fault_id=_fault, run_id=int(r), delay=delay, detect=int(det),
                              t_detect=tdet, t_onset=_onset, thr=thr_run, pos=pos, neg=neg, n=nn))

        by_run=_pd.DataFrame(rows).sort_values("run_id")
        by_run_path=os.path.join(_outdir, "tep_metrics_by_run.csv")
        by_run.to_csv(by_run_path, index=False)

        delay_by_run=_pd.DataFrame(drows).sort_values("run_id")
        delay_path=os.path.join(_outdir, "tep_delay_by_run.csv")
        delay_by_run.to_csv(delay_path, index=False)

        # summary dict があれば mean±std を入れる（列名は今の出力に合わせる）
        if "summary" in _loc and isinstance(summary, dict):
            valid=by_run.dropna(subset=["auc","pr_auc"])
            if len(valid)>0:
                summary["AUC_mean"]=float(valid["auc"].mean())
                summary["AUC_std"]=float(valid["auc"].std(ddof=1)) if len(valid)>=2 else 0.0
                summary["PRAUC_mean"]=float(valid["pr_auc"].mean())
                summary["PRAUC_std"]=float(valid["pr_auc"].std(ddof=1)) if len(valid)>=2 else 0.0
                dd=delay_by_run.dropna(subset=["delay"])
                summary["Delay_mean"]=float(dd["delay"].mean()) if len(dd)>0 else float("nan")
                summary["Delay_std"]=float(dd["delay"].std(ddof=1)) if len(dd)>=2 else 0.0
                summary["n_runs"]=int(len(valid))
            summary["thr"]=thr_run

        print("[OK] wrote:", by_run_path)
        print("[OK] wrote:", delay_path)

    except Exception as _e:
        print("[WARN] runwise export failed:", _e)
    # === END AUTO RUNWISE EXPORT ===
    """)

    runwise_block = "\n".join(indent + line if line.strip() else line for line in runwise_block.splitlines()) + "\n"
    src = src[:m_met.start()] + runwise_block + src[m_met.start():]

    open(SCRIPT, "w", encoding="utf-8").write(src)
    print("[OK] patched:", SCRIPT)
else:
    print("[OK] already patched:", SCRIPT)

# (C) 実行（plotは落ちるので環境変数で殺す）
env = os.environ.copy()
env["TEP_SKIP_PLOTS"] = "1"   # plot部が壊れてても先へ進める

cmd = [
    "python", SCRIPT,
    "--dataset", DATASET,
    "--fault_id", str(FAULT_ID),
    "--onset_test", str(ONSET_TEST),
    "--fpr", str(FPR),
    "--outdir", OUTDIR
]
print("[RUN]", " ".join(cmd))
p = subprocess.run(cmd, text=True, capture_output=True, env=env)
print(p.stdout)
if p.returncode != 0:
    print(p.stderr)
    raise SystemExit("benchmark failed")

# (D) 生成物チェック
need = ["tep_metrics.csv", "tep_metrics_by_run.csv", "tep_delay_by_run.csv"]
for f in need:
    path = os.path.join(OUTDIR, f)
    print(f, "->", "OK" if os.path.exists(path) else "MISSING", path)

display(pd.read_csv(os.path.join(OUTDIR, "tep_metrics.csv")).head())
display(pd.read_csv(os.path.join(OUTDIR, "tep_metrics_by_run.csv")).head())
display(pd.read_csv(os.path.join(OUTDIR, "tep_delay_by_run.csv")).head())


[OK] patched: /content/tep_dtt_benchmark.py
[RUN] python /content/tep_dtt_benchmark.py --dataset small_tep --fault_id 1 --onset_test 91 --fpr 0.05 --outdir /content/out_tep_f1_onset91
[OK] wrote: /content/out_tep_f1_onset91/tep_metrics_by_run.csv
[OK] wrote: /content/out_tep_f1_onset91/tep_delay_by_run.csv


Reading data/small_tep/dataset.csv: 100%|██████████| 153300/153300 [00:01<00:00, 85815.83it/s]

Reading data/small_tep/labels.csv: 100%|██████████| 153300/153300 [00:00<00:00, 1879842.02it/s]

Reading data/small_tep/train_mask.csv: 100%|██████████| 153300/153300 [00:00<00:00, 1820213.57it/s]

Reading data/small_tep/test_mask.csv: 100%|██████████| 153300/153300 [00:00<00:00, 1868322.15it/s]
Traceback (most recent call last):
  File "/content/tep_dtt_benchmark.py", line 435, in <module>
    main()
  File "/content/tep_dtt_benchmark.py", line 426, in main
    plot_boundary(theta_df, y, test_mask, args.fault_id, w, b, os.path.join(args.outdir, "FigTEP_boundary.pdf"))
  File "/content/t

SystemExit: benchmark failed

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
import pandas as pd

test_mask = pd.read_csv("data/small_tep/test_mask.csv")
train_mask = pd.read_csv("data/small_tep/train_mask.csv")
labels = pd.read_csv("data/small_tep/labels.csv")

print("test_mask columns:", test_mask.columns)
print("train_mask columns:", train_mask.columns)
print("labels columns:", labels.columns)

display(test_mask.head())
display(labels.head())

# まず「テストが存在するか」
for name, df in [("train_mask", train_mask), ("test_mask", test_mask)]:
    s = df.select_dtypes("number").sum(numeric_only=True)
    print(name, "numeric col sums:\n", s)

# fault_id が列としてあるなら分布を見る（列名が違っても head/columns で分かる）
if "fault_id" in labels.columns:
    print("fault_id counts:\n", labels["fault_id"].value_counts().head(30))


test_mask columns: Index(['run_id', 'sample', 'test_mask'], dtype='object')
train_mask columns: Index(['run_id', 'sample', 'train_mask'], dtype='object')
labels columns: Index(['run_id', 'sample', 'labels'], dtype='object')


,run_id,sample,test_mask
0,413402073,1,False
1,413402073,2,False
2,413402073,3,False
3,413402073,4,False
4,413402073,5,False


,run_id,sample,labels
0,413402073,1,0
1,413402073,2,0
2,413402073,3,0
3,413402073,4,0
4,413402073,5,0


train_mask numeric col sums:
 run_id    52818299438900
sample          61585650
dtype: int64
test_mask numeric col sums:
 run_id    52818299438900
sample          61585650
dtype: int64


In [ ]:
import pandas as pd
from pathlib import Path

base = Path("data/small_tep")

def to01(x):
    s = str(x).strip().lower()
    return 1 if s in ("1","true","t","yes") else 0

for fn, col in [("test_mask.csv","test_mask"), ("train_mask.csv","train_mask")]:
    p = base / fn
    df = pd.read_csv(p)
    df[col] = df[col].map(to01).astype(int)
    df.to_csv(p, index=False)
    print("[OK] rewrote", p, "->", col, "sum=", int(df[col].sum()), "/", len(df))


[OK] rewrote data/small_tep/test_mask.csv -> test_mask sum= 100800 / 153300
[OK] rewrote data/small_tep/train_mask.csv -> train_mask sum= 52500 / 153300


In [ ]:
import pandas as pd
base="data/small_tep"
tm = pd.read_csv(f"{base}/test_mask.csv")
lb = pd.read_csv(f"{base}/labels.csv")

m_test = tm["test_mask"].astype(int).values == 1
runs_test = set(tm.loc[m_test, "run_id"].unique())

fault_id = 1
m_fault_test = m_test & (lb["labels"].astype(int).values == fault_id)
runs_fault_test = set(tm.loc[m_fault_test, "run_id"].unique())

print("unique test runs:", len(runs_test))
print(f"unique test runs with fault_id={fault_id}:", len(runs_fault_test))


unique test runs: 105
unique test runs with fault_id=1: 5


In [ ]:
!python /content/tep_dtt_benchmark.py \
  --dataset small_tep \
  --fault_id 1 \
  --onset_test 91 \
  --fpr 0.05 \
  --outdir /content/out_tep_f1_onset91


Reading data/small_tep/dataset.csv: 100% 153300/153300 [00:02<00:00, 74613.42it/s]
Reading data/small_tep/labels.csv: 100% 153300/153300 [00:00<00:00, 1942969.22it/s]
Reading data/small_tep/train_mask.csv: 100% 153300/153300 [00:00<00:00, 2018384.96it/s]
Reading data/small_tep/test_mask.csv: 100% 153300/153300 [00:00<00:00, 2033198.43it/s]
[OK] wrote: /content/out_tep_f1_onset91/tep_metrics_by_run.csv
[OK] wrote: /content/out_tep_f1_onset91/tep_delay_by_run.csv
[WARN] plot_boundary skipped: No suitable test runs found for plotting.
[WARN] plot_timeseries skipped: Too many levels: Index has only 1 level, not 2
[OK] wrote: /content/out_tep_f1_onset91/tep_metrics.csv
[OK] figs : SKIP /content/out_tep_f1_onset91/FigTEP_boundary.pdf SKIP /content/out_tep_f1_onset91/FigTEP_timeseries.pdf


In [ ]:
import os, glob, pandas as pd
OUT="/content/out_tep_f1_onset91"
print("files:", sorted([os.path.basename(p) for p in glob.glob(OUT+"/*")]))

for f in ["tep_metrics.csv","tep_metrics_by_run.csv","tep_delay_by_run.csv","FigTEP_boundary.pdf"]:
    p=os.path.join(OUT,f)
    print(f, "=>", "OK" if os.path.exists(p) else "MISSING", p)

# 先頭確認
for f in ["tep_metrics_by_run.csv","tep_delay_by_run.csv","tep_metrics.csv"]:
    p=os.path.join(OUT,f)
    if os.path.exists(p):
        display(pd.read_csv(p).head())


files: ['tep_delay_by_run.csv', 'tep_metrics.csv', 'tep_metrics_by_run.csv']
tep_metrics.csv => OK /content/out_tep_f1_onset91/tep_metrics.csv
tep_metrics_by_run.csv => OK /content/out_tep_f1_onset91/tep_metrics_by_run.csv
tep_delay_by_run.csv => OK /content/out_tep_f1_onset91/tep_delay_by_run.csv
FigTEP_boundary.pdf => MISSING /content/out_tep_f1_onset91/FigTEP_boundary.pdf


,fault_id,run_id,auc,pr_auc,pos,neg,n
0,1,0,0.499328,0.662580,13044,6661,19705
1,1,1,0.515605,0.685145,13155,6421,19576
2,1,2,0.487179,0.695059,13845,6079,19924
3,1,3,0.492309,0.690742,12308,5508,17816
4,1,4,0.479071,0.667267,9239,4413,13652


,fault_id,run_id,delay,detect,t_detect,t_onset,thr,pos,neg,n
0,1,0,NaN,0,NaN,91,56.467538,13044,6661,19705
1,1,1,NaN,0,NaN,91,56.467538,13155,6421,19576
2,1,2,NaN,0,NaN,91,56.467538,13845,6079,19924
3,1,3,NaN,0,NaN,91,56.467538,12308,5508,17816
4,1,4,NaN,0,NaN,91,56.467538,9239,4413,13652


,AUC_mean,AUC_std,PRAUC_mean,PRAUC_std,Delay_mean,Delay_std,n_runs,fault_id,dataset,eps,delta,thr
0,NaN,NaN,NaN,NaN,NaN,NaN,0,1,small_tep,0.503966,0.029125,0.273455


In [ ]:
!rm -rf /content/out_tep_f1_onset91
!mkdir -p /content/out_tep_f1_onset91


In [ ]:
!cp -f /content/tep_dtt_benchmark_patched.py /content/tep_dtt_benchmark.py
!python -c "import hashlib;print(hashlib.md5(open('/content/tep_dtt_benchmark.py','rb').read()).hexdigest())"


cp: cannot stat '/content/tep_dtt_benchmark_patched.py': No such file or directory
1d423781087a9cb701f6ea4249bf791d


In [ ]:
!python /content/tep_dtt_benchmark_patched.py --help


python3: can't open file '/content/tep_dtt_benchmark_patched.py': [Errno 2] No such file or directory


In [ ]:
!ls -l /content | grep tep_dtt


-rw-r--r-- 1 root root 18219 Feb 14 06:23 tep_dtt_benchmark.py


In [ ]:
import pathlib

p = pathlib.Path("/content/tep_dtt_benchmark.py")
s = p.read_text(encoding="utf-8")

a = s.index("    # 6) plots")
b = s.index('    print("[OK] wrote:"', a)

new_block = """    # 6) plots (best-effort; never fail benchmark on plotting)
    boundary_pdf = os.path.join(args.outdir, "FigTEP_boundary.pdf")
    timeseries_pdf = os.path.join(args.outdir, "FigTEP_timeseries.pdf")
    boundary_ok = False
    timeseries_ok = False

    if os.environ.get("DTT_SKIP_PLOTS","0") != "1":
        try:
            plot_boundary(theta_df, y, test_mask, args.fault_id, w, b, boundary_pdf)
            boundary_ok = True
        except ValueError as e:
            print(f"[WARN] plot_boundary skipped: {e}")
        except Exception as e:
            print(f"[WARN] plot_boundary failed: {e}")

        try:
            plot_timeseries(feat, y, test_mask, args.fault_id, args.onset_test, thr, timeseries_pdf)
            timeseries_ok = True
        except ValueError as e:
            print(f"[WARN] plot_timeseries skipped: {e}")
        except Exception as e:
            print(f"[WARN] plot_timeseries failed: {e}")
    else:
        print("[NOTE] DTT_SKIP_PLOTS=1 -> skip plotting")

"""

p.write_text(s[:a] + new_block + s[b:], encoding="utf-8")
print("[OK] patched in-place:", p)


[OK] patched in-place: /content/tep_dtt_benchmark.py


In [ ]:
OUT=/content/out_tep_f1_onset91
!rm -rf "$OUT"
!mkdir -p "$OUT"


SyntaxError: invalid syntax (ipython-input-963840139.py, line 1)

In [ ]:
!OUT=/content/out_tep_f1_onset91
!rm -rf /content/out_tep_f1_onset91
!mkdir -p /content/out_tep_f1_onset91


In [ ]:
!python /content/tep_dtt_benchmark.py \
  --dataset small_tep --fault_id 1 --onset_test 91 --fpr 0.05 --outdir /content/out_tep_f1_onset91

!echo "exit_code=$?"


Reading data/small_tep/dataset.csv: 100% 153300/153300 [00:02<00:00, 73248.99it/s]
Reading data/small_tep/labels.csv: 100% 153300/153300 [00:00<00:00, 1851472.58it/s]
Reading data/small_tep/train_mask.csv: 100% 153300/153300 [00:00<00:00, 2082426.95it/s]
Reading data/small_tep/test_mask.csv: 100% 153300/153300 [00:00<00:00, 2026495.64it/s]
[OK] wrote: /content/out_tep_f1_onset91/tep_metrics_by_run.csv
[OK] wrote: /content/out_tep_f1_onset91/tep_delay_by_run.csv
[WARN] plot_boundary skipped: No suitable test runs found for plotting.
[WARN] plot_timeseries failed: Too many levels: Index has only 1 level, not 2
[OK] wrote: /content/out_tep_f1_onset91/tep_metrics.csv
[OK] figs : SKIP /content/out_tep_f1_onset91/FigTEP_boundary.pdf SKIP /content/out_tep_f1_onset91/FigTEP_timeseries.pdf
exit_code=0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, datetime, pathlib

stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
DST = f"/content/drive/MyDrive/colab_backup/tep_{stamp}"
os.makedirs(DST, exist_ok=True)

# 退避したいもの（必要に応じて追加）
targets = [
    "/content/tep_dtt_benchmark.py",
    "/content/out_tep_f1_onset91",
]

for t in targets:
    if os.path.exists(t):
        name = os.path.basename(t.rstrip("/"))
        dst = os.path.join(DST, name)
        if os.path.isdir(t):
            shutil.copytree(t, dst, dirs_exist_ok=True)
        else:
            shutil.copy2(t, dst)
        print("[OK] saved:", t, "->", dst)
    else:
        print("[MISS]", t)

print("Backup folder:", DST)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[OK] saved: /content/tep_dtt_benchmark.py -> /content/drive/MyDrive/colab_backup/tep_20260214-072817/tep_dtt_benchmark.py
[OK] saved: /content/out_tep_f1_onset91 -> /content/drive/MyDrive/colab_backup/tep_20260214-072817/out_tep_f1_onset91
Backup folder: /content/drive/MyDrive/colab_backup/tep_20260214-072817


In [43]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [44]:
import shutil, os

SRC="/content/drive/MyDrive/colab_backup/tep_YYYYMMDD-HHMMSS"  # ←ここを最新に
shutil.copy2(f"{SRC}/tep_dtt_benchmark.py", "/content/tep_dtt_benchmark.py")
shutil.copytree(f"{SRC}/out_tep_f1_onset91", "/content/out_tep_f1_onset91", dirs_exist_ok=True)

print("[OK] restored")


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/colab_backup/tep_YYYYMMDD-HHMMSS/tep_dtt_benchmark.py'

In [45]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import os, shutil
from pathlib import Path

BASE = Path("/content/drive/MyDrive/colab_backup")

# 最新の tep_YYYYMMDD-HHMMSS を自動で拾う
cands = [p for p in BASE.glob("tep_*") if p.is_dir()]
if not cands:
    raise FileNotFoundError(f"No backup folders found under: {BASE}")

SRC = max(cands, key=lambda p: p.stat().st_mtime)  # 更新時刻が最新
print("[INFO] Using backup folder:", SRC)

# 期待ファイル/フォルダ
src_py = SRC / "tep_dtt_benchmark.py"
src_out = SRC / "out_tep_f1_onset91"

print("[INFO] Exists tep_dtt_benchmark.py ?", src_py.exists())
print("[INFO] Exists out_tep_f1_onset91   ?", src_out.exists())

# もし名前が違っていた場合に備えて中身一覧も出す
if not src_py.exists():
    print("[WARN] tep_dtt_benchmark.py not found. Files in backup root are:")
    for p in sorted(SRC.iterdir()):
        print("  -", p.name)
    raise FileNotFoundError(f"Missing: {src_py}")

# 復元
shutil.copy2(src_py, "/content/tep_dtt_benchmark.py")
print("[OK] restored:", "/content/tep_dtt_benchmark.py")

if src_out.exists():
    shutil.copytree(src_out, "/content/out_tep_f1_onset91", dirs_exist_ok=True)
    print("[OK] restored:", "/content/out_tep_f1_onset91")
else:
    print("[WARN] out_tep_f1_onset91 not found in backup (skipped).")


Mounted at /content/drive
[INFO] Using backup folder: /content/drive/MyDrive/colab_backup/tep_20260214-072817
[INFO] Exists tep_dtt_benchmark.py ? True
[INFO] Exists out_tep_f1_onset91   ? True
[OK] restored: /content/tep_dtt_benchmark.py
[OK] restored: /content/out_tep_f1_onset91


In [46]:
!python /content/tep_dtt_benchmark.py --help


usage: tep_dtt_benchmark.py [-h] [--dataset {small_tep,rieth_tep}]
                            [--fault_id FAULT_ID] [--onset_train ONSET_TRAIN]
                            [--onset_test ONSET_TEST] [--q_eps Q_EPS]
                            [--q_delta Q_DELTA] [--fpr FPR] [--outdir OUTDIR]

options:
  -h, --help            show this help message and exit
  --dataset {small_tep,rieth_tep}
  --fault_id FAULT_ID
  --onset_train ONSET_TRAIN
  --onset_test ONSET_TEST
  --q_eps Q_EPS
  --q_delta Q_DELTA
  --fpr FPR
  --outdir OUTDIR


In [47]:
# (1) 出力先を作る（既にあってもOK）
!mkdir -p /content/out_tep_f1_onset91

# (2) fault_id=1 を実行（onset_test=91 を明示）
!python /content/tep_dtt_benchmark.py \
  --dataset small_tep \
  --fault_id 1 \
  --onset_test 91 \
  --outdir /content/out_tep_f1_onset91


Reading data/small_tep/dataset.csv: 100% 153300/153300 [00:01<00:00, 114832.91it/s]
Reading data/small_tep/labels.csv: 100% 153300/153300 [00:00<00:00, 3104416.78it/s]
Reading data/small_tep/train_mask.csv: 100% 153300/153300 [00:00<00:00, 3142576.20it/s]
Reading data/small_tep/test_mask.csv: 100% 153300/153300 [00:00<00:00, 3208436.93it/s]
[OK] wrote: /content/out_tep_f1_onset91/tep_metrics_by_run.csv
[OK] wrote: /content/out_tep_f1_onset91/tep_delay_by_run.csv
[WARN] plot_boundary skipped: No suitable test runs found for plotting.
[WARN] plot_timeseries failed: Too many levels: Index has only 1 level, not 2
[OK] wrote: /content/out_tep_f1_onset91/tep_metrics.csv
[OK] figs : SKIP /content/out_tep_f1_onset91/FigTEP_boundary.pdf SKIP /content/out_tep_f1_onset91/FigTEP_timeseries.pdf


In [48]:
!python /content/tep_dtt_benchmark.py \
  --dataset small_tep \
  --fault_id 1 \
  --onset_train 91 \
  --onset_test 91 \
  --outdir /content/out_tep_f1_onset91


Reading data/small_tep/dataset.csv: 100% 153300/153300 [00:01<00:00, 97963.63it/s]
Reading data/small_tep/labels.csv: 100% 153300/153300 [00:00<00:00, 1911830.41it/s]
Reading data/small_tep/train_mask.csv: 100% 153300/153300 [00:00<00:00, 1952990.00it/s]
Reading data/small_tep/test_mask.csv: 100% 153300/153300 [00:00<00:00, 2062573.95it/s]
[OK] wrote: /content/out_tep_f1_onset91/tep_metrics_by_run.csv
[OK] wrote: /content/out_tep_f1_onset91/tep_delay_by_run.csv
[WARN] plot_boundary skipped: No suitable test runs found for plotting.
[WARN] plot_timeseries failed: Too many levels: Index has only 1 level, not 2
[OK] wrote: /content/out_tep_f1_onset91/tep_metrics.csv
[OK] figs : SKIP /content/out_tep_f1_onset91/FigTEP_boundary.pdf SKIP /content/out_tep_f1_onset91/FigTEP_timeseries.pdf


In [49]:
!python /content/tep_dtt_benchmark.py \
  --dataset small_tep \
  --fault_id 1 \
  --onset_test 91 \
  --fpr 0.05 \
  --outdir /content/out_tep_f1_onset91


Reading data/small_tep/dataset.csv: 100% 153300/153300 [00:01<00:00, 136665.78it/s]
Reading data/small_tep/labels.csv: 100% 153300/153300 [00:00<00:00, 3103832.34it/s]
Reading data/small_tep/train_mask.csv: 100% 153300/153300 [00:00<00:00, 3130108.09it/s]
Reading data/small_tep/test_mask.csv: 100% 153300/153300 [00:00<00:00, 3281767.21it/s]
[OK] wrote: /content/out_tep_f1_onset91/tep_metrics_by_run.csv
[OK] wrote: /content/out_tep_f1_onset91/tep_delay_by_run.csv
[WARN] plot_boundary skipped: No suitable test runs found for plotting.
[WARN] plot_timeseries failed: Too many levels: Index has only 1 level, not 2
[OK] wrote: /content/out_tep_f1_onset91/tep_metrics.csv
[OK] figs : SKIP /content/out_tep_f1_onset91/FigTEP_boundary.pdf SKIP /content/out_tep_f1_onset91/FigTEP_timeseries.pdf


In [50]:
!ls -lah /content/out_tep_f1_onset91 | head -200
!find /content/out_tep_f1_onset91 -maxdepth 2 -type f | sed -n '1,200p'


total 28K
drwx------ 2 root root 4.0K Feb 14 07:16 .
drwxr-xr-x 1 root root 4.0K Feb 14 08:33 ..
-rw------- 1 root root 6.8K Feb 14 09:03 tep_delay_by_run.csv
-rw------- 1 root root 6.9K Feb 14 09:03 tep_metrics_by_run.csv
-rw------- 1 root root  177 Feb 14 09:03 tep_metrics.csv
/content/out_tep_f1_onset91/tep_metrics_by_run.csv
/content/out_tep_f1_onset91/tep_delay_by_run.csv
/content/out_tep_f1_onset91/tep_metrics.csv


In [51]:
import os, re
import numpy as np
import pandas as pd
from pathlib import Path

OUTDIR = Path("/content/out_tep_f1_onset91")
m_by = OUTDIR / "tep_metrics_by_run.csv"
d_by = OUTDIR / "tep_delay_by_run.csv"

assert m_by.exists(), f"Missing: {m_by}"
assert d_by.exists(), f"Missing: {d_by}"

dfm = pd.read_csv(m_by)
dfd = pd.read_csv(d_by)

def pick_col(df, candidates):
    cols = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols:
            return cols[cand.lower()]
    # fuzzy
    for c in df.columns:
        cl = c.lower()
        for cand in candidates:
            if cand.lower() in cl:
                return c
    raise KeyError(f"None of {candidates} found in columns={list(df.columns)}")

roc_col = pick_col(dfm, ["roc_auc", "roc-auc", "auc_roc"])
pr_col  = pick_col(dfm, ["pr_auc", "pr-auc", "average_precision", "ap"])
# delay file側
delay_col = pick_col(dfd, ["delay", "delay@fpr", "delay_at_fpr"])

roc = dfm[roc_col].astype(float).to_numpy()
pra = dfm[pr_col].astype(float).to_numpy()
dly = dfd[delay_col].astype(float).to_numpy()

def fmt_mean_std(x, nd=3, force_int=False):
    mu = float(np.mean(x))
    sd = float(np.std(x, ddof=1)) if len(x) > 1 else 0.0
    if force_int or (np.all(np.isfinite(x)) and np.max(np.abs(x - np.round(x))) < 1e-6):
        return f"{mu:.0f}$\\pm${sd:.0f}"
    return f"{mu:.{nd}f}$\\pm${sd:.{nd}f}"

roc_s = fmt_mean_std(roc, nd=3)
pr_s  = fmt_mean_std(pra, nd=3)

# Delayは整数ステップっぽければ整数表示、そうでなければ小数1桁
delay_force_int = (np.max(np.abs(dly - np.round(dly))) < 1e-6)
dly_s = fmt_mean_std(dly, nd=1, force_int=delay_force_int)

row = f"DTT (TEP, fault=1) & {roc_s} & {pr_s} & {dly_s} \\\\\n"

tex_path = OUTDIR / "table_tep_fault1.tex"
tex_path.write_text(row, encoding="utf-8")

print("[OK] wrote:", tex_path)
print("---- LaTeX row preview ----")
print(row)


KeyError: "None of ['roc_auc', 'roc-auc', 'auc_roc'] found in columns=['fault_id', 'run_id', 'auc', 'pr_auc', 'pos', 'neg', 'n']"

In [52]:
import numpy as np
import pandas as pd
from pathlib import Path

OUTDIR = Path("/content/out_tep_f1_onset91")
m_by = OUTDIR / "tep_metrics_by_run.csv"
d_by = OUTDIR / "tep_delay_by_run.csv"

dfm = pd.read_csv(m_by)
dfd = pd.read_csv(d_by)

def pick_col(df, candidates):
    cols_lower = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]
    # contains-match fallback
    for c in df.columns:
        cl = c.lower()
        for cand in candidates:
            if cand.lower() in cl:
                return c
    raise KeyError(f"None of {candidates} found in columns={list(df.columns)}")

# --- filter (robust) ---
if "fault_id" in dfm.columns:
    dfm = dfm[dfm["fault_id"] == 1].copy()
if "fault_id" in dfd.columns:
    dfd = dfd[dfd["fault_id"] == 1].copy()
# if delay file has fpr column, lock to 0.05
fpr_cols = [c for c in dfd.columns if "fpr" in c.lower()]
if fpr_cols:
    fpr_c = fpr_cols[0]
    dfd = dfd[np.isclose(dfd[fpr_c].astype(float), 0.05)].copy()

# --- column mapping (THIS fixes your error) ---
roc_col = pick_col(dfm, ["auc", "roc_auc", "roc-auc", "auc_roc"])
pr_col  = pick_col(dfm, ["pr_auc", "pr-auc", "average_precision", "ap"])
delay_col = pick_col(dfd, ["delay", "delay_steps", "delay@fpr", "delay_at_fpr"])

roc = dfm[roc_col].astype(float).to_numpy()
pra = dfm[pr_col].astype(float).to_numpy()
dly = dfd[delay_col].astype(float).to_numpy()

def fmt_mean_std(x, nd=3, force_int=False):
    mu = float(np.mean(x))
    sd = float(np.std(x, ddof=1)) if len(x) > 1 else 0.0
    if force_int:
        return f"{mu:.0f}$\\pm${sd:.0f}"
    return f"{mu:.{nd}f}$\\pm${sd:.{nd}f}"

roc_s = fmt_mean_std(roc, nd=3)
pr_s  = fmt_mean_std(pra, nd=3)

delay_force_int = (len(dly) > 0) and (np.max(np.abs(dly - np.round(dly))) < 1e-6)
dly_s = fmt_mean_std(dly, nd=1, force_int=delay_force_int)

row = f"DTT (TEP, fault=1) & {roc_s} & {pr_s} & {dly_s} \\\\\n"

tex_path = OUTDIR / "table_tep_fault1.tex"
tex_path.write_text(row, encoding="utf-8")

print("[OK] wrote:", tex_path)
print("LaTeX row:", row)
print("[INFO] used columns:", {"roc": roc_col, "pr": pr_col, "delay": delay_col})
print("[INFO] n_runs(metrics)=", len(dfm), " n_runs(delay)=", len(dfd))


[OK] wrote: /content/out_tep_f1_onset91/table_tep_fault1.tex
LaTeX row: DTT (TEP, fault=1) & nan$\pm$nan & nan$\pm$nan & nan$\pm$nan \\

[INFO] used columns: {'roc': 'auc', 'pr': 'pr_auc', 'delay': 'delay'}
[INFO] n_runs(metrics)= 149  n_runs(delay)= 149


In [53]:
!sed -n '1,5p' /content/out_tep_f1_onset91/table_tep_fault1.tex


DTT (TEP, fault=1) & nan$\pm$nan & nan$\pm$nan & nan$\pm$nan \\


In [54]:
import numpy as np
import pandas as pd
from pathlib import Path

OUTDIR = Path("/content/out_tep_f1_onset91")
dfm = pd.read_csv(OUTDIR/"tep_metrics_by_run.csv")
dfd = pd.read_csv(OUTDIR/"tep_delay_by_run.csv")

# fault=1
if "fault_id" in dfm.columns: dfm = dfm[dfm["fault_id"]==1].copy()
if "fault_id" in dfd.columns: dfd = dfd[dfd["fault_id"]==1].copy()

# delay側に fpr 列があれば 0.05 に固定
fpr_cols = [c for c in dfd.columns if "fpr" in c.lower()]
if fpr_cols:
    fpr_c = fpr_cols[0]
    dfd[fpr_c] = pd.to_numeric(dfd[fpr_c], errors="coerce")
    dfd = dfd[np.isclose(dfd[fpr_c], 0.05)].copy()

# 数値化（ダメなものは NaN に落とす）
dfm["auc"]   = pd.to_numeric(dfm["auc"], errors="coerce")
dfm["pr_auc"]= pd.to_numeric(dfm["pr_auc"], errors="coerce")
dfd["delay"] = pd.to_numeric(dfd["delay"], errors="coerce")

roc = dfm["auc"].replace([np.inf,-np.inf], np.nan).to_numpy()
pra = dfm["pr_auc"].replace([np.inf,-np.inf], np.nan).to_numpy()
dly = dfd["delay"].replace([np.inf,-np.inf], np.nan).to_numpy()

def mean_std_ignore_nan(x):
    x = x[np.isfinite(x)]
    if len(x)==0:
        return (np.nan, np.nan, 0)
    mu = float(np.mean(x))
    sd = float(np.std(x, ddof=1)) if len(x) > 1 else 0.0
    return (mu, sd, len(x))

mu_roc, sd_roc, n_roc = mean_std_ignore_nan(roc)
mu_pr,  sd_pr,  n_pr  = mean_std_ignore_nan(pra)
mu_dly, sd_dly, n_dly = mean_std_ignore_nan(dly)

def fmt(mu, sd, nd=3, force_int=False):
    if not np.isfinite(mu):  # 全部NaNならここ
        return r"\text{NA}"
    if force_int:
        return f"{mu:.0f}$\\pm${sd:.0f}"
    return f"{mu:.{nd}f}$\\pm${sd:.{nd}f}"

delay_force_int = np.isfinite(mu_dly) and np.max(np.abs(dly[np.isfinite(dly)] - np.round(dly[np.isfinite(dly)]))) < 1e-6
row = (
    f"DTT (TEP, fault=1) & {fmt(mu_roc, sd_roc, nd=3)} & {fmt(mu_pr, sd_pr, nd=3)} & "
    f"{fmt(mu_dly, sd_dly, nd=1, force_int=delay_force_int)} \\\\\n"
)

tex_path = OUTDIR / "table_tep_fault1.tex"
tex_path.write_text(row, encoding="utf-8")

print("[OK] rewrote:", tex_path)
print("[INFO] valid runs:", {"auc": n_roc, "pr_auc": n_pr, "delay": n_dly})
print("LaTeX row:", row)


[OK] rewrote: /content/out_tep_f1_onset91/table_tep_fault1.tex
[INFO] valid runs: {'auc': 126, 'pr_auc': 126, 'delay': 93}
LaTeX row: DTT (TEP, fault=1) & 0.493$\pm$0.103 & 0.670$\pm$0.185 & 231$\pm$187 \\



In [55]:
!sed -n '1,5p' /content/out_tep_f1_onset91/table_tep_fault1.tex


DTT (TEP, fault=1) & 0.493$\pm$0.103 & 0.670$\pm$0.185 & 231$\pm$187 \\


In [56]:
import numpy as np, pandas as pd
from pathlib import Path

OUT = Path("/content/out_tep_f1_onset91")  # ここはあなたの outdir に合わせる
FPR = 0.05

m = pd.read_csv(OUT/"tep_metrics_by_run.csv")
d = pd.read_csv(OUT/"tep_delay_by_run.csv")

# 列名ゆれに強く
def colpick(df, cands):
    for c in df.columns:
        cl = c.lower()
        for k in cands:
            if k in cl:
                return c
    raise KeyError((df.columns, cands))

run_c  = colpick(m, ["run_id"])
roc_c  = colpick(m, ["roc", "auc"])         # だいたい "auc"
pr_c   = colpick(m, ["pr", "average_precision"])
d_run  = colpick(d, ["run_id"])
d_fpr  = colpick(d, ["fpr"])
d_del  = colpick(d, ["delay"])

# FPR=0.05 の delay だけ使う
d05 = d[np.isclose(d[d_fpr].astype(float), FPR)].copy()

roc = m[[run_c, roc_c]].set_index(run_c)[roc_c].astype(float)
pr  = m[[run_c, pr_c ]].set_index(run_c)[pr_c ].astype(float)
dl  = d05[[d_run, d_del]].set_index(d_run)[d_del].astype(float)

# 3指標すべて有限な run の共通集合で mean±std（これが“再現向き”）
valid = roc.index.intersection(pr.index).intersection(dl.index)
valid = [r for r in valid if np.isfinite(roc.loc[r]) and np.isfinite(pr.loc[r]) and np.isfinite(dl.loc[r])]

def mean_std(x):
    x = np.asarray(x, float)
    return float(np.mean(x)), float(np.std(x, ddof=1)) if len(x) > 1 else 0.0

mu_roc, sd_roc = mean_std(roc.loc[valid])
mu_pr,  sd_pr  = mean_std(pr.loc[valid])
mu_dl,  sd_dl  = mean_std(dl.loc[valid])

print("[INFO] valid runs (intersection):", len(valid))
print("[RESULT] ROC-AUC  :", mu_roc, "+/-", sd_roc)
print("[RESULT] PR-AUC   :", mu_pr,  "+/-", sd_pr)
print("[RESULT] Delay@5% :", mu_dl,  "+/-", sd_dl)

# 論文Table IIIターゲットとの差
T_ROC, T_PR, T_DL = 0.663, 0.0169, 28.8
print("[DIFF] |ΔROC|,|ΔPR|,|ΔDelay| =", abs(mu_roc-T_ROC), abs(mu_pr-T_PR), abs(mu_dl-T_DL))


KeyError: (Index(['fault_id', 'run_id', 'delay', 'detect', 't_detect', 't_onset', 'thr',
       'pos', 'neg', 'n'],
      dtype='object'), ['fpr'])

In [57]:
import numpy as np, pandas as pd
from pathlib import Path

OUT = Path("/content/out_tep_f1_onset91")
FPR_TARGET = 0.05

m = pd.read_csv(OUT/"tep_metrics_by_run.csv")
d = pd.read_csv(OUT/"tep_delay_by_run.csv")

def colpick(df, keys):
    # keys: ["run_id"] みたいに “含まれていればOK” の部分一致
    for c in df.columns:
        cl = c.lower()
        for k in keys:
            if k.lower() in cl:
                return c
    raise KeyError(f"keys={keys} not found in columns={list(df.columns)}")

# --- columns (metrics) ---
run_m = colpick(m, ["run_id"])
roc_c = colpick(m, ["auc"])          # tep_metrics_by_run.csv は "auc" が来る
pr_c  = colpick(m, ["pr_auc", "pr"]) # "pr_auc" or "pr..."

# --- columns (delay) ---
run_d = colpick(d, ["run_id"])
del_c = colpick(d, ["delay"])
fpr_cols = [c for c in d.columns if "fpr" in c.lower()]

# --- optional: filter by FPR if column exists ---
d_use = d.copy()
if fpr_cols:
    fpr_c = fpr_cols[0]
    d_use[fpr_c] = pd.to_numeric(d_use[fpr_c], errors="coerce")
    d_use = d_use[np.isclose(d_use[fpr_c], FPR_TARGET)].copy()
    print(f"[INFO] delay filtered by {fpr_c} == {FPR_TARGET}")
else:
    print("[INFO] delay file has NO fpr column; assume already computed at fixed FPR")

# --- numeric conversion ---
m_use = m.copy()
m_use[roc_c] = pd.to_numeric(m_use[roc_c], errors="coerce")
m_use[pr_c ] = pd.to_numeric(m_use[pr_c ], errors="coerce")
d_use[del_c] = pd.to_numeric(d_use[del_c], errors="coerce")

roc = m_use[[run_m, roc_c]].dropna().set_index(run_m)[roc_c]
pr  = m_use[[run_m, pr_c ]].dropna().set_index(run_m)[pr_c]
dl  = d_use[[run_d, del_c]].dropna().set_index(run_d)[del_c]

# --- if duplicates per run in delay, keep first (or min) ---
if dl.index.duplicated().any():
    dl = dl.groupby(level=0).min()
    print("[WARN] delay has duplicate run_id; using min delay per run")

# --- intersection runs where all 3 exist and finite ---
valid = roc.index.intersection(pr.index).intersection(dl.index)
valid = [r for r in valid if np.isfinite(roc.loc[r]) and np.isfinite(pr.loc[r]) and np.isfinite(dl.loc[r])]

def mean_std(x):
    x = np.asarray(x, float)
    return float(np.mean(x)), float(np.std(x, ddof=1)) if len(x) > 1 else 0.0

mu_roc, sd_roc = mean_std(roc.loc[valid])
mu_pr,  sd_pr  = mean_std(pr.loc[valid])
mu_dl,  sd_dl  = mean_std(dl.loc[valid])

print("[INFO] valid runs (intersection):", len(valid))
print("[RESULT] ROC-AUC  :", mu_roc, "+/-", sd_roc)
print("[RESULT] PR-AUC   :", mu_pr,  "+/-", sd_pr)
print("[RESULT] Delay@5% :", mu_dl,  "+/-", sd_dl)


[INFO] delay file has NO fpr column; assume already computed at fixed FPR
[INFO] valid runs (intersection): 70
[RESULT] ROC-AUC  : 0.4931777766669702 +/- 0.13306997292998407
[RESULT] PR-AUC   : 0.7076815485460058 +/- 0.23100239388560562
[RESULT] Delay@5% : 147.21428571428572 +/- 122.91529932214873


In [58]:
# ===== Repro Pack (TEP fault1) : one-shot =====
import os, sys, json, subprocess, textwrap, hashlib, time
from pathlib import Path
import pandas as pd
import numpy as np

TS = time.strftime("%Y%m%d-%H%M%S")
OUTDIR = Path(f"/content/repro_tep_fault1_{TS}")
OUTDIR.mkdir(parents=True, exist_ok=True)

# --- (A) environment snapshot ---
(OUTDIR/"cmd.txt").write_text("", encoding="utf-8")
def sh(cmd):
    (OUTDIR/"cmd.txt").write_text((OUTDIR/"cmd.txt").read_text(encoding="utf-8") + cmd + "\n", encoding="utf-8")
    return subprocess.check_output(cmd, shell=True, text=True)

env_txt = []
env_txt.append("### python\n" + sys.version)
env_txt.append("\n### pip freeze\n" + sh("python -m pip freeze | sed -n '1,200p'"))
(OUTDIR/"env.txt").write_text("\n".join(env_txt), encoding="utf-8")

# --- (B) hash input files (so you can prove which dataset you used) ---
def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1<<20), b""):
            h.update(chunk)
    return h.hexdigest()

DATA_ROOT = Path("/content/data/small_tep")
inputs = ["dataset.csv","labels.csv","train_mask.csv","test_mask.csv"]
hashes = {}
for fn in inputs:
    p = DATA_ROOT/fn
    hashes[str(p)] = sha256_file(p) if p.exists() else None
(OUTDIR/"input_hashes.json").write_text(json.dumps(hashes, indent=2), encoding="utf-8")

# --- (C) run benchmark (you can change onset here if needed) ---
FAULT_ID = 1
ONSET = 91       # ←今と同じ設定で再現を固める。あとで160等に変えるならここだけ変える
FPR = 0.05

RUN_OUT = OUTDIR/"run_out"
RUN_OUT.mkdir(exist_ok=True)

cmd = f"""
python /content/tep_dtt_benchmark.py \
  --dataset small_tep \
  --fault_id {FAULT_ID} \
  --onset_train {ONSET} \
  --onset_test {ONSET} \
  --fpr {FPR} \
  --outdir {RUN_OUT}
""".strip()
print(cmd)
print(sh(cmd))

# --- (D) verify outputs exist ---
expected = ["tep_metrics_by_run.csv","tep_delay_by_run.csv","tep_metrics.csv"]
print("\n[CHECK] outputs:")
for fn in expected:
    p = RUN_OUT/fn
    print(" ", fn, "OK" if p.exists() else "MISSING", str(p))

# --- (E) zip repro pack ---
zip_path = f"/content/repro_tep_fault1_{TS}.zip"
sh(f"cd /content && zip -r {zip_path} {OUTDIR.name} >/dev/null")
print("\n[OK] repro pack:", zip_path)


python /content/tep_dtt_benchmark.py   --dataset small_tep   --fault_id 1   --onset_train 91   --onset_test 91   --fpr 0.05   --outdir /content/repro_tep_fault1_20260214-100340/run_out
[OK] wrote: /content/repro_tep_fault1_20260214-100340/run_out/tep_metrics_by_run.csv
[OK] wrote: /content/repro_tep_fault1_20260214-100340/run_out/tep_delay_by_run.csv
[WARN] plot_boundary skipped: No suitable test runs found for plotting.
[WARN] plot_timeseries failed: Too many levels: Index has only 1 level, not 2
[OK] wrote: /content/repro_tep_fault1_20260214-100340/run_out/tep_metrics.csv
[OK] figs : SKIP /content/repro_tep_fault1_20260214-100340/run_out/FigTEP_boundary.pdf SKIP /content/repro_tep_fault1_20260214-100340/run_out/FigTEP_timeseries.pdf


[CHECK] outputs:
  tep_metrics_by_run.csv OK /content/repro_tep_fault1_20260214-100340/run_out/tep_metrics_by_run.csv
  tep_delay_by_run.csv OK /content/repro_tep_fault1_20260214-100340/run_out/tep_delay_by_run.csv
  tep_metrics.csv OK /content/repro_te

In [59]:
import pandas as pd, numpy as np
from pathlib import Path

# 直近のreproフォルダを自動で拾う（ダメなら手でRUN_OUTを指定）
cands = sorted(Path("/content").glob("repro_tep_fault1_*/run_out"), key=lambda p: p.stat().st_mtime, reverse=True)
RUN_OUT = cands[0] if cands else Path("/content/out_tep_f1_onset91")
print("[RUN_OUT]", RUN_OUT)

m = pd.read_csv(RUN_OUT/"tep_metrics_by_run.csv")
d = pd.read_csv(RUN_OUT/"tep_delay_by_run.csv")

print("[metrics_by_run cols]", list(m.columns))
print("[delay_by_run cols]", list(d.columns))

# ここは「あなたのCSV列名に合わせて」吸収（auc / pr_auc / delay 等）
def pick(df, candidates):
    cols = [c.lower() for c in df.columns]
    for cand in candidates:
        cand = cand.lower()
        for c0,c in zip(df.columns, cols):
            if cand == c or cand in c:
                return c0
    raise KeyError((df.columns, candidates))

run_c = pick(m, ["run_id"])
auc_c = pick(m, ["auc","roc_auc","auc_roc"])
pra_c = pick(m, ["pr_auc","average_precision","ap"])
run_d = pick(d, ["run_id"])
del_c = pick(d, ["delay","delay_steps","delay_at_fpr"])

m2 = m[[run_c, auc_c, pra_c]].copy()
d2 = d[[run_d, del_c]].copy()

# 数値化
m2[auc_c] = pd.to_numeric(m2[auc_c], errors="coerce")
m2[pra_c] = pd.to_numeric(m2[pra_c], errors="coerce")
d2[del_c] = pd.to_numeric(d2[del_c], errors="coerce")

# 3指標すべて有限のrunだけ
m2 = m2.dropna()
d2 = d2.dropna()
merged = m2.merge(d2, left_on=run_c, right_on=run_d, how="inner")
valid = merged[np.isfinite(merged[auc_c]) & np.isfinite(merged[pra_c]) & np.isfinite(merged[del_c])]

mu_auc, sd_auc = valid[auc_c].mean(), valid[auc_c].std(ddof=1)
mu_pra, sd_pra = valid[pra_c].mean(), valid[pra_c].std(ddof=1)
mu_del, sd_del = valid[del_c].mean(), valid[del_c].std(ddof=1)

print("\n[AGG] valid runs:", len(valid))
print(f"[AGG] ROC-AUC  : {mu_auc:.6f} ± {sd_auc:.6f}")
print(f"[AGG] PR-AUC   : {mu_pra:.6f} ± {sd_pra:.6f}")
print(f"[AGG] Delay@5% : {mu_del:.6f} ± {sd_del:.6f}")

# Table III target (paper)
T_AUC, T_PRA, T_DEL = 0.663, 0.0169, 28.8
print("\n[DIFF vs Table III]")
print(" |ΔROC| =", abs(mu_auc - T_AUC))
print(" |ΔPR | =", abs(mu_pra - T_PRA))
print(" |ΔDel| =", abs(mu_del - T_DEL))


[RUN_OUT] /content/repro_tep_fault1_20260214-100340/run_out
[metrics_by_run cols] ['fault_id', 'run_id', 'auc', 'pr_auc', 'pos', 'neg', 'n']
[delay_by_run cols] ['fault_id', 'run_id', 'delay', 'detect', 't_detect', 't_onset', 'thr', 'pos', 'neg', 'n']

[AGG] valid runs: 57
[AGG] ROC-AUC  : 0.523838 ± 0.209883
[AGG] PR-AUC   : 0.822204 ± 0.154468
[AGG] Delay@5% : 206.842105 ± 122.169640

[DIFF vs Table III]
 |ΔROC| = 0.13916201749291268
 |ΔPR | = 0.8053035988637718
 |ΔDel| = 178.04210526315788


In [60]:
import pandas as pd, numpy as np
from pathlib import Path

RUN_OUT = Path("/content/repro_tep_fault1_20260214-100340/run_out")  # スクショのrun_outに合わせて
m = pd.read_csv(RUN_OUT/"tep_metrics_by_run.csv")
d = pd.read_csv(RUN_OUT/"tep_delay_by_run.csv")

print("[metrics cols]", list(m.columns))
print("[delay cols]", list(d.columns))
print("[runs] metrics:", len(m), " delay:", len(d))

# NaN count
for c in ["auc","pr_auc"]:
    if c in m.columns:
        print("[NaN]", c, int(m[c].isna().sum()))
if "delay" in d.columns:
    print("[NaN] delay", int(d["delay"].isna().sum()))

# pos-rate (重要)
if {"pos","neg"}.issubset(m.columns):
    pr = m["pos"] / (m["pos"] + m["neg"])
    print("[pos-rate] mean=", float(pr.mean()), "median=", float(pr.median()))
    print(pr.describe())
else:
    print("[WARN] pos/neg列がない")

# onset/detect sanity
if "t_onset" in d.columns:
    print("[t_onset] unique head:", sorted(d["t_onset"].unique())[:10])
if {"t_detect","t_onset"}.issubset(d.columns):
    dd = (pd.to_numeric(d["t_detect"], errors="coerce") - pd.to_numeric(d["t_onset"], errors="coerce"))
    print("[delay recompute] mean±std:", float(dd.mean()), float(dd.std(ddof=1)))


[metrics cols] ['fault_id', 'run_id', 'auc', 'pr_auc', 'pos', 'neg', 'n']
[delay cols] ['fault_id', 'run_id', 'delay', 'detect', 't_detect', 't_onset', 'thr', 'pos', 'neg', 'n']
[runs] metrics: 86  delay: 86
[NaN] auc 14
[NaN] pr_auc 14
[NaN] delay 15
[pos-rate] mean= 0.642487799597699 median= 0.6747427101200686
count    86.000000
mean      0.642488
std       0.319479
min       0.000000
25%       0.571330
50%       0.674743
75%       0.900000
max       0.994220
dtype: float64
[t_onset] unique head: [np.int64(91)]
[delay recompute] mean±std: 237.8450704225352 126.35852030600685


In [61]:
%%writefile /content/tep_dtt_benchmark.py
import os
import argparse
import json
import sys
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import LinearSVC
from sklearn.metrics import roc_auc_score, average_precision_score

SKIP_PLOTS = os.environ.get("TEP_SKIP_PLOTS", "0") == "1"

def _get_label(ds):
    if hasattr(ds, "label"):
        return ds.label
    if hasattr(ds, "labels"):
        return ds.labels
    raise AttributeError("Dataset has neither .label nor .labels")

def _get_mask(ds, name):
    if hasattr(ds, name):
        return getattr(ds, name)
    raise AttributeError(f"Dataset has no .{name}")

def _infer_levels(df: pd.DataFrame):
    if not isinstance(df.index, pd.MultiIndex) or df.index.nlevels < 2:
        raise ValueError("Expected a MultiIndex with (run_id, sample)")
    return 0, 1  # (run_id, time/sample)

def build_theta_pca(df, train_nominal_mask, n_components=2, seed=0):
    scaler = StandardScaler()
    X_nom = df.loc[train_nominal_mask].values
    X_nom_s = scaler.fit_transform(X_nom)

    pca = PCA(n_components=n_components, random_state=seed)
    pca.fit(X_nom_s)

    X_all_s = scaler.transform(df.values)
    theta = pca.transform(X_all_s)
    theta_df = pd.DataFrame(theta, index=df.index, columns=[f"theta{i+1}" for i in range(n_components)])
    return scaler, pca, theta_df

def fit_linear_boundary(theta_df, y, train_mask, fault_id, onset_train=20, seed=0):
    run_level, time_level = _infer_levels(theta_df)
    t = theta_df.index.get_level_values(time_level).astype(int)

    in_train = train_mask.astype(bool)
    is_fault = (y == fault_id)
    is_nominal = (y == 0)

    train_fault_pts = in_train & is_fault & (t >= onset_train)
    train_nom_pts   = in_train & (is_nominal | (is_fault & (t < onset_train)))

    theta_tr = pd.concat([theta_df.loc[train_nom_pts], theta_df.loc[train_fault_pts]], axis=0)
    y_tr = np.r_[np.zeros(train_nom_pts.sum(), dtype=int), np.ones(train_fault_pts.sum(), dtype=int)]

    if theta_tr.shape[0] < 100:
        raise ValueError("Too few training points. Try a different fault_id or dataset=rieth_tep.")

    clf = LinearSVC(C=1.0, class_weight="balanced", random_state=seed)
    clf.fit(theta_tr.values, y_tr)

    w = clf.coef_.reshape(-1)
    b = float(clf.intercept_[0])
    return w, b

def compute_dist_ti_score(theta_df, w, b, eps, delta):
    wnorm = np.linalg.norm(w) + 1e-12
    g = theta_df.values @ w + b
    dist = np.abs(g) / wnorm

    run_level, time_level = _infer_levels(theta_df)
    v = theta_df.groupby(level=run_level).diff().fillna(0.0).values
    vnorm = np.linalg.norm(v, axis=1) + 1e-12
    ti = np.abs(v @ w) / (wnorm * vnorm)

    score = np.maximum(0.0, eps - dist) + np.maximum(0.0, delta - ti)
    out = pd.DataFrame({"dist": dist, "TI": ti, "score": score}, index=theta_df.index)
    return out

def pick_threshold_from_nominal(features_df, y, test_mask, fault_id, onset_test=160, fpr=0.05):
    run_level, time_level = _infer_levels(features_df)
    t = features_df.index.get_level_values(time_level).astype(int)

    in_test = test_mask.astype(bool)
    is_fault = (y == fault_id)
    is_nominal = (y == 0)

    # negatives = nominal test + fault pre-onset (test)
    nominal_pts = in_test & (is_nominal | (is_fault & (t < onset_test)))
    s_nom = features_df.loc[nominal_pts, "score"].values
    if s_nom.size == 0:
        return float("nan")
    thr = np.quantile(s_nom, 1.0 - fpr)
    return float(thr)

def eval_fault_runs(features_df, y, test_mask, fault_id, onset_test=160, pos_window=8, thr=None):
    run_level, time_level = _infer_levels(features_df)

    in_test = test_mask.astype(bool)
    is_fault = (y == fault_id)
    idx_fault_test = features_df.index[in_test & is_fault]
    runs = idx_fault_test.get_level_values(run_level).unique().tolist()
    if not runs:
        raise ValueError("No fault test runs found.")

    rows = []
    drows = []
    for r in runs:
        g = features_df.xs(r, level=run_level).copy()
        tt = g.index.to_numpy(dtype=int)  # xs() 後は time index だけになる
        ss = g["score"].to_numpy()

        # ======= time-indexed label y_t ∈ {0,1} (short positive window) =======
        ybin = ((tt >= onset_test) & (tt < onset_test + pos_window)).astype(int)
        pos = int(ybin.sum())
        neg = int((ybin == 0).sum())
        n = int(ybin.size)

        auc = np.nan
        pr  = np.nan
        if pos > 0 and neg > 0:
            auc = float(roc_auc_score(ybin, ss))
            pr  = float(average_precision_score(ybin, ss))

        det = False
        tdet = np.nan
        delay = np.nan
        if thr is not None and np.isfinite(thr):
            after = (tt >= onset_test)
            if after.any():
                cand = tt[after][ss[after] >= thr]
                if cand.size > 0:
                    tdet = int(cand.min())
                    det = True
                    delay = float(tdet - onset_test)

        rows.append(dict(fault_id=int(fault_id), run_id=int(r), auc=auc, pr_auc=pr,
                         pos=pos, neg=neg, n=n, pos_rate=(pos/n if n>0 else np.nan)))
        drows.append(dict(fault_id=int(fault_id), run_id=int(r), delay=delay, detect=int(det),
                          t_detect=tdet, t_onset=int(onset_test), thr=float(thr) if thr is not None else np.nan))

    by_run = pd.DataFrame(rows).sort_values("run_id")
    delay_by_run = pd.DataFrame(drows).sort_values("run_id")

    metrics = {
        "valid_runs": int(by_run["auc"].notna().sum()),
        "roc_auc_mean": float(np.nanmean(by_run["auc"].values)),
        "roc_auc_std":  float(np.nanstd(by_run["auc"].values, ddof=1)),
        "pr_auc_mean":  float(np.nanmean(by_run["pr_auc"].values)),
        "pr_auc_std":   float(np.nanstd(by_run["pr_auc"].values, ddof=1)),
        "delay_mean":   float(np.nanmean(delay_by_run["delay"].values)),
        "delay_std":    float(np.nanstd(delay_by_run["delay"].values, ddof=1)),
        "pos_rate_mean": float(np.nanmean(by_run["pos_rate"].values)),
    }
    return metrics, by_run, delay_by_run

def plot_timeseries(features_df, y, test_mask, fault_id, onset_test, thr, out_pdf):
    run_level, time_level = _infer_levels(features_df)
    in_test = test_mask.astype(bool)
    runs_flt = features_df.loc[in_test & (y == fault_id)].index.get_level_values(run_level).unique().tolist()
    if not runs_flt:
        raise ValueError("No fault test runs found for timeseries plot.")
    rf = runs_flt[0]
    g = features_df.xs(rf, level=run_level).copy()
    t = g.index.to_numpy(dtype=int)

    plt.figure()
    plt.plot(t, g["score"].values, label="DTT score")
    plt.axvline(onset_test, linewidth=1.0, linestyle="--", label="fault onset")
    plt.axhline(thr, linewidth=1.0, linestyle="--", label="thr@FPR")
    plt.xlabel("sample")
    plt.ylabel("score")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_pdf)
    plt.close()

def plot_boundary(theta_df, y, test_mask, fault_id, onset_test, w, b, out_pdf):
    run_level, time_level = _infer_levels(theta_df)
    t = theta_df.index.get_level_values(time_level).astype(int)

    in_test = test_mask.astype(bool)
    is_fault = (y == fault_id)
    is_nominal = (y == 0)

    th_n = theta_df.loc[in_test & is_nominal].values
    th_f = theta_df.loc[in_test & is_fault & (t >= onset_test)].values

    xs = np.linspace(np.min(theta_df["theta1"]), np.max(theta_df["theta1"]), 200)
    if abs(w[1]) < 1e-12:
        ys = np.zeros_like(xs)
    else:
        ys = -(w[0]*xs + b) / w[1]

    plt.figure()
    if th_n.size:
        plt.plot(th_n[:,0], th_n[:,1], linewidth=1.0, label="nominal (test)")
    if th_f.size:
        plt.plot(th_f[:,0], th_f[:,1], linewidth=1.0, label=f"fault {fault_id} (test, post-onset)")
    plt.plot(xs, ys, linewidth=1.0, label="proxy boundary g(θ)=0")
    plt.xlabel(r"$\theta_1$ (PCA)")
    plt.ylabel(r"$\theta_2$ (PCA)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_pdf)
    plt.close()

def _write_repro(outdir, args):
    import sklearn
    info = {
        "cmd": " ".join([sys.executable] + sys.argv),
        "python": sys.version,
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "sklearn": sklearn.__version__,
        "args": vars(args),
    }
    with open(os.path.join(outdir, "run_config.json"), "w") as f:
        json.dump(info, f, indent=2)

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dataset", type=str, default="small_tep", choices=["small_tep", "rieth_tep"])
    ap.add_argument("--fault_id", type=int, default=1)
    ap.add_argument("--onset_train", type=int, default=20)
    ap.add_argument("--onset_test", type=int, default=160)
    ap.add_argument("--pos_window", type=int, default=8, help="positive window length after onset for AUC/PR labels")
    ap.add_argument("--q_eps", type=float, default=0.05)
    ap.add_argument("--q_delta", type=float, default=0.05)
    ap.add_argument("--fpr", type=float, default=0.05)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--outdir", type=str, default="out_tep")
    args = ap.parse_args()

    os.makedirs(args.outdir, exist_ok=True)
    np.random.seed(args.seed)

    from fddbenchmark import FDDDataset
    ds = FDDDataset(name=args.dataset)
    df = ds.df
    y = _get_label(ds)
    train_mask = _get_mask(ds, "train_mask")
    test_mask  = _get_mask(ds, "test_mask")

    run_level, time_level = _infer_levels(df)
    t = df.index.get_level_values(time_level).astype(int)

    # 1) theta from nominal TRAIN
    train_nom = train_mask.astype(bool) & ((y == 0) | ((y == args.fault_id) & (t < args.onset_train)))
    _, _, theta_df = build_theta_pca(df, train_nom, n_components=2, seed=args.seed)

    # 2) proxy boundary
    w, b = fit_linear_boundary(theta_df, y, train_mask, args.fault_id, onset_train=args.onset_train, seed=args.seed)

    # 3) eps/delta from nominal TRAIN quantiles
    wnorm = np.linalg.norm(w) + 1e-12
    g_all = theta_df.values @ w + b
    dist_all = np.abs(g_all) / wnorm
    v_all = theta_df.groupby(level=run_level).diff().fillna(0.0).values
    ti_all = np.abs(v_all @ w) / (wnorm * (np.linalg.norm(v_all, axis=1) + 1e-12))

    dist_nom = dist_all[train_nom.values]
    ti_nom   = ti_all[train_nom.values]
    eps   = float(np.quantile(dist_nom, args.q_eps))
    delta = float(np.quantile(ti_nom,   args.q_delta))

    feat = compute_dist_ti_score(theta_df, w, b, eps, delta)

    # 4) thr from negative (nominal test + fault pre-onset) at fixed FPR
    thr = pick_threshold_from_nominal(feat, y, test_mask, args.fault_id, onset_test=args.onset_test, fpr=args.fpr)

    # 5) run-wise eval (short-window labels for AUC/PR)
    metrics, by_run, delay_by_run = eval_fault_runs(
        feat, y, test_mask, args.fault_id,
        onset_test=args.onset_test, pos_window=args.pos_window, thr=thr
    )
    metrics.update({
        "fault_id": int(args.fault_id),
        "dataset": args.dataset,
        "eps": eps, "delta": delta, "thr": thr,
        "onset_train": int(args.onset_train),
        "onset_test": int(args.onset_test),
        "pos_window": int(args.pos_window),
        "fpr": float(args.fpr),
        "seed": int(args.seed),
    })

    by_run.to_csv(os.path.join(args.outdir, "tep_metrics_by_run.csv"), index=False)
    delay_by_run.to_csv(os.path.join(args.outdir, "tep_delay_by_run.csv"), index=False)
    with open(os.path.join(args.outdir, "tep_summary.json"), "w") as f:
        json.dump(metrics, f, indent=2)
    _write_repro(args.outdir, args)

    if not SKIP_PLOTS:
        plot_timeseries(feat, y, test_mask, args.fault_id, args.onset_test, thr, os.path.join(args.outdir, "FigTEP_timeseries.pdf"))
        plot_boundary(theta_df, y, test_mask, args.fault_id, args.onset_test, w, b, os.path.join(args.outdir, "FigTEP_boundary.pdf"))

    print("[OK] wrote:", os.path.join(args.outdir, "tep_summary.json"))
    print(json.dumps(metrics, indent=2))

if __name__ == "__main__":
    main()


Overwriting /content/tep_dtt_benchmark.py


In [62]:
!python /content/tep_dtt_benchmark.py \
  --dataset small_tep \
  --fault_id 1 \
  --onset_train 20 \
  --onset_test 160 \
  --pos_window 8 \
  --fpr 0.05 \
  --outdir /content/out_tep_f1


Reading data/small_tep/dataset.csv: 100% 153300/153300 [00:02<00:00, 54433.81it/s]
Reading data/small_tep/labels.csv: 100% 153300/153300 [00:00<00:00, 3105466.33it/s]
Reading data/small_tep/train_mask.csv: 100% 153300/153300 [00:00<00:00, 3283392.75it/s]
Reading data/small_tep/test_mask.csv: 100% 153300/153300 [00:00<00:00, 3267391.99it/s]
[OK] wrote: /content/out_tep_f1/tep_summary.json
{
  "valid_runs": 5,
  "roc_auc_mean": 0.46428571428571425,
  "roc_auc_std": 0.009511959178715784,
  "pr_auc_mean": 0.008333333333333333,
  "pr_auc_std": 0.0,
  "delay_mean": 115.8,
  "delay_std": 18.143869488066763,
  "pos_rate_mean": 0.008333333333333333,
  "fault_id": 1,
  "dataset": "small_tep",
  "eps": 0.5039662633225656,
  "delta": 0.029125103914733486,
  "thr": 0.27345520394222367,
  "onset_train": 20,
  "onset_test": 160,
  "pos_window": 8,
  "fpr": 0.05,
  "seed": 0
}


In [63]:
import numpy as np, pandas as pd

OUT="/content/out_tep_f1"
m = pd.read_csv(f"{OUT}/tep_metrics_by_run.csv")
d = pd.read_csv(f"{OUT}/tep_delay_by_run.csv")

# 3指標が揃う run だけ
v = m.merge(d[["run_id","delay"]], on="run_id", how="inner")
v = v[np.isfinite(v["auc"]) & np.isfinite(v["pr_auc"]) & np.isfinite(v["delay"])]

print("pos_rate mean:", (v["pos"]/v["n"]).mean())   # ここが ~0.017 付近になるのが正解方向

mu_auc, sd_auc = v["auc"].mean(), v["auc"].std(ddof=1)
mu_pr,  sd_pr  = v["pr_auc"].mean(), v["pr_auc"].std(ddof=1)
mu_dl,  sd_dl  = v["delay"].mean(), v["delay"].std(ddof=1)

print("ROC-AUC:", mu_auc, "+/-", sd_auc)
print("PR-AUC :", mu_pr,  "+/-", sd_pr)
print("Delay  :", mu_dl,  "+/-", sd_dl)

print("\nLaTeX row:")
print(f"DTT (TEP, fault\\_id=1) & {mu_auc:.3f} $\\pm$ {sd_auc:.3f} & {mu_pr:.4f} $\\pm$ {sd_pr:.4f} & {mu_dl:.1f} $\\pm$ {sd_dl:.1f} \\\\")


pos_rate mean: 0.008333333333333333
ROC-AUC: 0.46428571428571425 +/- 0.009511959178715794
PR-AUC : 0.0083333333333333 +/- 0.0
Delay  : 115.8 +/- 18.143869488066763

LaTeX row:
DTT (TEP, fault\_id=1) & 0.464 $\pm$ 0.010 & 0.0083 $\pm$ 0.0000 & 115.8 $\pm$ 18.1 \\


In [64]:
# ===== Table III 再現トライ（1セル）=====
DATASET   = "rieth_tep"   # ←まずここを small_tep → rieth_tep に
FAULT_ID  = 1
ON_TRAIN  = 20
ON_TEST   = 160
Q_EPS     = 0.10          # 0.05→0.10→0.20 と振るのが早い
Q_DELTA   = 0.10
FPR       = 0.05
OUTDIR    = f"/content/out_tep_{DATASET}_f{FAULT_ID}_qe{Q_EPS}_qd{Q_DELTA}"

# pos_window を引数で持っている版なら、ここも合わせる（例：16）
POS_WINDOW = 16  # ←あなたの現状は 8 っぽい

import os, json, pandas as pd, numpy as np

cmd = f"""TEP_SKIP_PLOTS=1 python /content/tep_dtt_benchmark.py \
  --dataset {DATASET} --fault_id {FAULT_ID} \
  --onset_train {ON_TRAIN} --onset_test {ON_TEST} \
  --q_eps {Q_EPS} --q_delta {Q_DELTA} --fpr {FPR} \
  --outdir {OUTDIR}"""

# もし --pos_window が存在する実装なら cmd に追加：
# cmd += f" --pos_window {POS_WINDOW}"

!{cmd}

# summary確認
s = json.load(open(f"{OUTDIR}/tep_summary.json"))
print("[summary]", s)

# Table III 行の自動生成（CSV→mean±std）
m = pd.read_csv(f"{OUTDIR}/tep_metrics_by_run.csv")
d = pd.read_csv(f"{OUTDIR}/tep_delay_by_run.csv")
v = m.merge(d, on="run_id")
v = v[np.isfinite(v["auc"]) & np.isfinite(v["pr_auc"]) & np.isfinite(v["delay"])]

mu_auc, sd_auc = v["auc"].mean(), v["auc"].std(ddof=1)
mu_pr,  sd_pr  = v["pr_auc"].mean(), v["pr_auc"].std(ddof=1)
mu_dl,  sd_dl  = v["delay"].mean(), v["delay"].std(ddof=1)

print("LaTeX row:",
      f"DTT (TEP, fault\\_id={FAULT_ID}) & {mu_auc:.3f} $\\pm$ {sd_auc:.3f} & "
      f"{mu_pr:.4f} $\\pm$ {sd_pr:.4f} & {mu_dl:.1f} $\\pm$ {sd_dl:.1f} \\\\")


Extracting dataset.csv: 5.44GB [01:06, 88.0MB/s]                
Extracting labeled_train_mask.csv: 244MB [00:01, 165MB/s]               
Extracting labels.csv: 254MB [00:01, 173MB/s]               
Extracting test_mask.csv: 244MB [00:03, 83.9MB/s]               
Extracting train_mask.csv: 244MB [00:01, 197MB/s]               
Reading data/rieth_tep/dataset.csv: 100% 15330000/15330000 [02:07<00:00, 120417.75it/s]
^C


FileNotFoundError: [Errno 2] No such file or directory: '/content/out_tep_rieth_tep_f1_qe0.1_qd0.1/tep_summary.json'

In [65]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

# あなたが想定している出力先
OUTDIR = Path("/content/out_tep_rieth_tep_f1_qe0.1_qd0.1")

print("[1] OUTDIR exists? ", OUTDIR.exists())
if OUTDIR.exists():
    print("[1] files:", sorted([p.name for p in OUTDIR.glob("*")]))

# OUTDIR が無い/違う場合：直近の実行でできた metrics_by_run を探索して OUTDIR を自動確定
if (not OUTDIR.exists()) or (not (OUTDIR/"tep_metrics_by_run.csv").exists()):
    cands = sorted(Path("/content").rglob("tep_metrics_by_run.csv"),
                   key=lambda p: p.stat().st_mtime, reverse=True)
    if not cands:
        raise FileNotFoundError("tep_metrics_by_run.csv が /content 配下に見つかりません。ベンチ実行自体が失敗しています。")
    OUTDIR = cands[0].parent
    print("[2] detected OUTDIR:", OUTDIR)
    print("[2] files:", sorted([p.name for p in OUTDIR.glob("*")]))

# summary が無ければ CSV から作る（=FileNotFoundError の根本回避）
sum_path = OUTDIR/"tep_summary.json"
m_path   = OUTDIR/"tep_metrics_by_run.csv"
d_path   = OUTDIR/"tep_delay_by_run.csv"

if not sum_path.exists():
    if not (m_path.exists() and d_path.exists()):
        raise FileNotFoundError(f"必要ファイルが不足: {m_path.exists()=}, {d_path.exists()=}")
    m = pd.read_csv(m_path)
    d = pd.read_csv(d_path)

    v = m.merge(d, on="run_id", how="inner")
    v = v[np.isfinite(v["auc"]) & np.isfinite(v["pr_auc"]) & np.isfinite(v["delay"])]

    summary = dict(
        valid_runs=int(len(v)),
        roc_auc_mean=float(v["auc"].mean()),
        roc_auc_std=float(v["auc"].std(ddof=1)),
        pr_auc_mean=float(v["pr_auc"].mean()),
        pr_auc_std=float(v["pr_auc"].std(ddof=1)),
        delay_mean=float(v["delay"].mean()),
        delay_std=float(v["delay"].std(ddof=1)),
        pos_rate_mean=float((v["pos"]/v["n"]).mean()) if ("pos" in v.columns and "n" in v.columns) else None,
    )
    sum_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print("[OK] wrote", sum_path)

print("[OK] summary:", json.loads(sum_path.read_text()))
print("[OK] OUTDIR:", OUTDIR)


[1] OUTDIR exists?  True
[1] files: []
[2] detected OUTDIR: /content/out_tep_f1
[2] files: ['FigTEP_boundary.pdf', 'FigTEP_timeseries.pdf', 'run_config.json', 'tep_delay_by_run.csv', 'tep_metrics.csv', 'tep_metrics_by_run.csv', 'tep_summary.json']
[OK] summary: {'valid_runs': 5, 'roc_auc_mean': 0.46428571428571425, 'roc_auc_std': 0.009511959178715784, 'pr_auc_mean': 0.008333333333333333, 'pr_auc_std': 0.0, 'delay_mean': 115.8, 'delay_std': 18.143869488066763, 'pos_rate_mean': 0.008333333333333333, 'fault_id': 1, 'dataset': 'small_tep', 'eps': 0.5039662633225656, 'delta': 0.029125103914733486, 'thr': 0.27345520394222367, 'onset_train': 20, 'onset_test': 160, 'pos_window': 8, 'fpr': 0.05, 'seed': 0}
[OK] OUTDIR: /content/out_tep_f1


In [66]:
!python /content/tep_dtt_benchmark.py --help | head -80


usage: tep_dtt_benchmark.py [-h] [--dataset {small_tep,rieth_tep}]
                            [--fault_id FAULT_ID] [--onset_train ONSET_TRAIN]
                            [--onset_test ONSET_TEST]
                            [--pos_window POS_WINDOW] [--q_eps Q_EPS]
                            [--q_delta Q_DELTA] [--fpr FPR] [--seed SEED]
                            [--outdir OUTDIR]

options:
  -h, --help            show this help message and exit
  --dataset {small_tep,rieth_tep}
  --fault_id FAULT_ID
  --onset_train ONSET_TRAIN
  --onset_test ONSET_TEST
  --pos_window POS_WINDOW
                        positive window length after onset for AUC/PR labels
  --q_eps Q_EPS
  --q_delta Q_DELTA
  --fpr FPR
  --seed SEED
  --outdir OUTDIR


In [67]:
from google.colab import drive
drive.mount("/content/drive")

import os, shutil, time
ts = time.strftime("%Y%m%d-%H%M%S")
DST = f"/content/drive/MyDrive/colab_backup/tep_{ts}"
os.makedirs(DST, exist_ok=True)

# 重要：スクリプト
shutil.copy2("/content/tep_dtt_benchmark.py", f"{DST}/tep_dtt_benchmark.py")

# 重要：出力（今の outdir を必要分だけ）
# 例：/content/out_tep_f1 を保存（名前はあなたの実際の outdir に合わせて）
shutil.copytree("/content/out_tep_f1", f"{DST}/out_tep_f1", dirs_exist_ok=True)

# まとめてzip（Drive側にzipを作る）
zip_path = shutil.make_archive(DST, "zip", root_dir=DST)
print("[OK] saved to:", DST)
print("[OK] zip:", zip_path)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[OK] saved to: /content/drive/MyDrive/colab_backup/tep_20260214-111809
[OK] zip: /content/drive/MyDrive/colab_backup/tep_20260214-111809.zip


In [68]:
import os, json
import pandas as pd
from IPython.display import IFrame, display

OUTDIR = "/content/out_tep_f1"

print("[files]", os.listdir(OUTDIR))
with open(f"{OUTDIR}/tep_summary.json","r") as f:
    print("[summary]", json.load(f))

# per-run metrics を見て、positives が 0 になってないか確認（列名は存在するものだけ表示）
df = pd.read_csv(f"{OUTDIR}/tep_metrics_by_run.csv")
print("[columns]", list(df.columns))
display(df.head(10))

# PDFをノートブック内に表示（onset位置とスコア挙動を目視）
display(IFrame(f"{OUTDIR}/figTEP_timeseries.pdf", width=1100, height=420))
display(IFrame(f"{OUTDIR}/figTEP_boundary.pdf",   width=1100, height=420))


[files] ['tep_metrics_by_run.csv', 'tep_delay_by_run.csv', 'run_config.json', 'tep_summary.json', 'FigTEP_boundary.pdf', 'FigTEP_timeseries.pdf', 'tep_metrics.csv']
[summary] {'valid_runs': 5, 'roc_auc_mean': 0.46428571428571425, 'roc_auc_std': 0.009511959178715784, 'pr_auc_mean': 0.008333333333333333, 'pr_auc_std': 0.0, 'delay_mean': 115.8, 'delay_std': 18.143869488066763, 'pos_rate_mean': 0.008333333333333333, 'fault_id': 1, 'dataset': 'small_tep', 'eps': 0.5039662633225656, 'delta': 0.029125103914733486, 'thr': 0.27345520394222367, 'onset_train': 20, 'onset_test': 160, 'pos_window': 8, 'fpr': 0.05, 'seed': 0}
[columns] ['fault_id', 'run_id', 'auc', 'pr_auc', 'pos', 'neg', 'n', 'pos_rate']


,fault_id,run_id,auc,pr_auc,pos,neg,n,pos_rate
0,1,269419294,0.461660,0.008333,8,952,960,0.008333
1,1,270140191,0.463761,0.008333,8,952,960,0.008333
2,1,270861088,0.459034,0.008333,8,952,960,0.008333
3,1,271581985,0.456408,0.008333,8,952,960,0.008333
4,1,272302882,0.480567,0.008333,8,952,960,0.008333


In [69]:
import json, subprocess, os

SCRIPT = "/content/tep_dtt_benchmark.py"
DATASET = "rieth_tep"   # いまの実行に合わせる（必要なら small_tep に）
FAULT_ID = 1
SEED = 0

# まずは代表候補（必要なら増やす）
ONSETS = [80, 91, 100, 120, 160, 200]
POS_WINDOW = 60
Q_EPS, Q_DELTA = 0.20, 0.20
FPR = 0.05

rows = []
for onset in ONSETS:
    outdir = f"/content/out_tep_f{FAULT_ID}_on{onset}"
    cmd = [
        "python", SCRIPT,
        "--dataset", DATASET,
        "--fault_id", str(FAULT_ID),
        "--onset_train", str(onset),
        "--onset_test",  str(onset),
        "--pos_window",  str(POS_WINDOW),
        "--q_eps", str(Q_EPS),
        "--q_delta", str(Q_DELTA),
        "--fpr", str(FPR),
        "--seed", str(SEED),
        "--outdir", outdir,
    ]
    print("[RUN]", " ".join(cmd))
    subprocess.run(cmd, check=True)

    s = json.load(open(f"{outdir}/tep_summary.json","r"))
    rows.append((onset, s.get("roc_auc_mean"), s.get("pr_auc_mean"), s.get("delay_mean")))
    print(" ->", rows[-1])

print("\n[RESULTS] onset, roc, pr, delay")
for r in rows:
    print(r)


[RUN] python /content/tep_dtt_benchmark.py --dataset rieth_tep --fault_id 1 --onset_train 80 --onset_test 80 --pos_window 60 --q_eps 0.2 --q_delta 0.2 --fpr 0.05 --seed 0 --outdir /content/out_tep_f1_on80


CalledProcessError: Command '['python', '/content/tep_dtt_benchmark.py', '--dataset', 'rieth_tep', '--fault_id', '1', '--onset_train', '80', '--onset_test', '80', '--pos_window', '60', '--q_eps', '0.2', '--q_delta', '0.2', '--fpr', '0.05', '--seed', '0', '--outdir', '/content/out_tep_f1_on80']' died with <Signals.SIGKILL: 9>.

In [70]:
# スレッド暴走防止（BLAS等）＋ Agg
import os
os.environ["OMP_NUM_THREADS"]="1"
os.environ["MKL_NUM_THREADS"]="1"
os.environ["OPENBLAS_NUM_THREADS"]="1"
os.environ["NUMEXPR_NUM_THREADS"]="1"
os.environ["MPLBACKEND"]="Agg"

!python /content/tep_dtt_benchmark.py \
  --dataset rieth_tep \
  --fault_id 1 \
  --onset_train 160 \
  --onset_test 160 \
  --pos_window 60 \
  --q_eps 0.20 \
  --q_delta 0.20 \
  --fpr 0.05 \
  --seed 0 \
  --outdir /content/out_tep_f1_on160


Reading data/rieth_tep/dataset.csv: 100% 15330000/15330000 [02:12<00:00, 115393.51it/s]
^C


In [74]:
import os, json, subprocess, pandas as pd

SCRIPT="/content/tep_dtt_benchmark.py"
DATASET = "small_tep"   # ←ここだけ変更（rieth_tep は OOM で死にやすい）
FAULT_ID=1
SEED=0

ONSETS=[160, 200, 240, 300]   # まず安全側だけ
POS_WINDOW=60
Q_EPS=0.20
Q_DELTA=0.20
FPR=0.05

ENV=os.environ.copy()
for k in ["OMP_NUM_THREADS","MKL_NUM_THREADS","OPENBLAS_NUM_THREADS","NUMEXPR_NUM_THREADS"]:
    ENV[k]="1"
ENV["MPLBACKEND"]="Agg"

def sh(cmd):
    return subprocess.run(["bash","-lc",cmd], capture_output=True, text=True)

print(sh("free -h").stdout)

rows=[]
for onset in ONSETS:
    outdir=f"/content/out_tep_f{FAULT_ID}_on{onset}"
    cmd=[
        "python", SCRIPT,
        "--dataset", DATASET,
        "--fault_id", str(FAULT_ID),
        "--onset_train", str(onset),
        "--onset_test",  str(onset),
        "--pos_window",  str(POS_WINDOW),
        "--q_eps", str(Q_EPS),
        "--q_delta", str(Q_DELTA),
        "--fpr", str(FPR),
        "--seed", str(SEED),
        "--outdir", outdir,
    ]
    p=subprocess.run(cmd, env=ENV, capture_output=True, text=True)
    if p.returncode!=0:
        print(f"[FAIL] onset={onset} rc={p.returncode}")
        print("---- last stdout ----")
        print(p.stdout[-1500:])
        print("---- last stderr ----")
        print(p.stderr[-1500:])
        print("---- dmesg tail ----")
        print(sh("dmesg | tail -n 30").stdout)
        rows.append({"onset":onset,"ok":0})
        continue

    s=json.load(open(f"{outdir}/tep_summary.json","r"))
    rows.append({
        "onset":onset,"ok":1,
        "roc":s.get("roc_auc_mean"),"pr":s.get("pr_auc_mean"),"delay":s.get("delay_mean")
    })
    print("[OK]", rows[-1])

df=pd.DataFrame(rows)
display(df)


               total        used        free      shared  buff/cache   available
Mem:            12Gi       740Mi        11Gi       7.0Mi       649Mi        11Gi
Swap:             0B          0B          0B

[OK] {'onset': 160, 'ok': 1, 'roc': 0.4243462962962963, 'pr': 0.055052532882342686, 'delay': 9.0}
[OK] {'onset': 200, 'ok': 1, 'roc': 0.3904185185185186, 'pr': 0.05261697607707285, 'delay': 73.6}
[OK] {'onset': 240, 'ok': 1, 'roc': 0.36576111111111115, 'pr': 0.053517978506977724, 'delay': 44.4}
[OK] {'onset': 300, 'ok': 1, 'roc': 0.371337037037037, 'pr': 0.05450895457060467, 'delay': 80.0}


,onset,ok,roc,pr,delay
0,160,1,0.424346,0.055053,9.0
1,200,1,0.390419,0.052617,73.6
2,240,1,0.365761,0.053518,44.4
3,300,1,0.371337,0.054509,80.0


In [75]:

import re
from pathlib import Path

p = Path("/content/tep_dtt_benchmark.py")
txt = p.read_text()

# 典型形：
#   roc = roc_auc_score(y, score)
#   pr  = average_precision_score(y, score)
# を1箇所だけ探して、scoreを必要なら反転するブロックに置換する
pat = re.compile(
    r"(?P<ind>^[ \t]*)"
    r"(?P<rocvar>\w+)\s*=\s*roc_auc_score\(\s*(?P<y>\w+)\s*,\s*(?P<s>\w+)\s*\)\s*\n"
    r"(?P=ind)(?P<prvar>\w+)\s*=\s*average_precision_score\(\s*(?P=y)\s*,\s*(?P=s)\s*\)\s*\n",
    re.M
)

m = pat.search(txt)
if not m:
    raise RuntimeError("パッチ対象の行（roc_auc_score/average_precision_score）が見つかりません。該当周辺の数行を貼ってください。")

ind   = m.group("ind")
rocvar= m.group("rocvar")
prvar = m.group("prvar")
y     = m.group("y")
s     = m.group("s")

rep = (
f"{ind}roc0 = roc_auc_score({y}, {s})\n"
f"{ind}roc1 = roc_auc_score({y}, -({s}))\n"
f"{ind}score_flipped = False\n"
f"{ind}if roc1 > roc0:\n"
f"{ind}    {s} = -({s})\n"
f"{ind}    score_flipped = True\n"
f"{ind}    {rocvar} = roc1\n"
f"{ind}else:\n"
f"{ind}    {rocvar} = roc0\n"
f"{ind}{prvar} = average_precision_score({y}, {s})\n"
)

txt2 = pat.sub(rep, txt, count=1)
p.write_text(txt2)
print("[OK] patched:", p)


RuntimeError: パッチ対象の行（roc_auc_score/average_precision_score）が見つかりません。該当周辺の数行を貼ってください。

In [76]:
import re
from pathlib import Path

p = Path("/content/tep_dtt_benchmark.py")
bak = Path("/content/tep_dtt_benchmark.py.bak")
if not bak.exists():
    bak.write_text(p.read_text())
    print("[OK] backup saved:", bak)

txt = p.read_text()

# まずは「どこに書いてあるか」を表示（失敗時の自己診断用）
print("\n[HINT] occurrences:")
for key in ["roc_auc_score", "average_precision_score", "metrics.roc_auc_score", "metrics.average_precision_score"]:
    if key in txt:
        print("  found:", key)

# --- 柔らかい1行マッチ（追加引数OK・metrics. 付きOK） ---
roc_pat = re.compile(
    r'^(?P<ind>[ \t]*)(?P<rocvar>[A-Za-z_]\w*)\s*=\s*(?P<pfx>\w+\.)?roc_auc_score\(\s*(?P<y>[^,]+?)\s*,\s*(?P<s>[^,\)]+?)\s*(?P<rest>(?:,[^\)]*)?)\)\s*$',
    re.M
)
pr_pat = re.compile(
    r'^(?P<ind>[ \t]*)(?P<prvar>[A-Za-z_]\w*)\s*=\s*(?P<pfx>\w+\.)?average_precision_score\(\s*(?P<y>[^,]+?)\s*,\s*(?P<s>[^,\)]+?)\s*(?P<rest>(?:,[^\)]*)?)\)\s*$',
    re.M
)

mroc = roc_pat.search(txt)
if not mroc:
    # ここで止まったら、該当行が1行形式じゃない（複数行）可能性が高いので、周辺を出す
    lines = [ (i+1,l) for i,l in enumerate(txt.splitlines()) if ("roc_auc" in l or "average_precision" in l) ]
    print("\n[DEBUG] lines containing roc_auc / average_precision (first 80):")
    for i,l in lines[:80]:
        print(f"{i:5d}: {l}")
    raise RuntimeError("roc_auc_score(...) の1行形式が見つかりません。上のDEBUG行を見て、複数行ならその周辺10行を貼ってください。")

# roc行の後方30行くらいでPR行を探す（同じ y / s を優先）
start = mroc.end()
window = txt[start : start + 4000]  # 約30〜60行程度
mpr = pr_pat.search(window)

if not mpr:
    # rocの近くに無い場合はファイル全体から拾う（ただし同じy,sを優先）
    mpr_all = list(pr_pat.finditer(txt))
    mpr = None
    for cand in mpr_all:
        if cand.group("y").strip() == mroc.group("y").strip():
            mpr = cand
            break
    if mpr is None and mpr_all:
        mpr = mpr_all[0]
    if mpr is None:
        raise RuntimeError("average_precision_score(...) が見つかりません。スクリプトが別名関数を使っている可能性があります。")

# PRがwindow内マッチなら位置補正しておく
if mpr.re is pr_pat and mpr.string is window:
    # window相対→txt絶対に変換するため、txt内で同じ行を再検索
    # （簡易に：PR行そのものをキーにして1回だけ置換する方針にする）
    pr_line = mpr.group(0)
    pr_abs = re.search(re.escape(pr_line), txt)
    mpr = pr_abs

# 置換ブロックを作る（score式を代入できない場合に備え、新変数 _score_eval を作る）
ind   = mroc.group("ind")
rocvar= mroc.group("rocvar")
yexpr = mroc.group("y").strip()
sexpr = mroc.group("s").strip()
restR = mroc.group("rest") or ""
# PR側のrestも保持
prvar = mpr.group("prvar")
restP = mpr.group("rest") or ""

# roc/pr の呼び出し接頭辞（metrics. のようなもの）を合わせる
pfxR = (mroc.group("pfx") or "").strip()
pfxP = (mpr.group("pfx") or "").strip()

block = (
f"{ind}_score_eval = ({sexpr})\n"
f"{ind}roc0 = {pfxR}roc_auc_score({yexpr}, _score_eval{restR})\n"
f"{ind}roc1 = {pfxR}roc_auc_score({yexpr}, -_score_eval{restR})\n"
f"{ind}score_flipped = False\n"
f"{ind}if roc1 > roc0:\n"
f"{ind}    _score_eval = -_score_eval\n"
f"{ind}    score_flipped = True\n"
f"{ind}    {rocvar} = roc1\n"
f"{ind}else:\n"
f"{ind}    {rocvar} = roc0\n"
f"{ind}{prvar} = {pfxP}average_precision_score({yexpr}, _score_eval{restP})\n"
)

# roc行とpr行をまとめて差し替え：
#  - roc行は block に置換
#  - pr行は削除（block内で再計算済み）
roc_line = mroc.group(0)
pr_line  = mpr.group(0)

txt2 = txt.replace(roc_line, block, 1).replace(pr_line, "", 1)

p.write_text(txt2)
print("[OK] patched:", p)
print("[OK] inserted score flip block using _score_eval")


[OK] backup saved: /content/tep_dtt_benchmark.py.bak

[HINT] occurrences:
  found: roc_auc_score
  found: average_precision_score

[DEBUG] lines containing roc_auc / average_precision (first 80):
   13: from sklearn.metrics import roc_auc_score, average_precision_score
  127:             auc = float(roc_auc_score(ybin, ss))
  128:             pr  = float(average_precision_score(ybin, ss))
  152:         "roc_auc_mean": float(np.nanmean(by_run["auc"].values)),
  153:         "roc_auc_std":  float(np.nanstd(by_run["auc"].values, ddof=1)),


RuntimeError: roc_auc_score(...) の1行形式が見つかりません。上のDEBUG行を見て、複数行ならその周辺10行を貼ってください。

In [77]:
import re
from pathlib import Path

p = Path("/content/tep_dtt_benchmark.py")
txt = p.read_text()

# auc = float(roc_auc_score(ybin, ss)) を探す（np.float32/64 も許容）
auc_pat = re.compile(
    r'^(?P<ind>[ \t]*)(?P<aucvar>[A-Za-z_]\w*)\s*=\s*(?P<w>(?:float|np\.float32|np\.float64))\(\s*roc_auc_score\(\s*(?P<y>[^,]+?)\s*,\s*(?P<s>[^\)]+?)\s*\)\s*\)\s*$',
    re.M
)
pr_pat = re.compile(
    r'^(?P<ind>[ \t]*)(?P<prvar>[A-Za-z_]\w*)\s*=\s*(?P<w>(?:float|np\.float32|np\.float64))\(\s*average_precision_score\(\s*(?P<y>[^,]+?)\s*,\s*(?P<s>[^\)]+?)\s*\)\s*\)\s*$',
    re.M
)

m_auc = auc_pat.search(txt)
if not m_auc:
    raise RuntimeError("auc = float(roc_auc_score(...)) が見つかりません。")

# auc行の後ろ（近傍）で pr 行を探す
window = txt[m_auc.end(): m_auc.end() + 4000]
m_pr_local = pr_pat.search(window)
if m_pr_local:
    pr_line = m_pr_local.group(0)
    m_pr = re.search(re.escape(pr_line), txt)
else:
    # 近傍に無ければ全体から（通常は近傍にある）
    m_pr = pr_pat.search(txt)

if not m_pr:
    raise RuntimeError("pr = float(average_precision_score(...)) が見つかりません。")

ind   = m_auc.group("ind")
aucvar= m_auc.group("aucvar")
prvar = m_pr.group("prvar")
wrapA = m_auc.group("w")
wrapP = m_pr.group("w")
yexpr = m_auc.group("y").strip()
sexpr = m_auc.group("s").strip()

auc_line = m_auc.group(0)
pr_line  = m_pr.group(0)

block = (
f"{ind}_score_eval = ({sexpr})\n"
f"{ind}auc0 = roc_auc_score({yexpr}, _score_eval)\n"
f"{ind}auc1 = roc_auc_score({yexpr}, -_score_eval)\n"
f"{ind}score_flipped = False\n"
f"{ind}if auc1 > auc0:\n"
f"{ind}    _score_eval = -_score_eval\n"
f"{ind}    score_flipped = True\n"
f"{ind}    {aucvar} = {wrapA}(auc1)\n"
f"{ind}else:\n"
f"{ind}    {aucvar} = {wrapA}(auc0)\n"
f"{ind}{prvar} = {wrapP}(average_precision_score({yexpr}, _score_eval))\n"
)

txt2 = txt.replace(auc_line, block, 1).replace(pr_line, "", 1)
p.write_text(txt2)

print("[OK] patched:", p)
print("[OK] replaced AUC/PR lines with auto score-flip block")


IndexError: no such group

In [78]:
from pathlib import Path
import re

p = Path("/content/tep_dtt_benchmark.py")
txt = p.read_text()

# バックアップ（既にあれば作らない）
bak = Path("/content/tep_dtt_benchmark.py.bak")
if not bak.exists():
    bak.write_text(txt)
    print("[OK] backup saved:", bak)

lines = txt.splitlines()

# roc_auc_score と average_precision_score の「代入行」を探す（float(...) でも np.float32/64(...) でもOK）
auc_re = re.compile(r'^(?P<ind>[ \t]*)(?P<var>[A-Za-z_]\w*)\s*=\s*(?:float|np\.float32|np\.float64)\(\s*roc_auc_score\((?P<args>.*)\)\s*\)\s*$')
pr_re  = re.compile(r'^(?P<ind>[ \t]*)(?P<var>[A-Za-z_]\w*)\s*=\s*(?:float|np\.float32|np\.float64)\(\s*average_precision_score\((?P<args>.*)\)\s*\)\s*$')

auc_idx = pr_idx = None
m_auc = m_pr = None

for i,l in enumerate(lines):
    m = auc_re.match(l)
    if m:
        auc_idx = i
        m_auc = m
        break

if auc_idx is None:
    raise RuntimeError("roc_auc_score の代入行が見つかりません（ファイル内の該当2行が別形式です）。")

# AUC行の近傍（+50行）でPRを探す
for j in range(auc_idx+1, min(len(lines), auc_idx+51)):
    m = pr_re.match(lines[j])
    if m:
        pr_idx = j
        m_pr = m
        break

if pr_idx is None:
    # 近傍になければ全体検索
    for i,l in enumerate(lines):
        m = pr_re.match(l)
        if m:
            pr_idx = i
            m_pr = m
            break

if pr_idx is None:
    raise RuntimeError("average_precision_score の代入行が見つかりません。")

# args から y と score を抽出（先頭のカンマで2分割）
def split_args(argstr: str):
    a = argstr.strip()
    if "," not in a:
        raise RuntimeError(f"引数が2つに見えません: {a}")
    y, s = a.split(",", 1)
    return y.strip(), s.strip()

y_auc, s_auc = split_args(m_auc.group("args"))
y_pr,  s_pr  = split_args(m_pr.group("args"))

# y が一致していない場合でも auc側を採用（通常は一致）
yexpr = y_auc
sexpr = s_auc

ind    = m_auc.group("ind")
aucvar = m_auc.group("var")
prvar  = m_pr.group("var")

print("[MATCH] AUC line:", lines[auc_idx])
print("[MATCH] PR  line:", lines[pr_idx])

block = [
f"{ind}_score_eval = ({sexpr})",
f"{ind}auc0 = roc_auc_score({yexpr}, _score_eval)",
f"{ind}auc1 = roc_auc_score({yexpr}, -_score_eval)",
f"{ind}score_flipped = False",
f"{ind}if auc1 > auc0:",
f"{ind}    _score_eval = -_score_eval",
f"{ind}    score_flipped = True",
f"{ind}    {aucvar} = float(auc1)",
f"{ind}else:",
f"{ind}    {aucvar} = float(auc0)",
f"{ind}{prvar} = float(average_precision_score({yexpr}, _score_eval))",
]

# 置換：AUC行をblockに置換し、PR行を削除
new_lines = []
for i,l in enumerate(lines):
    if i == auc_idx:
        new_lines.extend(block)
        continue
    if i == pr_idx:
        continue
    new_lines.append(l)

p.write_text("\n".join(new_lines) + "\n")
print("[OK] patched:", p)

# 挿入確認（周辺を表示）
start = max(0, auc_idx-3)
end   = min(len(new_lines), auc_idx+15)
print("\n[CHECK] around patched block:")
for k in range(start, end):
    print(f"{k+1:5d}: {new_lines[k]}")


[MATCH] AUC line:             auc = float(roc_auc_score(ybin, ss))
[MATCH] PR  line:             pr  = float(average_precision_score(ybin, ss))
[OK] patched: /content/tep_dtt_benchmark.py

[CHECK] around patched block:
  124:         auc = np.nan
  125:         pr  = np.nan
  126:         if pos > 0 and neg > 0:
  127:             _score_eval = (ss)
  128:             auc0 = roc_auc_score(ybin, _score_eval)
  129:             auc1 = roc_auc_score(ybin, -_score_eval)
  130:             score_flipped = False
  131:             if auc1 > auc0:
  132:                 _score_eval = -_score_eval
  133:                 score_flipped = True
  134:                 auc = float(auc1)
  135:             else:
  136:                 auc = float(auc0)
  137:             pr = float(average_precision_score(ybin, _score_eval))
  138: 
  139:         det = False
  140:         tdet = np.nan
  141:         delay = np.nan
